# Row-wise L2 normalisation: eight rows per block

Embedding models L2-normalise every output vector so that a dot product becomes a cosine similarity:

$$y_{r,:} = \frac{x_{r,:}}{\max\!\left(\lVert x_{r,:} \rVert_2,\ \varepsilon\right)}, \qquad \varepsilon = 10^{-12}$$

Embedding rows are short (a few hundred numbers), so a whole block per row would leave most threads idle on small dimensions. Instead, pack **several rows into one block** with a 2-D block shape.

## The mapping
- Block shape `(32, 8)`: `threadIdx.y` picks one of 8 rows handled by the block, and the 32 threads with the same `threadIdx.y` (one warp) cooperate on that row. Grid: $\lceil \text{rows}/8 \rceil$ blocks.
- Row index `r = blockIdx.x * 8 + threadIdx.y`. Lane `tx` sums squares over columns `tx, tx + 32, ...`.
- Partial sums go into a **2-D shared array** `cuda.shared.array((8, 32), float32)`; reduce along the second axis with a tree (`step = 16, 8, 4, 2, 1`), calling `cuda.syncthreads()` after every step.
- Then every lane writes its columns of `y`.

The last block may contain rows past the end. Those threads must **still reach every `cuda.syncthreads()`**: guard the loads and stores, never return early.

## Inputs and output
- `x`: `float32`, shape `(rows, dim)`, `rows >= 1`, `dim >= 1`.
- Return `y`: `float32`, same shape. An all-zero row stays zero.
- Compared with relative tolerance $10^{-5}$, absolute $10^{-7}$.

## Runtime
GPU in Colab; checked here on Numba's CUDA simulator.

**Runtime → Change runtime type → GPU** before running.

Write your solution in the next cell, then run the test cell.

In [ ]:
!pip -q install "numba-cuda[cu12]"

In [ ]:
import math
import numpy as np
from numba import cuda, float32

LANES, ROWS_PER_BLOCK = 32, 8
EPS = 1e-12

def l2_normalize_rows(x):
    """x: float32 (rows, dim). Returns float32 (rows, dim) with unit-L2 rows."""
    rows, dim = x.shape
    d_x = cuda.to_device(np.ascontiguousarray(x))
    d_y = cuda.device_array((rows, dim), dtype=np.float32)
    grid = (rows + ROWS_PER_BLOCK - 1) // ROWS_PER_BLOCK
    l2_kernel[grid, (LANES, ROWS_PER_BLOCK)](d_x, d_y)
    return d_y.copy_to_host()

@cuda.jit
def l2_kernel(x, y):
    tx, ty = cuda.threadIdx.x, cuda.threadIdx.y
    r = cuda.blockIdx.x * ROWS_PER_BLOCK + ty
    # TODO: partial sum of squares per lane, tree-reduce in a (8, 32) shared array, then scale
    pass


In [ ]:
# Tests (same harness and cases as the website)
_HARNESS = '"""Test harness shared by build.py (CPython) and the browser runner (Pyodide).\n\nOne file, so a test that passes at build time is graded the same way in the browser.\nValues cross JSON with small tags: ndarray, tuple, DataFrame/Series, non-finite floats.\n"""\nimport json, math\n\ntry:\n    import numpy as np\nexcept ImportError:  # pure-python problems never need it\n    np = None\n\n\ndef enc(x):\n    if np is not None and isinstance(x, np.ndarray):\n        return {\'__nd__\': enc(x.tolist()), \'dtype\': str(x.dtype)}\n    if np is not None and isinstance(x, np.generic):\n        return enc(x.item())\n    if isinstance(x, float) and not math.isfinite(x):\n        return {\'__f__\': repr(x)}\n    if isinstance(x, tuple):\n        return {\'__tup__\': [enc(v) for v in x]}\n    if isinstance(x, list):\n        return [enc(v) for v in x]\n    if isinstance(x, dict):\n        return {\'__dict__\': [[enc(k), enc(v)] for k, v in x.items()]}\n    t = type(x).__name__\n    if t == \'Tensor\':  # torch, in Colab and at build time\n        return enc(x.detach().cpu().numpy())\n    if t == \'DataFrame\':\n        # Cells go through enc() as Python values, so floats keep full precision (to_json caps at 15 digits).\n        return {\'__df__\': {\'columns\': [enc(c) for c in x.columns], \'data\': [enc(list(r)) for r in x.itertuples(index=False, name=None)]},\n                \'dtypes\': [str(d) for d in x.dtypes]}\n    if t == \'Series\':\n        return {\'__sr__\': enc(x.tolist()), \'name\': x.name}\n    return x\n\n\ndef dec(x):\n    if isinstance(x, list):\n        return [dec(v) for v in x]\n    if not isinstance(x, dict):\n        return x\n    if \'__nd__\' in x:\n        return np.array(dec(x[\'__nd__\']), dtype=x[\'dtype\'])\n    if \'__f__\' in x:\n        return float(x[\'__f__\'])\n    if \'__tup__\' in x:\n        return tuple(dec(v) for v in x[\'__tup__\'])\n    if \'__dict__\' in x:\n        return {_hashable(dec(k)): dec(v) for k, v in x[\'__dict__\']}\n    if \'__df__\' in x:\n        import pandas as pd\n        s = x[\'__df__\']\n        df = pd.DataFrame([dec(r) for r in s[\'data\']], columns=[dec(c) for c in s[\'columns\']])\n        for c, d in zip(df.columns, x[\'dtypes\']):\n            if d.startswith((\'int\', \'float\', \'bool\')):\n                df[c] = df[c].astype(d)\n        return df\n    if \'__sr__\' in x:\n        import pandas as pd\n        return pd.Series(dec(x[\'__sr__\']), name=x[\'name\'])\n    return x\n\n\ndef _hashable(k):\n    return tuple(k) if isinstance(k, list) else k\n\n\ndef same(got, exp, rtol=1e-6, atol=1e-8, path=\'output\'):\n    """(ok, why). Floats compare with tolerance, everything else exactly. Shapes must match."""\n    if type(exp).__name__ == \'DataFrame\':\n        if type(got).__name__ != \'DataFrame\':\n            return False, f\'{path}: expected a DataFrame, got {type(got).__name__}\'\n        if list(got.columns) != list(exp.columns):\n            return False, f\'{path}: columns {list(got.columns)} != expected {list(exp.columns)}\'\n        if got.shape != exp.shape:\n            return False, f\'{path}: shape {got.shape} != expected {exp.shape}\'\n        for c in exp.columns:\n            ok, why = same(got[c].tolist(), exp[c].tolist(), rtol, atol, f\'{path}[{c!r}]\')\n            if not ok:\n                return ok, why\n        return True, \'\'\n    if type(exp).__name__ == \'Series\':\n        exp = exp.tolist()\n        got = got.tolist() if type(got).__name__ == \'Series\' else got\n    if np is not None and isinstance(exp, np.ndarray):\n        try:\n            g = np.asarray(got)\n        except Exception:\n            return False, f\'{path}: expected an array, got {type(got).__name__}\'\n        if g.shape != exp.shape:\n            return False, f\'{path}: shape {g.shape} != expected {exp.shape}\'\n        if exp.dtype.kind in \'fc\' or g.dtype.kind in \'fc\':\n            if g.dtype.kind not in \'fciub\':\n                return False, f\'{path}: expected numbers, got dtype {g.dtype}\'\n            bad = ~np.isclose(g.astype(complex if exp.dtype.kind == \'c\' else float), exp, rtol=rtol, atol=atol, equal_nan=True)\n            if bad.any():\n                i = tuple(int(v) for v in np.argwhere(bad)[0])\n                return False, f\'{path}{list(i)}: {g[i].tolist()!r} != expected {exp[i].tolist()!r}\'\n            return True, \'\'\n        if not np.array_equal(g, exp):\n            i = tuple(int(v) for v in np.argwhere(g != exp)[0]) if g.shape else ()\n            return False, f\'{path}{list(i)}: {g[i].tolist()!r} != expected {exp[i].tolist()!r}\'\n        return True, \'\'\n    if isinstance(exp, float) or (isinstance(exp, int) and isinstance(got, float) and not isinstance(exp, bool)):\n        if np is not None and isinstance(got, np.generic):\n            got = got.item()\n        if not isinstance(got, (int, float)) or isinstance(got, bool):\n            return False, f\'{path}: expected a number, got {type(got).__name__}\'\n        if math.isnan(exp) and math.isnan(got):\n            return True, \'\'\n        return (True, \'\') if math.isclose(got, exp, rel_tol=rtol, abs_tol=atol) else (False, f\'{path}: {got!r} != expected {exp!r}\')\n    if isinstance(exp, (list, tuple)):\n        if np is not None and isinstance(got, np.ndarray):\n            got = got.tolist()\n        if not isinstance(got, (list, tuple)):\n            return False, f\'{path}: expected a {type(exp).__name__}, got {type(got).__name__}\'\n        if len(got) != len(exp):\n            return False, f\'{path}: length {len(got)} != expected {len(exp)}\'\n        for i, (g, e) in enumerate(zip(got, exp)):\n            ok, why = same(g, e, rtol, atol, f\'{path}[{i}]\')\n            if not ok:\n                return ok, why\n        return True, \'\'\n    if isinstance(exp, dict):\n        if not isinstance(got, dict):\n            return False, f\'{path}: expected a dict, got {type(got).__name__}\'\n        if set(got) != set(exp):\n            return False, f\'{path}: keys {sorted(map(str, got))} != expected {sorted(map(str, exp))}\'\n        for k in exp:\n            ok, why = same(got[k], exp[k], rtol, atol, f\'{path}[{k!r}]\')\n            if not ok:\n                return ok, why\n        return True, \'\'\n    if np is not None and isinstance(got, np.generic):\n        got = got.item()\n    return (True, \'\') if got == exp else (False, f\'{path}: {got!r} != expected {exp!r}\')\n\n\ndef run_sql(query, setup):\n    """Run `setup` (CREATE/INSERT script) in a fresh in-memory db, then `query`; rows as lists."""\n    import sqlite3  # a separate package in Pyodide: only SQL problems load it\n    con = sqlite3.connect(\':memory:\')\n    try:\n        con.executescript(setup)\n        cur = con.execute(query)\n        return [[c[0] for c in cur.description], [list(r) for r in cur.fetchall()]]\n    finally:\n        con.close()\n\n\ndef show_args(args, runtime, n=600):\n    """Arguments as the learner would write them: one per line, SQL setup as-is."""\n    if runtime == \'sql\':\n        return args[0] if len(args[0]) <= n else args[0][:n] + \' …\'\n    return \'\\n\'.join(short(a, n) for a in args)\n\n\ndef short(x, n=300):\n    s = repr(x)\n    return s if len(s) <= n else s[:n] + \' …\'\n\n\ndef grade(fn, tests, runtime, rtol=1e-6, atol=1e-8):\n    """Run `fn` (a callable, or SQL text for runtime \'sql\') on each test; list of result dicts."""\n    out = []\n    for t in tests:\n        args, exp = prep(dec(t[\'a\']), runtime), dec(t[\'e\'])\n        try:\n            got = run_sql(fn, args[0]) if runtime == \'sql\' else plain(fn(*_copy(args)))\n            ok, why = same(got, exp, rtol, atol)\n        except Exception as e:  # learner code raised: report it like a failed test\n            got, ok, why = None, False, f\'{type(e).__name__}: {e}\'\n        out.append({\'ok\': ok, \'why\': why, \'got\': short(got) if not ok else \'\',\n                    \'inp\': show_args(args, runtime, 400) if not ok else \'\'})\n    return out\n\n\ndef prep(args, runtime):\n    """Arrays cross JSON as ndarrays; torch problems take tensors."""\n    if runtime == \'torch\':\n        import torch\n        return [torch.as_tensor(a) if np is not None and isinstance(a, np.ndarray) else a for a in args]\n    if runtime == \'triton\':                                  # Colab only: kernels need CUDA tensors\n        import torch\n        return [torch.as_tensor(a).cuda() if np is not None and isinstance(a, np.ndarray) else a for a in args]\n    return args\n\n\ndef plain(x):\n    """Framework arrays (torch, jax) -> numpy, recursively, so results compare and encode the same way."""\n    t = type(x).__name__\n    if t == \'Tensor\':\n        return x.detach().cpu().numpy()\n    if t in (\'ArrayImpl\', \'Array\') and hasattr(x, \'__array__\'):\n        return np.asarray(x)\n    if isinstance(x, tuple):\n        return tuple(plain(v) for v in x)\n    if isinstance(x, list):\n        return [plain(v) for v in x]\n    if isinstance(x, dict):\n        return {k: plain(v) for k, v in x.items()}\n    return x\n\n\ndef _copy(args):\n    # Tests reuse inputs across Run and Submit, so a function that mutates its input mustn\'t poison later cases.\n    import copy\n    return copy.deepcopy(args)\n'
_TESTS = '[{"a": [{"__nd__": [[3.0, 4.0], [0.0, -2.0]], "dtype": "float32"}], "e": {"__nd__": [[0.6000000238418579, 0.800000011920929], [0.0, -1.0]], "dtype": "float32"}}, {"a": [{"__nd__": [[1.0, 1.0, 1.0, 1.0]], "dtype": "float32"}], "e": {"__nd__": [[0.5, 0.5, 0.5, 0.5]], "dtype": "float32"}}, {"a": [{"__nd__": [[0.304717093706131, -1.039984107017517, 0.7504512071609497, 0.9405646920204163, -1.9510351419448853], [-1.3021794557571411, 0.12784039974212646, -0.31624260544776917, -0.01680115796625614, -0.8530439138412476], [0.879397988319397, 0.7777919173240662, 0.06603069603443146, 1.1272412538528442, 0.46750932931900024], [0.0, 0.0, 0.0, 0.0, 0.0], [-0.18486236035823822, -0.6809295415878296, 1.222541332244873, -0.15452948212623596, -0.4283278286457062], [-0.35213354229927063, 0.5323091745376587, 0.3654440641403198, 0.4127326011657715, 0.4308210015296936], [2.1416475772857666, -0.40641501545906067, -0.5122427344322205, -0.8137727379798889, 0.6159794330596924], [1.1289722919464111, -0.11394745856523514, -0.8401564955711365, -0.824481189250946, 0.6505928039550781], [0.7432541847229004, 0.543154239654541, -0.6655097007751465, 0.23216132819652557, 0.11668580770492554], [0.21868859231472015, 0.8714287877082825, 0.2235955446958542, 0.6789135336875916, 0.06757906824350357]], "dtype": "float32"}], "e": {"__nd__": [[0.12017994374036789, -0.4101680815219879, 0.29597675800323486, 0.37095722556114197, -0.7694851756095886], [-0.8170616626739502, 0.08021435886621475, -0.19842864573001862, -0.010542004369199276, -0.5352483987808228], [0.5189206004142761, 0.4589642584323883, 0.03896379843354225, 0.6651694774627686, 0.27587080001831055], [0.0, 0.0, 0.0, 0.0, 0.0], [-0.12464010715484619, -0.4591044485569, 0.8242764472961426, -0.1041887104511261, -0.28879231214523315], [-0.3718315362930298, 0.5620859861373901, 0.38588663935661316, 0.43582043051719666, 0.4549206793308258], [0.8702809810638428, -0.16515101492404938, -0.20815521478652954, -0.3306851089000702, 0.25030970573425293], [0.6415790319442749, -0.06475473195314407, -0.47744908928871155, -0.4685410261154175, 0.3697226941585541], [0.6378370523452759, 0.46611765027046204, -0.5711191892623901, 0.19923345744609833, 0.10013604164123535], [0.19014972448349, 0.7577072978019714, 0.1944163292646408, 0.5903152823448181, 0.05875999853014946]], "dtype": "float32"}}, {"a": [{"__nd__": [[0.2891193926334381], [0.6312882304191589], [-1.4571558237075806], [-0.3196712136268616], [-0.47037264704704285], [-0.6388778686523438], [-0.27514225244522095], [1.4949413537979126], [-0.8658311367034912]], "dtype": "float32"}], "e": {"__nd__": [[1.0], [1.0], [-1.0], [-1.0], [-1.0], [-1.0], [-1.0], [1.0], [-1.0]], "dtype": "float32"}}, {"a": [{"__nd__": [[0.9682783484458923, -1.682869791984558, -0.33488503098487854, 0.16275306046009064, 0.5862223505973816, 0.7112265825271606, 0.7933472394943237, -0.3487250804901123, -0.46235179901123047, 0.8579759001731873, -0.19130432605743408, -1.275686264038086, -1.1332871913909912, -0.9194523096084595, 0.49716073274612427, 0.14242573082447052, 0.6904853582382202, -0.4272526502609253, 0.15853969752788544, 0.6255903840065002, -0.3093465268611908, 0.4567752480506897, -0.6619259119033813, -0.3630538582801819, -0.3817378878593445, -1.1958396434783936, 0.48697248101234436, -0.46940234303474426, 0.012494118884205818, 0.48074665665626526, 0.44653117656707764, 0.6653851270675659, -0.09848548471927643, -0.42329829931259155, -0.0797182098031044, -1.687334418296814, -1.4471124410629272, -1.3226996660232544, -0.9972468018531799, 0.3997742235660553, -0.9054790735244751, -0.37816256284713745, 1.299228310585022, -0.3562639653682709, 0.7375155687332153, -0.9336176514625549, -0.2054375559091568, -0.9500220417976379, -0.3390330672264099, 0.8403081297874451, -1.7273204326629639, 0.43442365527153015, 0.2377355992794037, -0.5941499471664429, -1.4460577964782715, 0.0721295103430748, -0.5294927358627319, 0.2326762080192566, 0.021852144971489906, 1.6017788648605347, -0.23935562372207642, -1.023497462272644, 0.17927563190460205, 0.21999669075012207, 1.3591876029968262, 0.8351112604141235, 0.3568710684776306, 1.4633028507232666, -1.1887630224227905, -0.6397515535354614, -0.9265759587287903, -0.38980981707572937, -1.3766860961914062, 0.6351509690284729, -0.222222700715065, -1.4708062410354614, -1.015579104423523, 0.313513845205307, 0.838126540184021, 1.996730923652649, 2.913862466812134, 0.41440942883491516, -0.9895381331443787, -2.1320462226867676, 0.26771146059036255, -0.8129410743713379, -0.4153572618961334, -0.6120967864990234, -0.1407908797264099, 1.0659801959991455, 0.1570485681295395, -0.15863484144210815, -1.0356537103652954, -1.6746829748153687, -0.48630791902542114, -0.05378255248069763, 1.7679299116134644, 0.13027451932430267, 0.9827395081520081, -0.49929559230804443], [-1.1849437952041626, -0.9651167392730713, -0.7252260446548462, 2.128469705581665, -0.8213866949081421, 0.8384891748428345, -0.9029271602630615, 0.9315730333328247, 0.3849509656429291, -0.15663789212703705, -0.0407625250518322, -0.6547877192497253, 0.44607219099998474, -0.4549834728240967, -1.2256057262420654, -1.2779375314712524, 0.17258791625499725, 1.5790913105010986, 0.15999160706996918, -0.11863832920789719, 0.2858261466026306, 1.3060017824172974, 0.21938249468803406, -0.4109272360801697, 1.1062886714935303, 0.428756445646286, 1.53575599193573, 0.18323443830013275, -1.2244690656661987, -1.3681591749191284, 1.6509279012680054, 1.723665714263916, -0.17951920628547668, -0.38318732380867004, 1.4614442586898804, -1.1070456504821777, -0.8947269916534424, 0.6433268189430237, -0.3946051299571991, -0.0051218667067587376, -0.16344289481639862, 0.3375745415687561, 1.4074819087982178, 0.0905849039554596, 0.6439387798309326, -2.0501720905303955, -0.0487184002995491, -0.8432302474975586, -1.2188130617141724, -0.8781523704528809, -0.334123432636261, 0.9159025549888611, -1.3263927698135376, 0.030631491914391518, -0.4841694235801697, -0.3276731073856354, 1.0027577877044678, 0.5381154417991638, 1.3373980522155762, -0.1545056849718094, -0.6959426403045654, -0.2238588184118271, 0.2424967885017395, 0.17657335102558136, -1.084388017654419, 0.09048978239297867, 0.22822833061218262, 2.5174739360809326, 1.8768446445465088, -0.853243350982666, -0.2873833477497101, -1.4634419679641724, -0.590707004070282, 0.3156050145626068, 1.2058535814285278, -0.7290838360786438, -0.6541464328765869, -2.1472890377044678, -0.16266591846942902, -1.0624144077301025, -0.5294394493103027, -0.8768607974052429, -0.09426255524158478, -1.7577283382415771, -1.4670451879501343, 2.1292471885681152, -1.28742253780365, -1.096785545349121, 1.8369135856628418, 2.905067205429077, -1.1715666055679321, -0.36824896931648254, 0.34155556559562683, 1.728697657585144, -0.9868570566177368, -0.24527785181999207, 0.7773375511169434, 0.43476608395576477, -0.3761560618877411, -0.133822962641716], [-1.3748958110809326, -0.2381737381219864, -0.2663874924182892, 0.23216989636421204, -0.5553272366523743, 0.4715385138988495, 1.0127158164978027, 0.15542933344841003, 0.35175639390945435, 0.05315534770488739, 8.439309021923691e-05, -0.7215580344200134, 0.31649425625801086, -0.09728659689426422, 2.093168258666992, 1.573354959487915, 0.3858465552330017, -0.7630572319030762, -1.1124114990234375, 1.1911429166793823, 0.26274922490119934, 0.48014339804649353, -1.7445859909057617, 0.927438497543335, 0.4544203281402588, -1.1104307174682617, -0.47152480483055115, 0.2637172043323517, 0.052466798573732376, -0.2921711802482605, -0.1034882664680481, -0.2519773840904236, 0.15256251394748688, 1.4714919328689575, -2.5666584968566895, -0.23685026168823242, 0.17651242017745972, 0.29599398374557495, -0.3719145953655243, -1.7567217350006104, 0.3279954791069031, 1.7273502349853516, -1.5338613986968994, 0.8638280034065247, -0.32852521538734436, -0.0613243468105793, -1.0528985261917114, -0.3344561755657196, 1.3000445365905762, 0.5826552510261536, 1.7323116064071655, 1.1774119138717651, 0.4390866756439209, 1.7439345121383667, 0.4389931559562683, 0.8279881477355957, -0.29657095670700073, 0.06654581427574158, -0.6974238157272339, 0.9895839095115662, -1.178303599357605, 0.7823503613471985, -0.1906510591506958, 1.171247124671936, 0.7508689761161804, 1.8206461668014526, 0.7307747006416321, -1.5720402002334595, -0.06695317476987839, -1.1720072031021118, -0.5182798504829407, 1.5112284421920776, 0.6375337839126587, -0.6989304423332214, -1.0137174129486084, 0.032782092690467834, -1.2165601253509521, -0.6711402535438538, 0.3120094835758209, 1.1553120613098145, 0.6087614893913269, -2.2912895679473877, 0.30436673760414124, 0.07203357666730881, 0.4138902723789215, 1.616209626197815, -2.0632383823394775, -0.5911034345626831, 0.5909063220024109, -1.5815943479537964, 1.4759490489959717, 0.3683566153049469, 0.8465839624404907, -0.5709436535835266, 0.8137636780738831, 1.0684715509414673, 0.23287801444530487, 0.2344008982181549, 0.27034324407577515, -0.8633452653884888], [-0.14752867817878723, -0.15252253413200378, 0.3833938539028168, 0.9998242259025574, -1.0585360527038574, -0.12500903010368347, 1.4814555644989014, -0.7435882091522217, -0.8222500085830688, 0.20230619609355927, 0.8443852066993713, 0.011426055803894997, 1.3289605379104614, 0.856793999671936, 0.8418200612068176, 0.5541164875030518, 2.327653169631958, -0.2051616907119751, -2.0035223960876465, 1.6042543649673462, -0.45769941806793213, 0.10788044333457947, 1.3095506429672241, -1.6022595167160034, -1.2516472339630127, -1.6012779474258423, -0.7941362857818604, 0.43963661789894104, 0.5241878628730774, 0.276274174451828, -1.4127658605575562, -2.310103416442871, 0.054353583604097366, -0.4717760384082794, 0.45938578248023987, 0.701953649520874, 0.13824142515659332, 0.7601330876350403, 0.22921137511730194, 0.5300647020339966, -0.7046732902526855, -0.1796114146709442, 0.19677609205245972, 0.8205284476280212, -0.3937411606311798, 0.5211672782897949, -0.2658388018608093, -0.11754216998815536, 0.8295190334320068, -1.9930603504180908, -1.2964723110198975, -1.4821853637695312, -2.333616018295288, -0.6782644391059875, 0.7494338750839233, -0.2848840653896332, 0.19779008626937866, 1.0892175436019897, 1.3276861906051636, -0.06913793832063675, 1.3535858392715454, 0.09212666004896164, -0.8373982310295105, -0.5944003462791443, -1.4805364608764648, -0.8881338834762573, -0.358016699552536, 0.8035849928855896, 1.7207698822021484, -1.3821815252304077, 0.39282748103141785, -1.0405439138412476, 0.474697083234787, -0.13108664751052856, -1.830905795097351, 0.9282969832420349, -0.6050007343292236, -0.533900260925293, -1.0697524547576904, -0.6542832851409912, 0.42789044976234436, -0.189244344830513, 0.3286620080471039, 0.36192184686660767, 1.3206616640090942, -0.34278616309165955, -1.4768577814102173, 1.0672224760055542, -0.33148816227912903, 1.114592432975769, 0.3833771049976349, -0.13113753497600555, 0.3487758934497833, 1.9510126113891602, 2.0769805908203125, 0.06938113272190094, 0.16019059717655182, 1.0762401819229126, -0.8456610441207886, 0.3330703675746918], [-0.02586284838616848, 0.3139082193374634, -0.8333687782287598, -1.5895675420761108, -2.072983503341675, -1.1173840761184692, -0.4586752951145172, -0.2931915819644928, 1.937231183052063, 1.105993390083313, -0.9620910882949829, 0.34770846366882324, -0.40707823634147644, -0.28436383605003357, 0.18532565236091614, 0.619171142578125, -0.33925849199295044, 1.0638514757156372, -1.1419382095336914, 0.006339062470942736, 2.5976736545562744, 0.22307974100112915, 1.433214545249939, 0.09152017533779144, 0.580777108669281, -0.056783195585012436, -0.17040757834911346, -0.7794824242591858, 0.43030136823654175, -0.8515371680259705, 0.6655852198600769, 1.0852869749069214, 0.3665314018726349, -0.28624874353408813, 0.45396557450294495, -0.30867305397987366, 0.9355471134185791, -1.8314061164855957, -0.3356073796749115, -1.9908119440078735, -1.4950608015060425, 1.3638622760772705, 0.8951849937438965, -0.7194802165031433, -1.502503514289856, -2.964528799057007, -0.5434955358505249, 2.420414924621582, 0.43488427996635437, -0.5595722794532776, 0.465080201625824, -1.5609583854675293, -0.2973233759403229, 0.09947746992111206, -0.08610065281391144, 0.7908061146736145, 0.344645231962204, 0.6683260202407837, -0.6883722543716431, 0.8978154063224792, 1.628937005996704, -0.9701495170593262, -0.8876956701278687, 1.3357843160629272, -0.1913439929485321, 1.403821349143982, -0.4425356984138489, 1.4550455808639526, 0.13148581981658936, 0.2582288086414337, 1.5647180080413818, -0.36177048087120056, -0.9411221146583557, -0.44856420159339905, 0.4523339569568634, -1.5657590627670288, 0.6374709010124207, -0.5387713313102722, 1.1478126049041748, -2.3942604064941406, -0.7865657806396484, -1.6864681243896484, -0.826229453086853, 0.2476658970117569, -0.17922662198543549, -0.2533775568008423, -0.15918487310409546, 0.2033882439136505, -1.0085361003875732, 0.7068496346473694, 0.6626659631729126, 0.38503792881965637, 0.5565334558486938, 0.2964180111885071, 2.0350732803344727, -0.08709417283535004, -0.30708321928977966, -0.7535275816917419, -1.0322626829147339, -1.244471788406372], [-0.8887973427772522, -0.07068037986755371, 0.3342951238155365, 0.05114205926656723, -0.765535295009613, 0.9001845717430115, 0.7394126653671265, -0.15964831411838531, -0.6529161334037781, 0.548427939414978, 0.18797358870506287, -1.448127269744873, -0.06798025965690613, 0.2620358169078827, -0.8996948003768921, 0.18984338641166687, -1.4548225402832031, 1.3361860513687134, 1.2479499578475952, -0.25251734256744385, 0.36345434188842773, -2.409921884536743, -1.156347632408142, -0.2937789261341095, -1.0721330642700195, 0.7143964767456055, 1.9972965717315674, -1.176614761352539, -0.83746337890625, 0.23544837534427643, 1.6111161708831787, -1.2223743200302124, 0.24903611838817596, 1.8212988376617432, -1.651759147644043, -1.2810691595077515, -0.42360660433769226, -0.5205883979797363, 0.8126012682914734, 0.2416597157716751, -1.774962067604065, 0.5154104232788086, -0.5775389075279236, 1.2744472026824951, -0.6275875568389893, -0.6366152763366699, 0.5411316156387329, 0.762926459312439, 0.44809937477111816, -1.68559730052948, 0.5380344390869141, -1.0343080759048462, 0.23527611792087555, -1.423734426498413, 0.44632214307785034, -0.8065989017486572, -1.2826346158981323, 0.7138201594352722, 0.24164451658725739, -0.613976776599884, 1.4511789083480835, -0.44065243005752563, 0.032107673585414886, 0.2689134478569031, -0.6196659207344055, 0.47113630175590515, -0.5334523320198059, -0.4116383194923401, 1.3626426458358765, -1.040585994720459, -2.412780284881592, 1.6109369993209839, 2.549327850341797, -0.4052692651748657, -1.9368380308151245, -0.3104839622974396, -0.2862229645252228, -0.18992383778095245, -1.1133880615234375, 0.5795611143112183, 0.5245074033737183, -1.4944056272506714, 0.6991967558860779, 2.052685022354126, 0.17196033895015717, -0.33732515573501587, -0.14200320839881897, 0.6152567863464355, -1.7306716442108154, 0.1643906980752945, -0.3904639482498169, 1.847825050354004, -0.17417272925376892, 1.667887568473816, -1.1037406921386719, 0.5872591733932495, 0.31940025091171265, -0.869047224521637, 0.17739611864089966, 1.2125188112258911], [-0.3237917125225067, -1.6919625997543335, -0.017562827095389366, -0.902423083782196, -0.34234094619750977, -0.0815877690911293, -1.7056522369384766, -1.6156582832336426, 0.4820668399333954, -0.5227186679840088, -2.564744234085083, 0.7848440408706665, 0.27236974239349365, -0.7138748168945312, -1.3168301582336426, 0.8358079195022583, 0.34935063123703003, 2.3826022148132324, 0.4201886057853699, 0.38770315051078796, -0.16692793369293213, 0.8167758584022522, 0.625085175037384, 1.2517249584197998, -0.5213229060173035, -0.43540745973587036, -0.4791031777858734, 0.7908017039299011, 1.4983744621276855, -0.4588404893875122, -0.4247737228870392, 0.314077228307724, -0.24576149880886078, 0.9520536661148071, -2.251777172088623, -0.8267050385475159, -0.7824163436889648, -2.3203561305999756, -0.963638424873352, -0.9151561260223389, -0.20110465586185455, 1.1129659414291382, -0.24506065249443054, -1.0308136940002441, -0.05695454031229019, 1.0491735935211182, -0.9759588241577148, -0.9105749130249023, 0.5585489869117737, -0.22153611481189728, 0.6474847197532654, -0.013646791689097881, 0.7016636729240417, -1.0350781679153442, -0.012084650807082653, -0.21069002151489258, -1.215890884399414, -1.5634779930114746, 0.6857490539550781, -0.3509523868560791, -1.0222803354263306, -0.09617893397808075, 1.128018856048584, -2.28073787689209, -1.4966386556625366, -0.9228864312171936, 1.4611788988113403, 0.2825869917869568, 0.7673172354698181, -1.1401609182357788, -1.1195359230041504, 0.4478137195110321, 0.05827433243393898, 0.5487388372421265, -0.1876709908246994, 0.2781437337398529, 0.1581190824508667, 0.7777673602104187, 0.8070082664489746, -1.619871973991394, -2.2472684383392334, 1.001745343208313, 1.1877250671386719, -1.020622968673706, -1.8598353862762451, 0.09903482347726822, 0.9308382272720337, 1.7975945472717285, 0.5162975788116455, -0.37171670794487, -0.8931306004524231, 0.011451291851699352, -0.29926469922065735, -1.0150679349899292, 2.048755645751953, 1.7851684093475342, 1.136048674583435, -0.920850396156311, 0.8550193309783936, 0.6396263837814331], [0.442545622587204, 1.2496652603149414, 0.6353711485862732, 0.7400142550468445, 0.6369062662124634, 0.34079140424728394, -1.7836111783981323, 0.0836210772395134, -0.5561919808387756, -1.2798409461975098, 1.681816577911377, 1.7289953231811523, 1.3592206239700317, 0.2552134096622467, 1.3506251573562622, 0.012053180485963821, 0.2027972787618637, -1.0934714078903198, 0.3969913125038147, 0.06038592755794525, -1.30265212059021, -0.0511971116065979, -0.07972956448793411, 1.7975611686706543, 0.8942133188247681, 0.011445439420640469, 0.2487873136997223, 0.044212378561496735, -0.20291398465633392, -1.0824271440505981, -0.15105187892913818, -0.7460982799530029, -1.250315546989441, 0.5112218260765076, 0.3912646472454071, -1.7867075204849243, -0.12268463522195816, 0.9957013726234436, 1.0592237710952759, 1.0258368253707886, 0.03891320526599884, -0.8450453281402588, -1.0836988687515259, 0.3446049094200134, 0.3792800307273865, 1.2873311042785645, 1.0999796390533447, -0.13222865760326385, -1.2442058324813843, -0.3191066086292267, 0.21723119914531708, -0.20208324491977692, -0.5778940916061401, 0.25288480520248413, -0.5039577484130859, -0.6280660629272461, 0.31145283579826355, -0.4019925892353058, 0.24410438537597656, 0.2732040286064148, -1.139428973197937, -0.4812411367893219, 1.4377769231796265, -1.1620794534683228, -2.116654396057129, -1.8618451356887817, 0.029109947383403778, 0.030917277559638023, -0.1176108568906784, 1.214189887046814, -2.6728343963623047, 0.3959338665008545, 1.5614382028579712, -1.127780795097351, -0.3798065185546875, -0.7528918981552124, -0.8943460583686829, -0.3262619972229004, 1.4274852275848389, 1.8373867273330688, -0.3359392285346985, 1.9050601720809937, 0.0355764776468277, 1.7536910772323608, -0.09329865127801895, 0.13105782866477966, 0.3654744625091553, 3.178853750228882, 0.8512855768203735, -0.7072645425796509, 0.9689978957176208, -0.36183252930641174, -0.4897501766681671, 0.9086013436317444, 0.031085588037967682, 0.27857592701911926, 0.013985222205519676, 0.33658313751220703, 0.42496588826179504, -1.936966896057129], [0.6665726900100708, -0.9820194244384766, -1.4422991275787354, -0.05841124430298805, 0.0839809998869896, -0.6935061812400818, 0.831031858921051, -1.3421751260757446, -0.40694373846054077, -0.584874153137207, -0.0465872697532177, 0.2788642942905426, -1.0078929662704468, 0.7242826819419861, 0.06300237029790878, -1.8919461965560913, -1.9585914611816406, -0.01231929287314415, -0.22096732258796692, -0.10342011600732803, -0.027982456609606743, 0.22555984556674957, 0.9476495981216431, -1.1111048460006714, -1.1719715595245361, -1.093323826789856, 0.288916677236557, 1.2449474334716797, -0.43133944272994995, -2.501699924468994, -1.7039928436279297, -0.8330367207527161, -0.5577288866043091, -0.408388614654541, 0.03858112171292305, -0.3117463290691376, 1.0493396520614624, -0.6760172843933105, -0.8623179793357849, 0.4788755476474762, -1.5356420278549194, 0.38969171047210693, 0.10255318135023117, -0.14744248986244202, 1.5882761478424072, -0.6222074627876282, 2.060298204421997, -0.2254314422607422, -1.2770164012908936, 0.06991957873106003, -1.0762451887130737, -0.7517558336257935, 0.39703330397605896, 0.5555821061134338, -0.622167706489563, 0.9874051213264465, 1.1575077772140503, 1.4363017082214355, 0.5294134020805359, 1.3634287118911743, -1.8807984590530396, -0.31790655851364136, -0.8670052289962769, 0.11922585964202881, -0.5714492201805115, -0.16615784168243408, 1.8821748495101929, -0.16971980035305023, 0.41379231214523315, -0.23226933181285858, 0.07571328431367874, 0.006016680505126715, 0.448324054479599, 1.1653075218200684, 1.6473939418792725, 0.30962008237838745, 0.5895468592643738, -1.1508644819259644, -0.08787674456834793, 0.9402894377708435, 0.865968644618988, 0.21160973608493805, 0.8863939642906189, 0.490766704082489, 1.2003062963485718, 0.2893591523170471, -0.3556983172893524, 0.33584126830101013, -2.9305944442749023, 0.38288572430610657, -3.6484129428863525, -1.7234634160995483, 0.4517686069011688, 0.4775293469429016, -1.1624287366867065, -0.7121020555496216, 1.370540976524353, -0.4840301275253296, 2.242920398712158, -0.0019198531517758965], [0.40803617238998413, 1.6168744564056396, 0.13102711737155914, -1.002344012260437, -0.10972744971513748, -0.035610735416412354, -1.3647416830062866, -0.25583207607269287, -0.7421920895576477, 0.9243577718734741, 0.03461187332868576, -0.2827957570552826, -0.10618192702531815, 0.2231220155954361, 0.6168141961097717, -0.9997122287750244, -1.0415875911712646, 1.1046792268753052, -0.41233688592910767, -1.4168421030044556, 0.44381269812583923, 0.46336981654167175, -1.5307152271270752, 0.22948171198368073, 0.7355830669403076, 0.3743864595890045, 0.6319814920425415, -1.4042690992355347, 0.3310401439666748, -0.3026197552680969, -0.48279017210006714, 0.870555579662323, 1.4792745113372803, 1.7943700551986694, 1.3148078918457031, -0.1097341850399971, 0.35272017121315, 0.766822874546051, 0.12117794901132584, 0.1307641863822937, 0.8237531185150146, -0.05928264558315277, -0.7292869091033936, -0.4144730567932129, 0.6339103579521179, 0.0029932912439107895, 0.3402099907398224, 0.670079231262207, -0.37484145164489746, 0.7562481760978699, 0.3788425922393799, -1.2348130941390991, 1.4423048496246338, -0.5007447600364685, -1.6550953388214111, -1.045043706893921, -1.021045207977295, 0.052173320204019547, -0.2738505005836487, -0.3368319571018219, 0.6197019815444946, 0.33987537026405334, 0.31604787707328796, 0.40982845425605774, 0.6161346435546875, -2.1079533100128174, -0.3644382655620575, -2.1802101135253906, 0.03605992719531059, -0.004632972180843353, 1.0455322265625, 1.1876343488693237, 0.2027750462293625, -0.5003610849380493, 0.4851611256599426, -0.5279176235198975, -0.0013926514657214284, 0.9861363172531128, -0.5577713251113892, 0.80567866563797, 0.6774013638496399, -0.9547918438911438, 0.9738941192626953, 0.6985663175582886, 0.10192588716745377, -0.7623234391212463, -0.8592060208320618, -0.5376625657081604, 0.5425944924354553, -0.9556251764297485, 0.43751224875450134, -1.241755485534668, -0.20406858623027802, 0.10964775085449219, 2.4451301097869873, -1.377315640449524, 1.4720145463943481, 0.14984241127967834, 0.41117191314697266, 0.11834791302680969], [0.444726824760437, -0.15368510782718658, 1.4540501832962036, -0.45647114515304565, 1.1322345733642578, -0.6443670988082886, -0.06025184690952301, -1.071962594985962, 0.45502370595932007, 1.445125937461853, -0.07735618203878403, -0.19688570499420166, -1.1146169900894165, -0.22929266095161438, -1.5927942991256714, -0.9128779768943787, 0.2267860472202301, 1.319013237953186, 2.809210777282715, -0.5865851044654846, 1.4353001117706299, 0.24375247955322266, -0.15124765038490295, 0.4325944483280182, 0.06191648542881012, 0.11039572209119797, -0.4083331525325775, -1.3981096744537354, -1.5436252355575562, 0.6532449126243591, -0.27670007944107056, -0.5960888862609863, 0.008505626581609249, 0.7949322462081909, 0.18036393821239471, -0.6560549139976501, 1.2262929677963257, 1.5791854858398438, 0.4945566654205322, 0.9736635088920593, 1.241960048675537, 1.1300992965698242, 0.6140992045402527, 0.5983470678329468, 0.5196605920791626, -1.0973377227783203, 0.7007231712341309, -1.355813980102539, -0.7945994734764099, 1.3035650253295898, 0.8402201533317566, 1.4873982667922974, -0.27167439460754395, -1.1522488594055176, -0.24043719470500946, 0.10203129798173904, 0.07898871600627899, 1.1336804628372192, -0.3612704277038574, 0.35203447937965393, -0.9886271357536316, 0.45031726360321045, 0.0031085500959306955, -0.7498133182525635, -0.23582936823368073, -0.1841757446527481, -0.270155131816864, 1.7713412046432495, -0.09844600409269333, -0.24389611184597015, -2.0974509716033936, -0.8942073583602905, -0.2630734145641327, -0.6858526468276978, 1.3817075490951538, -0.16493777930736542, 1.2884442806243896, 0.06170463562011719, 0.03722783550620079, -0.08879644423723221, 0.0037962517235428095, 1.7187950611114502, -2.319610834121704, -2.0015456676483154, -0.5431708097457886, 0.014527881518006325, 0.6901044249534607, 0.4730871915817261, -0.384245902299881, 1.0190436840057373, 1.0301953554153442, 0.1840578317642212, 0.9626575708389282, 0.2726452648639679, -0.5614830255508423, 0.6978168487548828, 0.11061312258243561, 0.0013398467563092709, 1.4729535579681396, -2.4508721828460693], [-1.4176841974258423, -1.187070369720459, -0.3632606565952301, -0.2546081840991974, -1.5073256492614746, -0.9851580262184143, -0.860841691493988, 2.4574241638183594, 1.8017419576644897, -0.4117494225502014, -0.3635832965373993, -1.1492040157318115, -1.908130407333374, -0.11688018590211868, -0.9978519678115845, -0.08485053479671478, -1.600205898284912, -0.7619744539260864, 0.14862710237503052, 0.36620959639549255, 0.4174719452857971, -1.3204889297485352, 0.8546857237815857, -0.8002116680145264, 0.6328579187393188, -0.010638676583766937, -1.3763874769210815, -0.3161976933479309, 0.365363746881485, 0.6129654049873352, -0.14098790287971497, 1.5317789316177368, 1.007510781288147, -0.25665411353111267, 0.7501218318939209, 1.933793544769287, 1.9604846239089966, -1.2279999256134033, -0.9265331029891968, 1.4849188327789307, -1.0582810640335083, -1.3225411176681519, -0.4861941337585449, 0.42022672295570374, -0.10239670425653458, -0.6505635976791382, -0.6742124557495117, -0.712337076663971, -0.8795096278190613, 2.281632900238037, 0.2975110709667206, 0.8867590427398682, -0.48907747864723206, -0.185956671833992, -0.7135539650917053, -2.6517083644866943, -1.37800931930542, -1.8106186389923096, -2.249783754348755, -1.1953589916229248, 1.324955701828003, -0.04443616792559624, 1.2905638217926025, 0.41098257899284363, 0.782558798789978, -0.9008969664573669, 0.5239509344100952, 0.7287142276763916, -0.5766472816467285, 0.5595403909683228, 0.5661877393722534, -0.5495526194572449, -1.1227684020996094, -1.1841282844543457, 0.08650553971529007, 0.3270098567008972, -0.7925258278846741, 0.02658534049987793, 0.5705370306968689, 0.6181907057762146, 1.551880121231079, 1.2769808769226074, -1.0012913942337036, 1.6835862398147583, -0.5309765338897705, 1.044872760772705, 0.0679866224527359, -0.4121493697166443, -1.80769681930542, -0.1715645045042038, -1.5593832731246948, 0.967299222946167, 1.51679265499115, -0.7969110012054443, 0.3019707500934601, -0.7253915071487427, -0.6284847259521484, 0.7744947075843811, -0.038538262248039246, 1.7396479845046997], [-0.4822324812412262, 0.9993363618850708, 0.09741341322660446, 0.7957780361175537, -0.4471576511859894, -0.049558836966753006, -0.11549887806177139, -0.8360571265220642, 0.6620815396308899, 0.6502490043640137, 0.593554675579071, 1.5369656085968018, 1.4493447542190552, -0.37494468688964844, 0.37305232882499695, -0.6236006617546082, 0.02335531637072563, -0.5997494459152222, 1.6200703382492065, 0.36369815468788147, -0.21802912652492523, 1.1936185359954834, -2.1706154346466064, -2.0141026973724365, 0.7958386540412903, 0.06046844646334648, 0.25044628977775574, -1.326396107673645, -0.02323371171951294, 2.003509521484375, 0.8241996169090271, 0.1697065383195877, -0.4390539228916168, -0.3945728540420532, -2.129197835922241, 0.25764042139053345, 0.845483124256134, -1.5756193399429321, -0.6614168882369995, -1.157464623451233, -0.9653512239456177, 0.05348162725567818, -2.084402084350586, 0.6142704486846924, 0.7538550496101379, -0.25110259652137756, -2.480708599090576, -0.9964186549186707, 1.232901930809021, -2.777994394302368, -0.34752342104911804, -1.1800007820129395, 0.8045703172683716, -0.6751139760017395, 0.40395402908325195, 0.5654604434967041, 1.8362258672714233, -0.20294193923473358, 0.376044362783432, -1.4850554466247559, 1.1904135942459106, -0.7605088353157043, -0.5607541799545288, -0.0222477950155735, -1.5628843307495117, 0.22877003252506256, 0.9674662947654724, 0.2929338216781616, -1.5888975858688354, -0.12380929291248322, 0.7420288920402527, -1.8779888153076172, -1.0747660398483276, 0.8765435814857483, 0.27009543776512146, 0.4119715690612793, 1.8594948053359985, 0.46855276823043823, 0.6578879952430725, 2.7107455730438232, -0.21380455791950226, -0.7978827357292175, 0.4277963936328888, 0.4389910399913788, -0.5267153978347778, -1.0323970317840576, 2.309844493865967, 0.3268434703350067, -0.7322853803634644, 0.9159303307533264, -0.1091451495885849, -0.2528814375400543, 0.562272846698761, -1.2918822765350342, 0.3526272177696228, 1.9129548072814941, 0.46617767214775085, -0.37621209025382996, -0.50710529088974, -0.41033998131752014], [-0.2539626657962799, -0.37780654430389404, 0.9163114428520203, -1.7332091331481934, 0.862000048160553, -0.37968653440475464, 0.3769477903842926, -1.1269879341125488, 1.328631043434143, 1.245634913444519, 0.9595905542373657, -1.5147547721862793, 0.8291210532188416, 0.4048551023006439, -1.6052712202072144, -0.024297384545207024, 0.36453449726104736, 0.5560752749443054, 0.1772613674402237, 0.2912314236164093, 1.4736113548278809, 1.2260339260101318, -2.8670549392700195, -0.317438542842865, -0.164651021361351, -1.752098560333252, 0.0941830575466156, 1.2487577199935913, -1.0865678787231445, 0.33640092611312866, -0.9158234596252441, -0.6719120144844055, 1.477393627166748, -0.5425578355789185, 0.4622125029563904, -1.8712180852890015, 1.8623175621032715, 0.6026524305343628, -0.18249984085559845, 0.5755465030670166, -1.409819483757019, 1.1825307607650757, -0.3239847719669342, -0.20406951010227203, -0.4916556179523468, -0.5806514024734497, 0.6946706175804138, 0.2559068500995636, 0.6517481207847595, -0.008794689550995827, 0.5131998062133789, -0.41884875297546387, 2.229731798171997, -1.5679304599761963, 1.2609373331069946, 1.3679074048995972, -0.852158784866333, 2.229273557662964, 0.995490312576294, 0.9711390137672424, 0.02802218683063984, -0.21722616255283356, -0.20940881967544556, 0.3905305862426758, 1.401186227798462, 0.18131421506404877, -0.9114436507225037, 0.82442706823349, 0.9211507439613342, 0.6546595096588135, 0.7454988956451416, -0.27092814445495605, -0.9289891719818115, -0.8681934475898743, -1.6354632377624512, 0.23694860935211182, -0.08344878256320953, -0.4399877190589905, -2.9556188583374023, -1.2473174333572388, 1.1208406686782837, -0.6646260023117065, 0.36059167981147766, -1.3906854391098022, 1.4387054443359375, -0.14402766525745392, 0.29004040360450745, 2.325779676437378, 1.5164401531219482, -0.30656561255455017, -0.5932201743125916, -0.3255450129508972, 0.5098982453346252, -0.1052241325378418, -0.39524415135383606, 1.4671812057495117, -0.11420989036560059, 2.299589157104492, 0.0907425656914711, 1.785318374633789], [0.3573194146156311, 0.9629606008529663, -0.7246110439300537, -0.853759229183197, 0.4597059488296509, 0.9885532259941101, 0.040256500244140625, -1.2863932847976685, -1.3602583408355713, -1.4271364212036133, -1.7676221132278442, 0.10948535799980164, 1.1250438690185547, -0.013415787369012833, -2.2604148387908936, -0.19076046347618103, -0.16279900074005127, 2.095998764038086, -1.457137942314148, -0.3906776010990143, 1.7058526277542114, 0.4499783515930176, 0.5633130669593811, 0.11458443850278854, 0.32994353771209717, 0.7776095271110535, -0.8094863295555115, 0.8780585527420044, -1.3730030059814453, 0.3283679783344269, 0.1787409633398056, -0.6162930727005005, -0.022685274481773376, -0.6225401759147644, -1.0717012882232666, -0.3536147475242615, -1.1039886474609375, 0.3227974772453308, -1.15008544921875, 0.7119770050048828, -1.1814790964126587, -0.5662949085235596, -0.6246033906936646, 1.3251466751098633, 0.33028995990753174, -0.21173380315303802, 0.4987194538116455, -2.107192277908325, -0.04313359037041664, 1.9975141286849976, 0.132346510887146, 0.06791316717863083, 0.5481283664703369, 0.8956052660942078, -1.9974666833877563, -1.6823264360427856, 0.2102912962436676, -0.7148765325546265, 1.3120253086090088, -0.315174400806427, -1.9345307350158691, -1.2680139541625977, -2.749675989151001, -0.2341255396604538, -0.18838413059711456, -0.4244627058506012, 1.0788789987564087, 1.1985198259353638, -0.17038843035697937, -2.1346936225891113, 0.6172815561294556, 0.5553602576255798, 0.3093915581703186, 0.9814797639846802, -1.1648280620574951, 2.0087554454803467, -0.9594277739524841, 0.7492229342460632, -1.687347650527954, -0.9302647113800049, 1.5163686275482178, -0.2608998119831085, 0.8399226069450378, 0.23695538938045502, -0.10495815426111221, 1.3638503551483154, -0.23665882647037506, -1.780738115310669, -0.8032401204109192, 0.5511917471885681, 0.39550909399986267, -1.5670582056045532, -0.7243425250053406, -0.5817213654518127, -0.3654696047306061, -1.3759136199951172, -0.6283711194992065, -0.7286104559898376, 0.7715332508087158, 0.809121310710907], [-0.14581045508384705, -0.39203980565071106, 0.1399243175983429, -0.8165751099586487, -1.1468321084976196, -0.9402211904525757, -0.6348131895065308, -1.1634738445281982, 0.16205956041812897, 0.7297156453132629, -1.5870983600616455, 0.025128711014986038, -0.21993578970432281, 0.341508686542511, -0.012938767671585083, -1.4535993337631226, -1.2623893022537231, 0.029423575848340988, 1.0287809371948242, -2.2141873836517334, 0.3457265794277191, 1.1896220445632935, -0.15302588045597076, 0.550202488899231, -0.7030690908432007, -0.11936619877815247, -1.6209135055541992, -2.344343423843384, 0.9427657723426819, -0.9849949479103088, -0.4555278420448303, -1.3864423036575317, -1.409969687461853, -0.17383770644664764, -0.7295771837234497, 0.37797099351882935, 0.4525384306907654, 0.3208496570587158, 0.23860079050064087, 1.5777682065963745, -1.0675938129425049, -0.05493486672639847, 1.1326898336410522, 1.2645398378372192, -1.4217636585235596, -0.3123554289340973, 1.3581210374832153, 0.5080791115760803, 0.19526302814483643, -0.21768468618392944, -0.3838009238243103, 0.28821906447410583, -0.2523741126060486, -0.042578622698783875, 0.07859870791435242, -0.6780714988708496, 0.5321293473243713, 2.9142446517944336, -1.7769865989685059, 0.32401251792907715, -0.7601900696754456, 0.5774025917053223, 0.0465574748814106, 0.7618547677993774, 1.175506591796875, 0.19242112338542938, 1.1320523023605347, -2.3442978858947754, 1.5904523134231567, 0.37506213784217834, 1.0778433084487915, -1.219875693321228, -0.8307698965072632, 0.14593829214572906, -0.0170378889888525, 0.28306859731674194, 1.0845152139663696, -1.765220284461975, 0.9738190770149231, 0.008281909860670567, -0.203781858086586, -1.471115231513977, 0.5030486583709717, -2.4141077995300293, 0.648393988609314, -1.4921058416366577, 0.05345791205763817, -0.11916884034872055, 1.169538140296936, -0.17865516245365143, -0.6941385865211487, -0.5066027045249939, 1.2728828191757202, 0.5362969636917114, -0.3856299817562103, -0.23644642531871796, -1.5671770572662354, 0.3884741961956024, -1.7598156929016113, 0.7517964839935303]], "dtype": "float32"}], "e": {"__nd__": [[0.10675670951604843, -0.18554338812828064, -0.03692246600985527, 0.017944201827049255, 0.06463345140218735, 0.07841568440198898, 0.08746983110904694, -0.03844838961958885, -0.05097620561718941, 0.09459540992975235, -0.021092096343636513, -0.1406497061252594, -0.1249496191740036, -0.10137343406677246, 0.054814036935567856, 0.015703028067946434, 0.07612887769937515, -0.04710637778043747, 0.017479659989476204, 0.06897393614053726, -0.03410673886537552, 0.05036136880517006, -0.07298008352518082, -0.04002819582819939, -0.042088188230991364, -0.13184629380702972, 0.05369073897600174, -0.051753558218479156, 0.0013775285333395004, 0.05300431698560715, 0.049231916666030884, 0.07336147129535675, -0.010858433321118355, -0.046670395880937576, -0.008789263665676117, -0.1860356330871582, -0.15955016016960144, -0.14583313465118408, -0.10995060205459595, 0.044076766818761826, -0.09983282536268234, -0.04169399291276932, 0.14324532449245453, -0.03927958011627197, 0.08131415396928787, -0.10293522477149963, -0.02265034429728985, -0.10474387556314468, -0.037379804998636246, 0.09264746308326721, -0.1904442459344864, 0.047897011041641235, 0.026211338117718697, -0.06550750136375427, -0.15943387150764465, 0.007952578365802765, -0.05837877467274666, 0.025653518736362457, 0.002409289823845029, 0.17660276591777802, -0.02638995088636875, -0.11284484714269638, 0.019765883684158325, 0.024255549535155296, 0.1498560756444931, 0.09207448363304138, 0.039346516132354736, 0.16133521497249603, -0.13106605410575867, -0.07053526490926743, -0.10215884447097778, -0.04297815263271332, -0.15178535878658295, 0.07002802938222885, -0.024500975385308266, -0.16216249763965607, -0.11197181791067123, 0.034566204994916916, 0.09240693598985672, 0.22014787793159485, 0.32126542925834656, 0.04569036141037941, -0.10910069197416306, -0.2350669503211975, 0.029516300186514854, -0.089630126953125, -0.04579486325383186, -0.06748621165752411, -0.01552277896553278, 0.11752874404191971, 0.017315257340669632, -0.017490150406956673, -0.11418512463569641, -0.18464075028896332, -0.05361746624112129, -0.005929749924689531, 0.194921612739563, 0.014363306574523449, 0.10835111141204834, -0.055049411952495575], [-0.11390037089586258, -0.09276993572711945, -0.06971091777086258, 0.2045949250459671, -0.07895416766405106, 0.08059810847043991, -0.08679208159446716, 0.0895456075668335, 0.037002649158239365, -0.015056506730616093, -0.003918217029422522, -0.06294017285108566, 0.04287780448794365, -0.04373438283801079, -0.11780892312526703, -0.12283921241760254, 0.01658967137336731, 0.15178702771663666, 0.015378875657916069, -0.01140387449413538, 0.027474472299218178, 0.12553682923316956, 0.021087707951664925, -0.039499565958976746, 0.10633980482816696, 0.04121336340904236, 0.1476214975118637, 0.01761304773390293, -0.11769966036081314, -0.13151158392429352, 0.1586921662092209, 0.16568395495414734, -0.017255928367376328, -0.03683312237262726, 0.14047843217849731, -0.10641256719827652, -0.0860038548707962, 0.06183851510286331, -0.03793063759803772, -0.0004923293017782271, -0.015710623934864998, 0.03244868293404579, 0.13529141247272491, 0.00870729424059391, 0.061897337436676025, -0.1970687210559845, -0.004682959523051977, -0.08105383813381195, -0.11715599149465561, -0.0844106525182724, -0.032116953283548355, 0.08803931623697281, -0.1274968683719635, 0.002944391453638673, -0.046539824455976486, -0.03149692714214325, 0.09638810157775879, 0.05172527953982353, 0.12855473160743713, -0.014851552434265614, -0.06689610332250595, -0.021517984569072723, 0.02330952323973179, 0.01697276346385479, -0.10423465073108673, 0.008698150515556335, 0.02193799614906311, 0.2419871836900711, 0.18040797114372253, -0.08201632648706436, -0.027624154463410378, -0.14067046344280243, -0.05678053945302963, 0.030336907133460045, 0.11591028422117233, -0.07008174061775208, -0.06287852674722672, -0.20640389621257782, -0.015635939314961433, -0.1021224781870842, -0.050891317427158356, -0.08428650349378586, -0.00906080100685358, -0.1689581573009491, -0.14101681113243103, 0.20466966927051544, -0.12375093996524811, -0.10542633384466171, 0.17656967043876648, 0.2792438268661499, -0.11261451989412308, -0.03539720177650452, 0.032831352204084396, 0.16616763174533844, -0.09485967457294464, -0.023576846346259117, 0.07472003251314163, 0.041791029274463654, -0.03615725412964821, -0.012863466516137123], [-0.14174029231071472, -0.024553725495934486, -0.027462327852845192, 0.023934779688715935, -0.057249605655670166, 0.04861168563365936, 0.10440254956483841, 0.01602346822619438, 0.03626314923167229, 0.005479872692376375, 8.700223588675726e-06, -0.0743866115808487, 0.03262791782617569, -0.010029436089098454, 0.21578818559646606, 0.16219976544380188, 0.03977756202220917, -0.07866483181715012, -0.11468034237623215, 0.12279690057039261, 0.027087252587080002, 0.0494987778365612, -0.17985226213932037, 0.09561116993427277, 0.04684694483876228, -0.11447614431381226, -0.048610273748636246, 0.02718704380095005, 0.005408889148384333, -0.03012041188776493, -0.01066877692937851, -0.025976765900850296, 0.015727922320365906, 0.1516985446214676, -0.26460108160972595, -0.02441728673875332, 0.01819695718586445, 0.030514510348439217, -0.03834129124879837, -0.18110334873199463, 0.03381359577178955, 0.1780754029750824, -0.1581283062696457, 0.08905345946550369, -0.033868208527565, -0.006322028581053019, -0.10854505747556686, -0.034479640424251556, 0.1340237408876419, 0.06006689369678497, 0.17858687043190002, 0.12138134241104126, 0.04526617377996445, 0.1797851026058197, 0.04525653272867203, 0.08535867184400558, -0.030573992058634758, 0.006860318128019571, -0.07189857959747314, 0.102017842233181, -0.1214732676744461, 0.08065379410982132, -0.01965453289449215, 0.12074580788612366, 0.0774083286523819, 0.18769343197345734, 0.07533677667379379, -0.16206422448158264, -0.00690231379121542, -0.1208241656422615, -0.053430330008268356, 0.15579503774642944, 0.06572441756725311, -0.07205389440059662, -0.10450580716133118, 0.0033795603085309267, -0.12541720271110535, -0.06918895989656448, 0.03216557577252388, 0.11910303682088852, 0.06275822967290878, -0.2362128347158432, 0.03137767314910889, 0.007426060736179352, 0.04266863316297531, 0.1666177213191986, -0.2127026617527008, -0.06093783304095268, 0.06091751158237457, -0.16304917633533478, 0.15215803682804108, 0.03797449544072151, 0.08727574348449707, -0.05885952711105347, 0.08389224857091904, 0.11015050113201141, 0.024007780477404594, 0.024164777249097824, 0.02787013351917267, -0.08900369703769684], [-0.01492142491042614, -0.01542651653289795, 0.0387774296104908, 0.10112476348876953, -0.10706302523612976, -0.012643731199204922, 0.1498381793498993, -0.0752084031701088, -0.08316445350646973, 0.020461762323975563, 0.08540326356887817, 0.001155660254880786, 0.1344144493341446, 0.08665832132101059, 0.08514381945133209, 0.05604474991559982, 0.23542475700378418, -0.02075057476758957, -0.2026413381099701, 0.1622583568096161, -0.04629288241267204, 0.010911301709711552, 0.13245128095149994, -0.1620565950870514, -0.12659478187561035, -0.16195732355117798, -0.08032096177339554, 0.044465962797403336, 0.053017690777778625, 0.027943070977926254, -0.14289072155952454, -0.23364973068237305, 0.005497459322214127, -0.04771662876009941, 0.04646344482898712, 0.07099737226963043, 0.01398208923637867, 0.0768817886710167, 0.023183021694421768, 0.05361209064722061, -0.07127244770526886, -0.018166353926062584, 0.019902434200048447, 0.08299033343791962, -0.0398239828646183, 0.05271218344569206, -0.02688761241734028, -0.011888513341546059, 0.08389966189861298, -0.20158319175243378, -0.13112850487232208, -0.1499119997024536, -0.23602785170078278, -0.06860139220952988, 0.07579964399337769, -0.02881389856338501, 0.020004991441965103, 0.11016622930765152, 0.1342855542898178, -0.006992786657065153, 0.13690510392189026, 0.00931792426854372, -0.08469658344984055, -0.060119159519672394, -0.14974522590637207, -0.08982811868190765, -0.036210719496011734, 0.08127662539482117, 0.17404304444789886, -0.13979734480381012, 0.03973156958818436, -0.1052432581782341, 0.04801207035779953, -0.013258436694741249, -0.18518246710300446, 0.09389031678438187, -0.061191312968730927, -0.05400002747774124, -0.10819748044013977, -0.06617587059736252, 0.04327792674303055, -0.01914065331220627, 0.03324171155691147, 0.03660569339990616, 0.13357508182525635, -0.03467026352882385, -0.1493731439113617, 0.10794159024953842, -0.03352755308151245, 0.11273270845413208, 0.038775734603405, -0.013263583183288574, 0.035276081413030624, 0.19733037054538727, 0.21007108688354492, 0.0070173838175833225, 0.016202084720134735, 0.10885366797447205, -0.0855323076248169, 0.03368758410215378], [-0.0025017205625772476, 0.03036442957818508, -0.08061199635267258, -0.15375931560993195, -0.20052026212215424, -0.10808487236499786, -0.04436778649687767, -0.028360502794384956, 0.18738891184329987, 0.10698304325342178, -0.0930633395910263, 0.03363393619656563, -0.039376791566610336, -0.027506591752171516, 0.017926601693034172, 0.05989259481430054, -0.0328165665268898, 0.10290664434432983, -0.11045999825000763, 0.0006131792906671762, 0.2512736916542053, 0.02157856523990631, 0.13863523304462433, 0.008852771483361721, 0.056178729981184006, -0.005492654163390398, -0.016483571380376816, -0.07539954781532288, 0.04162317141890526, -0.08236942440271378, 0.06438224017620087, 0.10498011112213135, 0.035454683005809784, -0.027688920497894287, 0.043912217020988464, -0.029858030378818512, 0.09049572795629501, -0.17715242505073547, -0.03246339410543442, -0.19257178902626038, -0.14461764693260193, 0.1319267749786377, 0.08659148961305618, -0.06959552317857742, -0.14533758163452148, -0.28675970435142517, -0.052572473883628845, 0.23412740230560303, 0.04206647723913193, -0.05412758141756058, 0.04498733580112457, -0.15099193155765533, -0.028760172426700592, 0.009622483514249325, -0.008328540250658989, 0.07649489492177963, 0.033337630331516266, 0.0646473616361618, -0.0665864422917366, 0.08684593439102173, 0.15756753087043762, -0.09384283423423767, -0.08586704730987549, 0.12921078503131866, -0.018508756533265114, 0.13579203188419342, -0.04280659928917885, 0.14074696600437164, 0.012718659825623035, 0.02497854456305504, 0.15135560929775238, -0.034994158893823624, -0.09103500097990036, -0.043389737606048584, 0.04375438764691353, -0.1514563113451004, 0.061662737280130386, -0.05211549997329712, 0.11102823168039322, -0.23159746825695038, -0.07608472555875778, -0.16313253343105316, -0.07992140203714371, 0.023956790566444397, -0.017336640506982803, -0.024509280920028687, -0.0153979966416955, 0.019673801958560944, -0.097555972635746, 0.06837376207113266, 0.06409986317157745, 0.037244826555252075, 0.05383363366127014, 0.0286725964397192, 0.19685320556163788, -0.00842464342713356, -0.029704244807362556, -0.07288893312215805, -0.09985105693340302, -0.12037809938192368], [-0.08658026903867722, -0.006885176058858633, 0.03256463631987572, 0.00498189264908433, -0.07457295805215836, 0.08768952637910843, 0.0720282793045044, -0.015551794320344925, -0.06360241025686264, 0.0534239187836647, 0.018311040475964546, -0.14106617867946625, -0.006622149609029293, 0.02552565187215805, -0.08764182031154633, 0.018493181094527245, -0.14171838760375977, 0.13016167283058167, 0.12156634032726288, -0.02459842897951603, 0.03540511801838875, -0.23475731909275055, -0.11264310032129288, -0.02861783653497696, -0.10443951934576035, 0.06959138065576553, 0.19456231594085693, -0.11461737751960754, -0.08157967776060104, 0.022935692220926285, 0.1569433957338333, -0.11907494068145752, 0.024259313941001892, 0.17741787433624268, -0.16090252995491028, -0.12479257583618164, -0.041264720261096954, -0.050711989402770996, 0.07915779203176498, 0.023540757596492767, -0.1729040890932083, 0.05020758882164955, -0.056259699165821075, 0.12414751201868057, -0.061135079711675644, -0.06201449781656265, 0.05271316319704056, 0.07431882619857788, 0.04365063086152077, -0.16419880092144012, 0.052411459386348724, -0.10075487941503525, 0.022918913513422012, -0.1386900097131729, 0.04347750544548035, -0.0785730853676796, -0.12494506686925888, 0.06953524053096771, 0.02353927679359913, -0.05980921536684036, 0.14136344194412231, -0.04292520135641098, 0.0031276994850486517, 0.026195621117949486, -0.06036341190338135, 0.045894719660282135, -0.051965102553367615, -0.040098853409290314, 0.1327388733625412, -0.10136643052101135, -0.2350357621908188, 0.15692593157291412, 0.24833723902702332, -0.03947842866182327, -0.18867288529872894, -0.03024512156844139, -0.027881789952516556, -0.01850101910531521, -0.10845828056335449, 0.05645668879151344, 0.051093753427267075, -0.14557428658008575, 0.06811073422431946, 0.19995786249637604, 0.01675114408135414, -0.03285979852080345, -0.013832935132086277, 0.05993390455842018, -0.16858962178230286, 0.01601376384496689, -0.038036197423934937, 0.1800018697977066, -0.01696665957570076, 0.1624736487865448, -0.1075185090303421, 0.05720657855272293, 0.031113682314753532, -0.08465635031461716, 0.017280658707022667, 0.1181148886680603], [-0.03098457306623459, -0.1619088351726532, -0.001680638175457716, -0.08635549992322922, -0.03275960311293602, -0.00780737167224288, -0.16321884095668793, -0.15460705757141113, 0.046130381524562836, -0.050020474940538406, -0.24542786180973053, 0.0751040130853653, 0.02606385573744774, -0.06831276416778564, -0.12601132690906525, 0.07998089492321014, 0.03343038260936737, 0.22799815237522125, 0.040209073573350906, 0.03710044547915459, -0.01597382128238678, 0.07815966755151749, 0.05981622263789177, 0.11978121101856232, -0.04988690838217735, -0.04166540876030922, -0.04584678262472153, 0.07567412406206131, 0.14338383078575134, -0.043907783925533295, -0.04064783826470375, 0.030054965987801552, -0.023517634719610214, 0.09110479056835175, -0.21547912061214447, -0.07910981774330139, -0.07487170398235321, -0.22204165160655975, -0.09221336990594864, -0.08757396042346954, -0.01924429088830948, 0.1065029576420784, -0.023450568318367004, -0.09864156693220139, -0.005450146272778511, 0.1003984808921814, -0.09339234977960587, -0.08713556826114655, 0.05344918370246887, -0.021199436858296394, 0.061959702521562576, -0.0013059013290330768, 0.0671442449092865, -0.0990496501326561, -0.0011564154410734773, -0.020161543041467667, -0.1163521483540535, -0.14961376786231995, 0.06562133133411407, -0.033583659678697586, -0.09782499074935913, -0.00920364260673523, 0.10794341564178467, -0.21825046837329865, -0.14321771264076233, -0.08831369876861572, 0.13982446491718292, 0.027041573077440262, 0.07342682778835297, -0.10910532623529434, -0.10713165998458862, 0.042852602899074554, 0.005576441064476967, 0.052510421723127365, -0.017958784475922585, 0.026616385206580162, 0.015130876563489437, 0.07442682981491089, 0.0772249773144722, -0.15501026809215546, -0.2150476723909378, 0.09585993736982346, 0.11365687847137451, -0.09766639024019241, -0.17797307670116425, 0.00947693083435297, 0.08907462656497955, 0.17201706767082214, 0.0494060218334198, -0.0355706550180912, -0.08546627312898636, 0.0010958075290545821, -0.028637511655688286, -0.09713481366634369, 0.19605140388011932, 0.1708279699087143, 0.10871180891990662, -0.08811885863542557, 0.08181929588317871, 0.061207715421915054], [0.04415062069892883, 0.1246730163693428, 0.06338788568973541, 0.07382761687040329, 0.06354103237390518, 0.03399910032749176, -0.1779422014951706, 0.008342467248439789, -0.05548856407403946, -0.12768349051475525, 0.16778664290905, 0.17249344289302826, 0.13560281693935394, 0.02546139806509018, 0.1347452998161316, 0.0012024871539324522, 0.020232096314430237, -0.10909031331539154, 0.03960588946938515, 0.006024410016834736, -0.12995925545692444, -0.005107686389237642, -0.007954230532050133, 0.17933392524719238, 0.08921130746603012, 0.001141855726018548, 0.024820297956466675, 0.004410853609442711, -0.02024373970925808, -0.10798848420381546, -0.015069710090756416, -0.07443459331989288, -0.12473789602518082, 0.05100211128592491, 0.03903456777334213, -0.17825110256671906, -0.012239648960530758, 0.09933627396821976, 0.10567359626293182, 0.10234274715185165, 0.0038821808993816376, -0.08430605381727219, -0.10811536014080048, 0.03437955304980278, 0.03783892095088959, 0.12843075394630432, 0.10973960906267166, -0.013191808946430683, -0.12412835657596588, -0.03183571249246597, 0.021672097966074944, -0.020160861313343048, -0.05765368044376373, 0.025229085236787796, -0.050277408212423325, -0.06265909224748611, 0.031072132289409637, -0.04010484367609024, 0.024353105574846268, 0.027256235480308533, -0.11367528140544891, -0.04801108315587044, 0.14344000816345215, -0.11593500524759293, -0.2111683040857315, -0.18574722111225128, 0.002904157619923353, 0.0030844660941511393, -0.0117334621027112, 0.12113381177186966, -0.26665568351745605, 0.039500392973423004, 0.15577708184719086, -0.11251319944858551, -0.0378914475440979, -0.07511235773563385, -0.08922454714775085, -0.03254957124590874, 0.14241324365139008, 0.18330712616443634, -0.033515021204948425, 0.19005857408046722, 0.0035492919851094484, 0.17495721578598022, -0.009307951666414738, 0.013075001537799835, 0.03646160662174225, 0.31713876128196716, 0.08492861688137054, -0.07056033611297607, 0.09667219966650009, -0.03609826788306236, -0.04885999113321304, 0.09064672887325287, 0.0031012576073408127, 0.02779216319322586, 0.001395237515680492, 0.0335792601108551, 0.042396776378154755, -0.19324176013469696], [0.06366484612226486, -0.09379339218139648, -0.13775503635406494, -0.005578900687396526, 0.00802108645439148, -0.06623727828264236, 0.07937245815992355, -0.12819212675094604, -0.038867492228746414, -0.05586175620555878, -0.0044495840556919575, 0.02663453109562397, -0.09626459330320358, 0.06917676329612732, 0.006017402280122042, -0.18070116639137268, -0.18706649541854858, -0.0011766246752813458, -0.02110474929213524, -0.009877731092274189, -0.0026726247742772102, 0.021543385460972786, 0.09051070362329483, -0.10612243413925171, -0.11193585395812988, -0.10442415624856949, 0.027594642713665962, 0.11890583485364914, -0.04119754582643509, -0.23893918097019196, -0.16274960339069366, -0.07956394553184509, -0.05326909199357033, -0.039005495607852936, 0.0036849109455943108, -0.029775118455290794, 0.10022319108247757, -0.06456690281629562, -0.08236061781644821, 0.045737750828266144, -0.14667028188705444, 0.03721974045038223, 0.009794929064810276, -0.014082339592278004, 0.15169741213321686, -0.05942748859524727, 0.19678057730197906, -0.02153112180531025, -0.12196876853704453, 0.006678069941699505, -0.10279296338558197, -0.07180074602365494, 0.03792094066739082, 0.053064052015542984, -0.059423692524433136, 0.0943077802658081, 0.11055441200733185, 0.13718222081661224, 0.050564661622047424, 0.13022206723690033, -0.17963643372058868, -0.03036348707973957, -0.08280830085277557, 0.011387349106371403, -0.05457952991127968, -0.015869855880737305, 0.1797678917646408, -0.01621006242930889, 0.039521604776382446, -0.022184213623404503, 0.00723143108189106, 0.0005746575188823044, 0.042819757014513016, 0.11129937320947647, 0.157343789935112, 0.02957203984260559, 0.05630804970860481, -0.10991990566253662, -0.008393172174692154, 0.08980772644281387, 0.08270929753780365, 0.020210999995470047, 0.0846601352095604, 0.046873483806848526, 0.11464212834835052, 0.027636904269456863, -0.033973004668951035, 0.03207644447684288, -0.27990320324897766, 0.03656969591975212, -0.3484625816345215, -0.16460925340652466, 0.04314874857664108, 0.04560917615890503, -0.1110244169831276, -0.06801338493824005, 0.13090136647224426, -0.046230070292949677, 0.2142229676246643, -0.00018336657376494259], [0.04755659028887749, 0.18844662606716156, 0.01527120266109705, -0.11682313680648804, -0.012788727879524231, -0.004150428809225559, -0.1590605527162552, -0.029817212373018265, -0.08650244027376175, 0.1077338382601738, 0.004034011624753475, -0.03295983001589775, -0.01237549725919962, 0.026004856452345848, 0.07188965380191803, -0.11651639640331268, -0.1213969737291336, 0.1287502944469452, -0.04805783927440643, -0.16513286530971527, 0.051726341247558594, 0.05400572344660759, -0.17840476334095, 0.026746079325675964, 0.08573216199874878, 0.043634720146656036, 0.07365740090608597, -0.1636674851179123, 0.038582708686590195, -0.035270314663648605, -0.05626916512846947, 0.1014631986618042, 0.17240935564041138, 0.20913371443748474, 0.15324078500270844, -0.012789512984454632, 0.041109513491392136, 0.0893731564283371, 0.01412328239530325, 0.015240557491779327, 0.09600837528705597, -0.006909388583153486, -0.08499834686517715, -0.04830680787563324, 0.07388221472501755, 0.00034886790672317147, 0.039651453495025635, 0.07809769362211227, -0.04368774965405464, 0.08814068138599396, 0.044154081493616104, -0.14391739666461945, 0.1681005358695984, -0.05836177244782448, -0.19290125370025635, -0.1217997819185257, -0.11900275945663452, 0.006080797407776117, -0.031917259097099304, -0.03925774246454239, 0.07222622632980347, 0.03961245343089104, 0.036835361272096634, 0.04776548221707344, 0.07181045413017273, -0.24568182229995728, -0.04247525706887245, -0.2541033625602722, 0.004202782176434994, -0.0005399726214818656, 0.12185671925544739, 0.13841870427131653, 0.02363341860473156, -0.05831705406308174, 0.056545499712228775, -0.0615287683904171, -0.00016231344488915056, 0.11493412405252457, -0.06500821560621262, 0.09390179812908173, 0.07895109057426453, -0.11128093302249908, 0.11350730061531067, 0.08141785860061646, 0.011879456229507923, -0.08884874731302261, -0.10014040768146515, -0.06266453862190247, 0.06323935836553574, -0.11137805879116058, 0.050992026925086975, -0.14472652971744537, -0.023784181103110313, 0.012779438868165016, 0.28497976064682007, -0.16052605211734772, 0.17156320810317993, 0.01746412366628647, 0.04792205989360809, 0.013793442398309708], [0.0463378019630909, -0.016013043001294136, 0.1515030860900879, -0.047561485320329666, 0.11797188222408295, -0.06713908910751343, -0.006277871783822775, -0.111691914498806, 0.047410670667886734, 0.1505732387304306, -0.008060038089752197, -0.02051427960395813, -0.11613623797893524, -0.023890884593129158, -0.16595937311649323, -0.0951162725687027, 0.02362971194088459, 0.13743306696414948, 0.29270249605178833, -0.061118561774492264, 0.1495494395494461, 0.025397509336471558, -0.015759075060486794, 0.04507368057966232, 0.006451317109167576, 0.011502555571496487, -0.042545802891254425, -0.1456744372844696, -0.16083626449108124, 0.06806410104036331, -0.028830446302890778, -0.06210879608988762, 0.000886233989149332, 0.08282705396413803, 0.018792813643813133, -0.06835688650608063, 0.12777218222618103, 0.16454142332077026, 0.051529765129089355, 0.10144975036382675, 0.12940460443496704, 0.1177494004368782, 0.06398536264896393, 0.062344085425138474, 0.054145436733961105, -0.11433584243059158, 0.07301104813814163, -0.14126747846603394, -0.08279237896203995, 0.13582345843315125, 0.08754577487707138, 0.15497775375843048, -0.02830680087208748, -0.12005724757909775, -0.025052078068256378, 0.010631034150719643, 0.008230138570070267, 0.11812253296375275, -0.037642158567905426, 0.03667982667684555, -0.10300886631011963, 0.0469202883541584, 0.0003238918143324554, -0.07812593877315521, -0.024571970105171204, -0.019189979881048203, -0.028148503974080086, 0.18456286191940308, -0.010257468558847904, -0.02541247569024563, -0.2185414880514145, -0.09317091107368469, -0.027410631999373436, -0.07146162539720535, 0.1439654380083084, -0.017185501754283905, 0.13424797356128693, 0.006429243832826614, 0.00387891149148345, -0.009252043440937996, 0.0003955460852012038, 0.17908787727355957, -0.24168919026851654, -0.20854875445365906, -0.056595057249069214, 0.0015137159498408437, 0.07190463691949844, 0.049292776733636856, -0.04003605991601944, 0.10617808252573013, 0.1073400229215622, 0.019177693873643875, 0.10030300170183182, 0.028407961130142212, -0.058503080159425735, 0.07270822674036026, 0.011525207199156284, 0.0001396037987433374, 0.15347270667552948, -0.2553658187389374], [-0.1323716640472412, -0.11083884537220001, -0.03391828387975693, -0.02377321384847164, -0.14074164628982544, -0.09198593348264694, -0.08037830144166946, 0.22945401072502136, 0.1682318150997162, -0.038445767015218735, -0.03394841030240059, -0.1073031947016716, -0.1781654804944992, -0.010913307778537273, -0.09317119419574738, -0.007922643795609474, -0.1494140326976776, -0.0711468905210495, 0.013877573423087597, 0.03419363498687744, 0.038980089128017426, -0.12329637259244919, 0.07980350404977798, -0.07471717149019241, 0.059091053903102875, -0.0009933519177138805, -0.12851572036743164, -0.029523933306336403, 0.03411465510725975, 0.05723365768790245, -0.013164288364350796, 0.14302489161491394, 0.09407304972410202, -0.02396424487233162, 0.07004019618034363, 0.18056170642375946, 0.1830538958311081, -0.11466050893068314, -0.0865120217204094, 0.13864947855472565, -0.09881356358528137, -0.12348798662424088, -0.045396801084280014, 0.03923730552196503, -0.009560960344970226, -0.0607442669570446, -0.06295239925384521, -0.06651216745376587, -0.0821213573217392, 0.2130400687456131, 0.02777913212776184, 0.08279825001955032, -0.04566602408885956, -0.01736310124397278, -0.06662578880786896, -0.2475946694612503, -0.12866714596748352, -0.16906064748764038, -0.21006625890731812, -0.11161276698112488, 0.12371344119310379, -0.00414908304810524, 0.12050221115350723, 0.0383741669356823, 0.07306888699531555, -0.0841183289885521, 0.04892221838235855, 0.06804132461547852, -0.05384257063269615, 0.05224527046084404, 0.05286594480276108, -0.05131269246339798, -0.10483485460281372, -0.11056413501501083, 0.008077174425125122, 0.030533485114574432, -0.07399953156709671, 0.002482319949194789, 0.05327204614877701, 0.05772155150771141, 0.14490176737308502, 0.11923395097255707, -0.09349233657121658, 0.15719939768314362, -0.049578212201595306, 0.09756160527467728, 0.006348030176013708, -0.03848310932517052, -0.16878782212734222, -0.01601927913725376, -0.14560236036777496, 0.09031842648983002, 0.14162559807300568, -0.07440897822380066, 0.028195539489388466, -0.06773108243942261, -0.058682721108198166, 0.07231593132019043, -0.0035983852576464415, 0.16243398189544678], [-0.04535582661628723, 0.09399143606424332, 0.009162107482552528, 0.07484599202871323, -0.04205690324306488, -0.004661199636757374, -0.010863115079700947, -0.07863439619541168, 0.062271323055028915, 0.061158426105976105, 0.055826105177402496, 0.14455755054950714, 0.1363164633512497, -0.0352649949491024, 0.03508701175451279, -0.05865204706788063, 0.002196657471358776, -0.05640874803066254, 0.15237386524677277, 0.034207213670015335, -0.0205064807087183, 0.11226442456245422, -0.20415475964546204, -0.18943412601947784, 0.07485169172286987, 0.005687290802598, 0.023555438965559006, -0.12475267052650452, -0.002185220131650567, 0.1884378045797348, 0.07751915603876114, 0.015961553901433945, -0.04129471629858017, -0.03711109980940819, -0.200259268283844, 0.02423207461833954, 0.07952094823122025, -0.14819307625293732, -0.062208808958530426, -0.10886400938034058, -0.09079500287771225, 0.005030153319239616, -0.19604605436325073, 0.05777450650930405, 0.07090297341346741, -0.02361716702580452, -0.2333202064037323, -0.09371701627969742, 0.11595918238162994, -0.2612810730934143, -0.03268591687083244, -0.11098362505435944, 0.0756729394197464, -0.06349707394838333, 0.03799343481659889, 0.05318373441696167, 0.17270413041114807, -0.019087472930550575, 0.0353684239089489, -0.13967519998550415, 0.11196298897266388, -0.07152879238128662, -0.05274109169840813, -0.002092490904033184, -0.14699530601501465, 0.021516703069210052, 0.09099393337965012, 0.02755155600607395, -0.1494419425725937, -0.011644741520285606, 0.06979067623615265, -0.1766320914030075, -0.10108589380979538, 0.08244230598211288, 0.025403518229722977, 0.03874751552939415, 0.17489266395568848, 0.04406919330358505, 0.06187690421938896, 0.25495606660842896, -0.0201091431081295, -0.0750439465045929, 0.04023589938879013, 0.04128880053758621, -0.04953961446881294, -0.09710092097520828, 0.2172497808933258, 0.03074088878929615, -0.06887426227331161, 0.086146779358387, -0.010265522636473179, -0.023784475401043892, 0.05288393050432205, -0.12150651216506958, 0.03316595032811165, 0.17992077767848969, 0.043845806270837784, -0.035384196788072586, -0.047695208340883255, -0.03859405964612961], [-0.023547600954771042, -0.03503049537539482, 0.08496105670928955, -0.16070438921451569, 0.07992526143789291, -0.03520480915904045, 0.034950871020555496, -0.10449513047933578, 0.12319162487983704, 0.11549616605043411, 0.08897392451763153, -0.1404491513967514, 0.076876699924469, 0.03753845393657684, -0.14884190261363983, -0.002252870937809348, 0.033799901604652405, 0.0515596978366375, 0.01643580198287964, 0.02700318582355976, 0.13663430511951447, 0.11367874592542648, -0.26583540439605713, -0.029433129355311394, -0.015266560949385166, -0.16245584189891815, 0.008732720278203487, 0.11578571051359177, -0.10074735432863235, 0.03119133599102497, -0.08491580933332443, -0.06230016425251961, 0.13698500394821167, -0.050306349992752075, 0.04285667464137077, -0.1735006868839264, 0.17267541587352753, 0.05587836727499962, -0.016921516507864, 0.053365085273981094, -0.13071946799755096, 0.10964509844779968, -0.03004010021686554, -0.018921472132205963, -0.045586660504341125, -0.05383841320872307, 0.06441035866737366, 0.023727865889668465, 0.06043055281043053, -0.00081544992281124, 0.047584254294633865, -0.038835957646369934, 0.2067423313856125, -0.14537963271141052, 0.1169150173664093, 0.12683334946632385, -0.07901277393102646, 0.20669984817504883, 0.09230257570743561, 0.09004470705986023, 0.002598237246274948, -0.020141365006566048, -0.019416535273194313, 0.03621027618646622, 0.12991899251937866, 0.016811583191156387, -0.0845097079873085, 0.07644146680831909, 0.08540976047515869, 0.06070049852132797, 0.06912319362163544, -0.025120651349425316, -0.08613654226064682, -0.08049952238798141, -0.15164132416248322, 0.021970044821500778, -0.007737430743873119, -0.04079597815871239, -0.2740471065044403, -0.1156521663069725, 0.10392515361309052, -0.06162459775805473, 0.033434320241212845, -0.12894535064697266, 0.13339780271053314, -0.01335434801876545, 0.026892753317952156, 0.21564796566963196, 0.14060541987419128, -0.028424983844161034, -0.05500379949808121, -0.03018476627767086, 0.047278132289648056, -0.009756457060575485, -0.03664731979370117, 0.13603809475898743, -0.010589622892439365, 0.213219553232193, 0.008413716219365597, 0.16553600132465363], [0.033967144787311554, 0.09154000878334045, -0.06888225674629211, -0.08115921914577484, 0.04370011389255524, 0.09397286921739578, 0.003826823551207781, -0.12228585034608841, -0.12930753827095032, -0.13566502928733826, -0.1680319458246231, 0.010407788679003716, 0.10694780945777893, -0.00127531832549721, -0.21487732231616974, -0.01813388243317604, -0.015475837513804436, 0.19924776256084442, -0.13851700723171234, -0.037138208746910095, 0.16216008365154266, 0.04277539625763893, 0.053549110889434814, 0.010892512276768684, 0.031364765018224716, 0.0739203467965126, -0.07695058733224869, 0.08346913754940033, -0.1305190473794937, 0.031214991584420204, 0.016991296783089638, -0.05858544260263443, -0.002156485104933381, -0.05917929857969284, -0.10187701135873795, -0.03361497446894646, -0.10494627803564072, 0.030685454607009888, -0.10932828485965729, 0.06768125295639038, -0.11231260001659393, -0.05383256822824478, -0.05937543138861656, 0.1259697824716568, 0.03139769658446312, -0.02012762986123562, 0.04740877449512482, -0.2003118246793747, -0.004100322723388672, 0.1898857206106186, 0.012580993585288525, 0.006455894559621811, 0.05210563540458679, 0.08513714373111725, -0.18988120555877686, -0.15992365777492523, 0.019990503787994385, -0.06795688718557358, 0.1247224509716034, -0.02996079809963703, -0.1838984489440918, -0.12053868919610977, -0.2613869905471802, -0.022256210446357727, -0.017907986417412758, -0.04034985601902008, 0.10255932807922363, 0.11393250524997711, -0.016197295859456062, -0.2029261291027069, 0.05867940932512283, 0.05279310792684555, 0.02941107377409935, 0.093300461769104, -0.11072973161935806, 0.190954327583313, -0.09120417386293411, 0.07122188806533813, -0.16040097177028656, -0.08843190222978592, 0.14414753019809723, -0.024801399558782578, 0.07984389364719391, 0.02252521924674511, -0.009977428242564201, 0.12964899837970734, -0.0224970281124115, -0.16927877068519592, -0.07635682076215744, 0.05239684507250786, 0.037597496062517166, -0.14896613359451294, -0.06885673105716705, -0.05529902130365372, -0.03474191203713417, -0.13079574704170227, -0.059733595699071884, -0.06926244497299194, 0.07334273308515549, 0.07691588997840881], [-0.0144798057153821, -0.03893177583813667, 0.013895278796553612, -0.0810905396938324, -0.11388693749904633, -0.09336929768323898, -0.06304055452346802, -0.1155395582318306, 0.016093434765934944, 0.07246490567922592, -0.15760786831378937, 0.0024954236578196287, -0.02184087224304676, 0.0339137502014637, -0.001284893020056188, -0.14435066282749176, -0.12536242604255676, 0.0029219279531389475, 0.10216378420591354, -0.21988137066364288, 0.03433261066675186, 0.1181362196803093, -0.015196338295936584, 0.05463822931051254, -0.06981875002384186, -0.01185374055057764, -0.1609659045934677, -0.23280660808086395, 0.09362199157476425, -0.09781558811664581, -0.0452364981174469, -0.13768158853054047, -0.14001798629760742, -0.017263071611523628, -0.072451151907444, 0.03753466531634331, 0.04493963345885277, 0.03186219185590744, 0.023694412782788277, 0.15668132901191711, -0.10601812601089478, -0.00545534398406744, 0.11248252540826797, 0.1255759745836258, -0.14118920266628265, -0.031018666923046112, 0.13486912846565247, 0.050455138087272644, 0.01939072646200657, -0.021617325022816658, -0.038113608956336975, 0.028621789067983627, -0.025062181055545807, -0.004228299017995596, 0.007805297616869211, -0.06733635067939758, 0.05284346267580986, 0.28940102458000183, -0.17646485567092896, 0.03217628225684166, -0.07549118995666504, 0.05733935534954071, 0.0046234214678406715, 0.07565650343894958, 0.11673447489738464, 0.01910850964486599, 0.11241921782493591, -0.23280207812786102, 0.15794093906879425, 0.037245798856019974, 0.10703595727682114, -0.12114057689905167, -0.0825001671910286, 0.014492500573396683, -0.001691959099844098, 0.028110317885875702, 0.10769851505756378, -0.17529639601707458, 0.09670575708150864, 0.0008224406628869474, -0.020236695185303688, -0.1460900902748108, 0.0499555878341198, -0.23973460495471954, 0.0643891990184784, -0.14817458391189575, 0.005308674182742834, -0.011834141798317432, 0.11614177376031876, -0.017741471529006958, -0.06893189996480942, -0.05030852183699608, 0.12640449404716492, 0.053257331252098083, -0.03829524666070938, -0.023480471223592758, -0.15562957525253296, 0.038577690720558167, -0.17475968599319458, 0.07465765625238419]], "dtype": "float32"}}, {"a": [{"__nd__": [[1.286215913680353e-24, 2.386951735324566e-23, -1.1001355576599078e-23, -1.0266209196490696e-24, 1.7982876350861806e-23, 1.1913896461123356e-23, -6.962120318020628e-24, 2.2396743231051525e-24, 1.999356408702988e-26, 2.819558677569745e-24, -6.986804564605898e-24, -8.932040301926608e-25, -1.5980523336913795e-23, 1.3512345197419609e-24, 4.500206917405254e-24, -2.2741940874567188e-24, -1.615050866249263e-23, 9.885318555242178e-24, 3.645250512379394e-24, -7.26015117915222e-24, -2.8060044694962376e-24, -1.5006693482080182e-23, -1.0254101562102138e-23, -5.378113652746708e-24, 1.2284208280417634e-23, -6.395033851236662e-24, 6.402136754827272e-24, -5.38316196810967e-24, 2.479475613733083e-24, -1.4139549767714336e-24, 8.73056581837884e-24, -5.348113272521153e-24], [-1.5725282657864106e-23, -1.3819490951880578e-23, -5.3187747465950785e-24, -1.2490599536772418e-23, 4.1021200944990485e-24, -4.657681697888188e-24, 2.3746065354744004e-23, -1.737714634934423e-23, 2.0363318169338217e-24, -1.1279530019885353e-23, -4.9340133620998665e-24, 1.4141946918790077e-24, -2.3253713597968414e-23, 2.3682612144541194e-24, 1.9029074727418585e-23, -4.534383132832596e-24, 2.5980784842503464e-24, 1.024921693537701e-23, -1.639534978822879e-24, -5.809602029671329e-24, -2.2757363894125164e-23, 6.409907034743699e-24, -1.1316840774119591e-24, -1.0126838576567358e-23, 1.2659671361423006e-24, -5.83680353140516e-24, -5.284261293130754e-24, 6.062911577225038e-24, -1.338382358431055e-23, 1.0473153247124814e-23, -1.3976862391584928e-23, -6.627013783204552e-24], [1.9229461174562826e-23, 1.1273547298780157e-23, 9.760399472672036e-25, -9.405302216441529e-24, -5.292723009630607e-24, 7.327531240333608e-25, 3.7523231453869146e-25, -9.087092885000061e-25, -3.083251508180812e-25, -2.143751006397767e-24, -1.441978727948299e-23, -5.165107403271194e-24, 9.146240295166948e-24, -1.3683343025970294e-23, -1.895555052446927e-24, -8.19753960446915e-24, 1.1135042848130751e-24, -3.886331927041272e-24, 1.47739369165511e-23, 5.065831624931845e-24, -1.3118031890653528e-23, 6.712280180728081e-24, -7.746518637100804e-24, -5.769963741336236e-24, 3.138190246810732e-24, -4.917474104361003e-24, 8.928558072675736e-24, -7.745100265193217e-24, -6.189465376806024e-24, -6.563708878851924e-24, -5.656387492507041e-24, 1.0991915278146298e-23], [6.287962943862372e-24, 1.7099438911713847e-23, 2.2676442542467592e-23, -1.0307170601779071e-23, 7.041912809722828e-25, -1.176200287154486e-23, -7.776059111418615e-24, -6.300824926491548e-24, -1.0571548656693936e-23, -1.0518872469747803e-23, -1.4428437533739191e-24, -1.2167183920571718e-23, 8.421269234659786e-24, -9.177535196137971e-24, 2.809568819167343e-23, -4.513844350304262e-24, 7.201764132290353e-25, -1.2953411608091102e-23, 1.7060917255990095e-23, 1.1195006729334542e-23, 2.9691586540663103e-24, -1.346802501961203e-23, 2.1066688246108164e-24, -1.12143101556853e-23, 1.46235532067452e-23, 8.091947416861696e-24, 9.54390349976705e-25, -2.436375606530654e-23, -9.83416013667769e-24, 6.877611226966111e-24, 1.212883660310802e-23, 1.1091698294047704e-23], [-1.5920129723731885e-23, 3.01552316490977e-24, 1.0155152372177551e-23, -1.6583355063465588e-23, -7.904515076509219e-26, 2.7384239390373113e-24, -3.0076457971254593e-24, -1.8521617862029823e-23, -3.2569810634386645e-24, -3.273054301597769e-24, -1.480724065181727e-24, -5.634714327996999e-24, -4.430121359141798e-24, -3.9706283214742184e-25, 4.424546084694149e-24, -3.0442676785742133e-24, -4.096489205357581e-24, 9.235403665562269e-25, -1.9209159050306058e-23, 4.7703511511068315e-24, -5.630971186083213e-27, 3.9693525854790563e-25, 9.607342904904018e-24, -1.952815546771571e-23, -1.9738337989580986e-23, 2.7380383832698846e-24, -7.116855335275923e-26, 1.7017693989271225e-24, -4.621270639516354e-24, 6.643536474864406e-24, 9.326388515787745e-24, 4.839822975586025e-24]], "dtype": "float32"}], "e": {"__nd__": [[1.2862159254334316e-12, 2.386951747368471e-11, -1.1001355208761776e-11, -1.02662095404632e-12, 1.7982875605282445e-11, 1.191389608495852e-11, -6.962120116527082e-12, 2.2396743747793435e-12, 1.9993563892700163e-14, 2.8195586056906086e-12, -6.986804364228494e-12, -8.932040220302473e-13, -1.5980522460878888e-11, 1.351234553935432e-12, 4.500207064633832e-12, -2.274194070908675e-12, -1.6150508014289855e-11, 9.885318258406883e-12, 3.645250334222139e-12, -7.260151247551194e-12, -2.8060045606514983e-12, -1.5006693804275884e-11, -1.0254101387441317e-11, -5.37811375295183e-12, 1.2284208372737027e-11, -6.395033808059836e-12, 6.402136633332223e-12, -5.383161798266922e-12, 2.4794756092044734e-12, -1.413954999092415e-12, 8.73056574951514e-12, -5.348113445158287e-12], [-1.5725282187517564e-11, -1.381949069068078e-11, -5.318774934370829e-12, -1.2490599700376581e-11, 4.1021201192814605e-12, -4.657681792336055e-12, 2.374606587751682e-11, -1.737714659666434e-11, 2.036331823648818e-12, -1.1279530261198989e-11, -4.934013267399173e-12, 1.4141947161927515e-12, -2.325371319111813e-11, 2.3682611861169933e-12, 1.9029075190579725e-11, -4.53438328551492e-12, 2.5980784364165732e-12, 1.0249217273494704e-11, -1.639534970461487e-12, -5.809601931472397e-12, -2.275736370183079e-11, 6.409906893461992e-12, -1.1316840476879864e-12, -1.0126838603796706e-11, 1.2659671488196578e-12, -5.83680369661832e-12, -5.2842613097736635e-12, 6.062911457604958e-12, -1.3383823563306585e-11, 1.0473153594370288e-11, -1.3976862202336182e-11, -6.627013608012655e-12], [1.9229461772907186e-11, 1.1273547199930345e-11, 9.760399953540055e-13, -9.405302058007603e-12, -5.292722857208609e-12, 7.327531051544434e-13, 3.752323159220833e-13, -9.087093057191797e-13, -3.0832514276088463e-13, -2.14375110545173e-12, -1.4419787412733864e-11, -5.1651074910175065e-12, 9.146240188828703e-12, -1.3683342653392216e-11, -1.8955551104454793e-12, -8.19753986913696e-12, 1.1135042540799667e-12, -3.8863317945725395e-12, 1.4773937281886695e-11, 5.065831434891699e-12, -1.3118031834402633e-11, 6.712280037346474e-12, -7.746518704276895e-12, -5.769963933727196e-12, 3.138190228513249e-12, -4.917473980098341e-12, 8.928558413445753e-12, -7.745100567835284e-12, -6.189465173028763e-12, -6.5637087778847025e-12, -5.656387684627173e-12, 1.099191484360551e-11], [6.287962771994726e-12, 1.7099438184842164e-11, 2.2676442321123425e-11, -1.0307170915380137e-11, 7.041912625932456e-13, -1.1762002830040252e-11, -7.776059310349304e-12, -6.300824879207356e-12, -1.0571548844651169e-11, -1.0518872231579657e-11, -1.4428437828187257e-12, -1.2167183927047631e-11, 8.421268890557165e-12, -9.177535467697062e-12, 2.8095687235052402e-11, -4.513844159559355e-12, 7.201764141637201e-13, -1.2953411780702861e-11, 1.7060916915334623e-11, 1.1195006727193757e-11, 2.9691585561736344e-12, -1.3468025306107023e-11, 2.1066687890675118e-12, -1.1214309862672689e-11, 1.462355236903079e-11, 8.091947251154252e-12, 9.543903211131632e-13, -2.4363755804501785e-11, -9.83416039573859e-12, 6.877611327671396e-12, 1.2128836129887688e-11, 1.1091697872667172e-11], [-1.592012979778623e-11, 3.01552316103737e-12, 1.01551527603716e-11, -1.6583354481292112e-11, -7.90451485190892e-14, 2.7384238539956973e-12, -3.0076457817329594e-12, -1.8521618266875706e-11, -3.256981056382058e-12, -3.2730543535891554e-12, -1.4807240720018933e-12, -5.634714483199188e-12, -4.430121200438286e-12, -3.9706283508529616e-13, 4.424546232867366e-12, -3.0442677458747402e-12, -4.09648920687844e-12, 9.235403240770435e-13, -1.9209158569344353e-11, 4.77035121473901e-12, -5.63097127181055e-15, 3.9693525159464893e-13, 9.607342699169408e-12, -1.9528155134618252e-11, -1.9738337700414554e-11, 2.7380383117031615e-12, -7.116855526125357e-14, 1.7017693677845447e-12, -4.6212708139370395e-12, 6.643536415440465e-12, 9.32638861972368e-12, 4.8398229868240605e-12]], "dtype": "float32"}}, {"a": [{"__nd__": [[-7.824853897094727, -63.230682373046875, 23.077085494995117, 36.651859283447266, 19.33675193786621, -26.389272689819336, 20.213830947875977, 25.81589126586914, 46.764488220214844, -26.28329849243164, -6.793637752532959, 22.418161392211914, 1.7223021984100342, -33.51982879638672, 12.438488960266113, 17.35984992980957, 2.6388869285583496, 6.587547779083252, -7.761177062988281, 44.34836959838867, -27.02076530456543, -17.6196231842041, 0.3295654058456421, 26.368619918823242, -44.92119598388672, -39.308902740478516, -61.13185501098633, 0.9808567762374878, -33.54917526245117, -11.536502838134766, 19.841344833374023, -19.246280670166016, 6.938460826873779, -8.98013687133789, 20.163429260253906, 14.2538423538208, -20.758010864257812, -5.0595526695251465, 31.312301635742188, 11.801305770874023, -41.90886306762695, 19.668540954589844, -32.5383186340332, 42.37539291381836, 30.722965240478516, 8.696564674377441, 4.71698522567749, -6.270967483520508, 20.3557186126709, -11.123345375061035, 7.06709623336792, 26.158031463623047, -16.854196548461914, -18.485782623291016, -35.830963134765625, -31.93418312072754, -54.85186004638672, 21.89486312866211, 54.26743698120117, 42.75986862182617, -48.782535552978516, 21.74563980102539, 42.66318893432617, -13.704298973083496], [3.0434229373931885, -60.23678970336914, 8.002084732055664, -31.90679168701172, 5.989050388336182, -46.83608627319336, 16.285747528076172, 30.735380172729492, -23.49539566040039, -34.1076774597168, -64.68069458007812, -7.992405891418457, -17.447818756103516, -46.530582427978516, 13.59056282043457, -13.49911117553711, -32.27207946777344, -16.63717269897461, 32.139137268066406, 18.08810806274414, -3.9007818698883057, 18.573150634765625, -34.82735824584961, 0.9711030125617981, 4.419670104980469, -14.09509563446045, 13.063750267028809, -23.41300392150879, 15.294621467590332, 16.555763244628906, -87.5147933959961, 35.709171295166016, -91.4289779663086, -18.481470108032227, 30.760440826416016, -9.518601417541504, -43.98558044433594, -32.812110900878906, 30.807172775268555, -57.645137786865234, 33.86973190307617, -1.6530629396438599, -35.77185821533203, -17.493452072143555, -30.5167293548584, 25.98675537109375, 21.97283363342285, -51.574832916259766, -12.107763290405273, -25.895570755004883, 38.8052978515625, -40.50297164916992, 48.12687301635742, -0.998842179775238, -34.8580322265625, -20.318225860595703, -24.048564910888672, -38.23855209350586, 2.8448057174682617, -36.837467193603516, -65.4109878540039, -11.390294075012207, -54.08568572998047, -22.56482696533203], [-27.273086547851562, -26.13648223876953, 60.028892517089844, 1.6923470497131348, 7.361130237579346, -53.29195022583008, -19.618894577026367, 6.459078788757324, -30.73586654663086, 27.241886138916016, -36.08089828491211, -8.590984344482422, -17.5815486907959, 4.929619789123535, -19.214702606201172, -8.126809120178223, -7.083902835845947, -22.031633377075195, 14.552816390991211, 7.151651382446289, -2.6584184169769287, -12.913749694824219, 23.896320343017578, -7.0750274658203125, -9.200918197631836, 3.1407008171081543, 50.89760208129883, -40.21068572998047, 23.095706939697266, -56.795135498046875, 6.518194675445557, 47.25248336791992, 14.660056114196777, 5.118752479553223, -16.81348419189453, 0.9314160346984863, -7.56188440322876, -23.57929801940918, -8.696268081665039, 5.95143461227417, -23.016511917114258, -24.188453674316406, -40.80730438232422, 11.72175121307373, 16.359697341918945, 6.949813365936279, 28.955114364624023, 2.636029005050659, 11.76032829284668, 9.11512279510498, -2.7462005615234375, -15.924654960632324, 66.42457580566406, -13.5585298538208, -19.97633171081543, 13.020295143127441, 7.555632591247559, -42.14374542236328, 33.68034744262695, -2.8258373737335205, -33.32793426513672, 35.66352462768555, 18.770267486572266, -37.93626022338867], [53.315486907958984, 19.7639217376709, -0.3058834671974182, 47.706565856933594, 4.778933525085449, -0.8424922823905945, -19.809003829956055, 17.45551300048828, 14.298089027404785, 0.4404042065143585, -23.6995906829834, 3.897895574569702, -2.093963623046875, -40.00423049926758, -44.15436935424805, 16.954601287841797, -78.05664825439453, 6.162315845489502, -14.34249210357666, 39.247398376464844, -14.804046630859375, 19.784526824951172, -11.907415390014648, 3.412444591522217, 22.1050968170166, -13.87667179107666, 29.89664649963379, -29.810935974121094, 8.837785720825195, 27.67919158935547, 19.801124572753906, 57.49334716796875, -22.903179168701172, -41.37447738647461, -29.830053329467773, 19.813276290893555, 14.937687873840332, 3.067699670791626, -33.68871307373047, 23.006019592285156, -36.74114227294922, 34.76848602294922, 85.72315216064453, 10.993534088134766, 17.493244171142578, -14.78579330444336, -54.306400299072266, -18.081581115722656, -32.78685760498047, -43.41013717651367, 51.670799255371094, 8.18587589263916, 11.39062786102295, -9.270670890808105, -5.186132431030273, 28.936519622802734, -0.9701135754585266, 15.920818328857422, -5.880128860473633, -10.264586448669434, -42.20742416381836, -59.33247375488281, -7.846857070922852, 1.6949007511138916], [-38.583900451660156, -1.0342459678649902, 1.6125768423080444, -64.68023681640625, -27.453739166259766, 12.217916488647461, 11.611762046813965, -34.61024856567383, 4.753441333770752, 30.672216415405273, 23.31218910217285, -2.8585736751556396, -14.039289474487305, -18.880334854125977, 14.03459358215332, 5.853119373321533, 18.88212013244629, 3.956214427947998, 24.562917709350586, 2.837775945663452, 23.540660858154297, 26.124683380126953, -24.135799407958984, -54.244319915771484, 46.304786682128906, 7.699761867523193, -51.43457794189453, -21.41458511352539, 23.15622901916504, -8.390304565429688, -15.361842155456543, 42.02645492553711, -31.418615341186523, -23.94460105895996, 6.341594219207764, 11.713647842407227, 14.890291213989258, 27.337141036987305, 5.9573588371276855, 14.6815185546875, -7.776277542114258, 1.8809491395950317, 3.226025342941284, 6.452944278717041, 23.17619514465332, 16.65437126159668, 1.3265100717544556, -24.273778915405273, 2.030137777328491, -0.8405335545539856, 5.79544734954834, 66.19576263427734, -15.713183403015137, 12.668670654296875, 32.053863525390625, 2.9400575160980225, 70.1353530883789, -11.341178894042969, 9.722676277160645, -23.96417236328125, -10.54574203491211, 7.468017101287842, 25.389122009277344, 9.572845458984375], [37.77344512939453, -2.84895920753479, -9.965055465698242, 81.11955261230469, 11.827129364013672, -21.58444595336914, -20.827913284301758, -28.280397415161133, 28.09708023071289, -10.931319236755371, 3.8208742141723633, -1.9052602052688599, 21.944772720336914, -25.40232276916504, -32.26716995239258, 6.72891902923584, 18.028778076171875, 9.975969314575195, -24.283611297607422, 20.32454490661621, -22.315828323364258, -15.646159172058105, -28.318605422973633, 39.034751892089844, 19.00264549255371, 18.14980125427246, 37.416343688964844, 84.37947845458984, -51.27678298950195, -22.151296615600586, -18.485252380371094, -25.52458953857422, -32.32749557495117, -8.989591598510742, -36.498111724853516, 10.051271438598633, 1.5509496927261353, 14.430859565734863, 15.231853485107422, 18.090206146240234, -11.20226764678955, 63.634300231933594, 7.032933712005615, -55.874542236328125, -30.071271896362305, -52.215087890625, 0.520038902759552, -3.6541566848754883, -14.052770614624023, 60.327781677246094, 16.469486236572266, 21.78295135498047, -27.932205200195312, -29.974346160888672, 1.104164958000183, 5.190214157104492, -55.68934631347656, -5.409210681915283, 0.4221624732017517, -6.454474925994873, -24.40171241760254, -16.18605613708496, -5.3057756423950195, 2.7240006923675537], [18.143747329711914, 32.42568588256836, -8.830431938171387, -13.684685707092285, 13.129670143127441, 18.325347900390625, -13.026013374328613, 23.711803436279297, 29.38755989074707, 38.75059509277344, 53.34962844848633, -25.99248504638672, 0.41694629192352295, 0.3383308947086334, -2.942207098007202, -19.477888107299805, -44.56686019897461, 2.780383348464966, -23.050670623779297, 2.5798044204711914, -23.821306228637695, -32.989749908447266, 7.0289506912231445, 4.527645587921143, 17.378459930419922, 65.57591247558594, 8.318685531616211, 14.55245304107666, -32.326839447021484, -38.60722351074219, 47.48488235473633, -27.70439338684082, -31.318321228027344, -1.1729676723480225, -2.7057723999023438, -27.0751953125, -9.14919662475586, 36.88127136230469, -24.179807662963867, -41.81017303466797, 28.277809143066406, 14.76890754699707, -18.615558624267578, -47.46501159667969, -22.338701248168945, -1.6411330699920654, 34.97016143798828, -31.09745979309082, -2.5355963706970215, 20.25433921813965, -71.43801879882812, 31.01358413696289, -35.47351837158203, 9.959864616394043, 7.400494575500488, 32.53759002685547, 16.52773094177246, -71.31871795654297, -23.543376922607422, 4.146153926849365, -5.849396705627441, 14.32832145690918, 23.68099594116211, 18.253551483154297], [-6.428248405456543, -55.39011001586914, -13.664437294006348, 12.491044044494629, -32.713417053222656, 18.938241958618164, 27.251798629760742, 3.806009292602539, -91.91339874267578, -38.3426513671875, 56.58567428588867, -16.396133422851562, -59.524295806884766, 37.05742263793945, 0.4006677567958832, 33.148216247558594, 8.308760643005371, 13.756773948669434, 10.890277862548828, 20.924650192260742, 68.32291412353516, 29.130189895629883, -47.410179138183594, 13.399845123291016, 34.244422912597656, 36.081024169921875, -12.798149108886719, 34.948333740234375, 27.832271575927734, 77.17552185058594, -22.500051498413086, -15.222017288208008, -9.183571815490723, -14.706664085388184, -26.70062255859375, -82.88026428222656, -13.040984153747559, 11.080741882324219, -14.796655654907227, 58.06143569946289, 0.4012743830680847, 38.730873107910156, 7.842612266540527, 24.792219161987305, 8.377246856689453, 41.06353759765625, 1.678180456161499, 14.883621215820312, 10.989988327026367, -8.648563385009766, 12.149469375610352, 67.03158569335938, -8.111907958984375, 12.58394718170166, 37.733367919921875, 34.20759963989258, 0.3514925241470337, -71.50166320800781, 15.501084327697754, -66.584228515625, 44.09811782836914, 5.084216117858887, 29.99022102355957, -27.06333351135254], [-46.575225830078125, 7.822631359100342, -1.0081032514572144, -14.746224403381348, 2.5398383140563965, 83.02307891845703, 25.0357666015625, 31.48495864868164, 25.31810760498047, 44.38386535644531, -5.702150344848633, -61.64283752441406, -5.657584190368652, 24.35207176208496, -29.094011306762695, -5.29097843170166, -20.75309181213379, 45.06087875366211, 22.48371124267578, 19.546218872070312, 18.648038864135742, -1.117477536201477, 5.9266886711120605, -35.555538177490234, -65.59326171875, 4.502433776855469, 9.719707489013672, 36.27793884277344, 47.14007568359375, 47.099952697753906, -29.94070816040039, -10.567951202392578, 14.410028457641602, 75.95713806152344, 44.402976989746094, 33.581912994384766, -11.633638381958008, -26.495988845825195, 23.667236328125, -33.01983642578125, -7.153090476989746, -3.443338394165039, 47.78736114501953, -10.640918731689453, -73.177490234375, 15.133339881896973, -20.97065544128418, 1.2789307832717896, -2.6011409759521484, -8.211119651794434, 23.710216522216797, -45.98594665527344, -15.51042652130127, 20.521137237548828, -18.422542572021484, -48.26420211791992, 20.056659698486328, 33.692840576171875, -1.3498562574386597, -58.82536697387695, 21.179092407226562, -49.796634674072266, 9.263978004455566, -26.62391471862793], [-34.7951774597168, -13.25730037689209, -5.498926639556885, -19.326248168945312, 17.66109848022461, -13.382081031799316, 29.405725479125977, 8.435194969177246, -20.926715850830078, -25.97819709777832, -49.424888610839844, 8.72199535369873, -10.637626647949219, -71.16586303710938, -16.07144546508789, -27.031021118164062, 47.270328521728516, -51.08335876464844, -43.24052429199219, 33.29908752441406, 40.70289611816406, -22.3494873046875, -18.962574005126953, 18.487062454223633, -69.81299591064453, -55.835811614990234, -8.422001838684082, 46.4967155456543, -13.07190227508545, 3.028103828430176, 34.01367950439453, 63.20654296875, -17.30545425415039, -0.50510174036026, 54.66165542602539, -45.665977478027344, 30.82743263244629, -30.54946517944336, 3.0741710662841797, -7.278571605682373, -28.74034881591797, -29.493057250976562, 8.292430877685547, 29.866180419921875, -3.8920860290527344, 2.1033947467803955, 17.135221481323242, -29.148563385009766, 22.19077491760254, 24.580293655395508, 78.1087875366211, -11.283125877380371, -13.742453575134277, 23.0689754486084, 24.5731258392334, -16.052614212036133, -12.094880104064941, 16.233924865722656, -24.82752799987793, -12.316020011901855, 2.6877830028533936, 29.87816047668457, -18.137958526611328, -55.27153396606445], [-57.4979248046875, -37.78239059448242, -39.3950309753418, -4.638322830200195, -35.666404724121094, -21.08264923095703, -23.222576141357422, -13.215235710144043, 39.67512893676758, 19.858427047729492, 0.48388245701789856, -13.830788612365723, -12.996979713439941, 9.832393646240234, 17.52507209777832, -27.228525161743164, 1.094422698020935, -60.948585510253906, 6.351413726806641, 22.864477157592773, -8.402058601379395, 54.495811462402344, -26.437152862548828, 6.523212432861328, -1.0747424364089966, -3.008929967880249, -3.4296650886535645, 3.213150978088379, 25.438762664794922, 41.332088470458984, -0.06750733405351639, 24.459482192993164, -21.053009033203125, 22.97866439819336, -22.859745025634766, 32.71979904174805, 66.31922912597656, -3.6625165939331055, -65.77630615234375, -16.442188262939453, 36.95625305175781, 16.592304229736328, 7.467408180236816, 30.26004981994629, 14.533602714538574, 36.53175735473633, 27.58271026611328, 45.6510124206543, 10.763882637023926, 27.15778160095215, -9.213811874389648, -18.043825149536133, 51.20302963256836, 6.486140251159668, 24.14641761779785, 3.712200403213501, -17.595470428466797, 0.32948678731918335, 43.07981872558594, 16.657201766967773, -51.34419631958008, -6.457333564758301, 53.468814849853516, -6.413905620574951], [-54.01436233520508, 56.791465759277344, 20.47177505493164, -2.1003856658935547, -11.142532348632812, 32.470558166503906, -16.259502410888672, 34.44769287109375, 16.265857696533203, 7.340457916259766, 4.365444183349609, -5.2490410804748535, 31.369203567504883, -25.960285186767578, -48.657752990722656, 18.433202743530273, 18.937149047851562, 14.407234191894531, 12.231850624084473, -9.6162109375, 23.624481201171875, -1.366743803024292, -14.613407135009766, -9.356739044189453, 60.61919021606445, -52.04928207397461, 14.993642807006836, -50.373313903808594, -3.247000217437744, -13.008824348449707, -62.55084991455078, 11.671127319335938, -11.401142120361328, -62.97618865966797, 25.187042236328125, 43.05018615722656, 44.703731536865234, 10.768692970275879, 1.381744146347046, 39.84469985961914, 50.47150802612305, 34.00706100463867, -28.24612808227539, 6.040345668792725, 13.185483932495117, -54.288509368896484, 14.080820083618164, 68.18975830078125, 19.19808578491211, 44.202392578125, 5.4874420166015625, -2.196549654006958, 11.438141822814941, -24.47480010986328, -28.91325569152832, 37.9891357421875, 41.17026901245117, -29.73318862915039, 29.03996467590332, -2.6087992191314697, 10.194256782531738, 13.806559562683105, 29.830514907836914, 2.48154616355896], [42.30485916137695, 9.282384872436523, 18.725528717041016, -24.99787712097168, 64.75977325439453, 14.094747543334961, 21.389923095703125, -48.245574951171875, -24.681060791015625, 37.13650894165039, 8.606733322143555, -57.41339111328125, 31.148698806762695, -16.31348419189453, 42.339508056640625, -2.635070323944092, 6.141200542449951, 35.36863708496094, -26.924789428710938, -29.785404205322266, -5.894952774047852, 24.815183639526367, -39.92631912231445, -50.041038513183594, 24.711627960205078, -18.0893497467041, -11.766780853271484, -55.70431137084961, -20.3693790435791, -0.22701889276504517, 27.911334991455078, -23.136905670166016, 49.721282958984375, -20.536680221557617, -20.492753982543945, 46.1207275390625, 6.343804359436035, -16.560741424560547, -4.653994083404541, 40.59220886230469, 43.986083984375, -24.503795623779297, -1.2730143070220947, 2.704704761505127, -29.587203979492188, -1.6705515384674072, 0.15968582034111023, 34.20628356933594, 36.22275161743164, -9.87628173828125, -6.4185943603515625, 19.928768157958984, -17.555652618408203, 17.659204483032227, -2.6371946334838867, -29.08195686340332, -28.705101013183594, 43.41176223754883, -4.6926445960998535, 49.21062469482422, -15.409088134765625, 8.736156463623047, 71.78030395507812, -31.721601486206055], [-56.9344367980957, 38.59770584106445, -24.80427360534668, -0.6211909055709839, -17.577547073364258, 8.044883728027344, -28.51608657836914, -5.398202419281006, 13.494460105895996, -30.249265670776367, -21.002521514892578, -10.42148494720459, -42.864463806152344, -12.67682933807373, 3.5638961791992188, -11.986468315124512, -8.141294479370117, -0.597050130367279, -19.672304153442383, 20.731876373291016, 43.079341888427734, -13.931411743164062, 14.555947303771973, -7.188514709472656, 10.964620590209961, 46.973384857177734, 15.259786605834961, -53.20243453979492, 13.619869232177734, 36.98591232299805, -6.647885322570801, 29.869043350219727, -20.703466415405273, -44.5661735534668, 4.235070705413818, -52.69430160522461, -7.1931304931640625, 38.60526657104492, -57.29288864135742, -38.38490295410156, -30.960968017578125, -1.8995957374572754, -10.971503257751465, 41.974090576171875, 28.630449295043945, 14.85818862915039, 4.3689093589782715, 11.796893119812012, -25.85947036743164, 25.716310501098633, -33.361328125, 25.061309814453125, 35.58427429199219, 6.3259382247924805, -5.286391735076904, 12.009297370910645, -85.84555053710938, 9.06774616241455, -3.4006199836730957, -1.518843412399292, 21.19597625732422, -2.243067741394043, -34.206207275390625, -4.242080211639404], [2.0266644954681396, 3.4480173587799072, 4.85758113861084, 31.181072235107422, -32.58103561401367, 5.2710137367248535, 38.69789123535156, -52.03834533691406, 5.389400482177734, -17.561355590820312, -11.12131118774414, -27.144224166870117, -5.086521625518799, 55.237667083740234, -37.93291091918945, -35.08829879760742, -39.05803680419922, 51.822547912597656, -13.87019157409668, -0.7691718339920044, -12.083488464355469, -28.332685470581055, -7.054519176483154, 73.74730682373047, 41.76212692260742, -25.588134765625, 19.427223205566406, -24.945682525634766, -53.29103088378906, 30.5853271484375, 34.80278396606445, 31.381567001342773, -14.09595775604248, -5.00327205657959, 22.492965698242188, 11.744880676269531, -11.809123992919922, -48.5721549987793, 3.4162557125091553, 32.37028884887695, -35.927650451660156, 19.16278076171875, 9.946250915527344, 22.50444793701172, 26.159265518188477, -20.83395767211914, -48.50878143310547, -16.005586624145508, 26.673818588256836, -16.260116577148438, -20.821300506591797, -54.64051055908203, 52.57942581176758, 24.77140998840332, 23.096879959106445, 29.031761169433594, 17.301250457763672, -7.329865455627441, -4.912923336029053, -12.82693099975586, 79.04222106933594, 16.67736053466797, -13.914012908935547, 13.964902877807617], [-30.326066970825195, 45.13520050048828, -19.832073211669922, 15.032563209533691, 31.979328155517578, 11.54305648803711, 2.834099769592285, 0.3240625560283661, -1.6932328939437866, -17.300954818725586, -13.484502792358398, 44.38771057128906, -3.256098985671997, -23.573888778686523, -15.488927841186523, 33.0111198425293, 20.268199920654297, 16.80987548828125, -7.725636005401611, 17.20087432861328, -19.306833267211914, -7.788421154022217, 29.094135284423828, 26.228330612182617, -29.457294464111328, -25.665794372558594, 3.2610645294189453, 51.78396987915039, -57.46695327758789, 8.349987983703613, -41.28129959106445, -19.029930114746094, -10.932852745056152, -12.615250587463379, -16.013877868652344, 0.21621181070804596, -16.057666778564453, -43.873722076416016, 23.396087646484375, 22.586164474487305, 25.295228958129883, -46.2220344543457, 8.967544555664062, 20.27936553955078, -30.93606185913086, 0.27519744634628296, -3.487999439239502, 55.26451873779297, 57.73488235473633, -49.93069076538086, 74.81781005859375, -27.456771850585938, 29.420198440551758, 29.114938735961914, -29.87070655822754, 14.798033714294434, -3.2260122299194336, 16.629899978637695, 6.920168876647949, 35.89018630981445, 2.261986255645752, -5.036221981048584, 32.881927490234375, 17.41187858581543], [21.95440101623535, -19.93760108947754, -31.35788345336914, -17.03166389465332, -18.62464141845703, 23.58597755432129, 29.279521942138672, -4.653908729553223, 42.92757034301758, 41.054080963134766, 7.301967620849609, -24.656753540039062, 47.26129913330078, -1.830520510673523, 3.7849559783935547, -10.188150405883789, -26.693710327148438, -44.059181213378906, 18.98604965209961, -23.502031326293945, 91.78593444824219, -32.58220672607422, -37.721473693847656, 15.304161071777344, 45.8794059753418, 1.9376896619796753, 17.167272567749023, -55.30642318725586, -20.325345993041992, -32.208473205566406, -56.851375579833984, 3.5094070434570312, -67.28482818603516, -39.66587829589844, -16.115814208984375, -24.938575744628906, 11.892254829406738, 9.16740608215332, 26.813962936401367, 22.57151222229004, 34.22438430786133, -0.057551998645067215, 9.10400676727295, 15.34541130065918, 4.249701023101807, -11.9158296585083, -9.973276138305664, -24.526752471923828, 12.648505210876465, -2.820455312728882, -19.895212173461914, -19.86734390258789, -36.31196212768555, 29.3652400970459, 2.358530044555664, -25.884000778198242, 24.256431579589844, -16.813465118408203, -42.19501876831055, 30.636669158935547, -16.1129093170166, -12.266972541809082, 1.2354005575180054, 34.27767562866211], [0.6011187434196472, -35.973514556884766, 16.31631088256836, -14.549692153930664, 56.637718200683594, -0.03359166160225868, -67.05106353759766, 10.720949172973633, 58.33793258666992, 47.27760696411133, 54.928627014160156, -30.592805862426758, 27.364572525024414, -21.659835815429688, 6.728682041168213, -38.18857192993164, -19.33367347717285, -20.28740119934082, 26.9566593170166, 24.021574020385742, 8.30459213256836, -40.51264190673828, -9.349183082580566, -22.383495330810547, 26.090272903442383, 35.34554672241211, 31.69156837463379, -22.65658187866211, -82.18599700927734, -53.00885772705078, 42.648807525634766, -12.03764820098877, -22.521738052368164, 49.279808044433594, 37.95479965209961, 0.7334616780281067, -8.802092552185059, -3.963036298751831, 6.814884185791016, 48.00141906738281, 12.674022674560547, 21.78087043762207, 35.078102111816406, -35.09027862548828, 22.645856857299805, -8.83706283569336, -75.48285675048828, -63.84508514404297, -30.543527603149414, 15.936697006225586, 3.413986921310425, 5.531001091003418, 58.52830505371094, -38.072689056396484, -0.8230006098747253, 9.133065223693848, -22.088193893432617, 0.8498023748397827, -47.99848175048828, 42.29718780517578, -20.448436737060547, 60.9420051574707, -0.9742296934127808, 30.84160614013672], [-1.599086046218872, -0.9223149418830872, -17.5208797454834, 3.5737178325653076, 9.539960861206055, -23.527755737304688, -3.7513325214385986, -18.30296516418457, 68.98992156982422, -2.691923141479492, 46.356964111328125, -5.8686699867248535, 7.778629302978516, -39.64092254638672, -13.412178039550781, 6.3160400390625, 47.580116271972656, 6.659483432769775, 41.51161193847656, 19.221874237060547, 48.82204055786133, -1.7712664604187012, 71.70668029785156, -14.561885833740234, -22.993854522705078, -36.30158615112305, 69.93074035644531, 24.58342742919922, -10.023621559143066, 37.470680236816406, 20.630765914916992, 58.98382568359375, -46.43498611450195, -7.960803031921387, -13.875641822814941, -59.210479736328125, 31.809816360473633, -8.111896514892578, 22.188936233520508, 70.79315948486328, -10.597311019897461, -12.634589195251465, 3.559925079345703, 29.617420196533203, -17.00139045715332, -18.928848266601562, 31.71567726135254, 11.359179496765137, -61.34184646606445, -36.76175308227539, 45.03707504272461, 12.057848930358887, -22.865877151489258, -5.112289905548096, 0.39710018038749695, 22.07200050354004, -40.747901916503906, -25.228473663330078, 8.18867301940918, -2.995037317276001, 17.15577507019043, -36.885223388671875, 10.713860511779785, 32.74750900268555], [14.6045560836792, 4.173352241516113, -2.8765079975128174, -25.045534133911133, -2.1599578857421875, 37.791175842285156, -3.570855140686035, -26.694120407104492, -14.767413139343262, 6.025508880615234, -11.205419540405273, 14.559098243713379, 15.68531322479248, 26.66949462890625, 42.706783294677734, 29.875761032104492, -28.027172088623047, -27.106033325195312, -2.273144245147705, -13.535261154174805, -34.149803161621094, -13.250753402709961, -14.235387802124023, -20.155431747436523, -9.473684310913086, 35.3535270690918, 5.977141380310059, 74.1542739868164, 5.115245342254639, -8.701972961425781, 0.12398648262023926, 7.694759368896484, 13.057619094848633, 62.30133819580078, -25.3596134185791, 25.771381378173828, 6.232517719268799, -47.132930755615234, 23.12096405029297, 38.02870559692383, -25.71529197692871, -0.2680160403251648, -51.76062774658203, -37.76419448852539, 31.440887451171875, -13.9924955368042, -33.26408767700195, 6.883810997009277, -15.672372817993164, -14.124650001525879, 22.179780960083008, -34.85200119018555, -27.16095542907715, 4.622399806976318, 29.082294464111328, 37.14480972290039, -19.95185089111328, 28.121309280395508, -23.524269104003906, 54.78628158569336, -34.01313018798828, 39.85436248779297, -19.694583892822266, 60.39739227294922], [17.739622116088867, -30.48049545288086, -15.667154312133789, 23.435436248779297, 31.57933235168457, -2.1214678287506104, 1.3397483825683594, 22.750591278076172, 3.9539527893066406, -38.805145263671875, -24.38922119140625, 7.053598403930664, 36.464473724365234, 48.47947311401367, 23.385883331298828, 8.618722915649414, 22.46441650390625, -20.002525329589844, -4.596928596496582, -24.723920822143555, -37.462276458740234, 10.006470680236816, 5.565399169921875, 20.629573822021484, 16.38335418701172, 7.279328346252441, 16.69849967956543, -1.280492901802063, 12.386741638183594, -21.784332275390625, 11.091177940368652, 20.05253028869629, -20.090442657470703, 1.5549629926681519, -18.70807456970215, -15.795201301574707, -33.74787902832031, 37.97324752807617, 31.97055435180664, 16.123016357421875, 59.16587829589844, -5.677371025085449, 90.95287322998047, -6.825882434844971, 7.6558003425598145, -11.933337211608887, 11.169766426086426, 25.06844711303711, -3.4174916744232178, 53.30159378051758, 7.174478530883789, 8.114562034606934, -27.269929885864258, 12.554972648620605, -4.020816802978516, -18.2471923828125, -23.294231414794922, 80.15290069580078, -7.904317378997803, -43.709564208984375, -20.946643829345703, -1.5470352172851562, 16.40001106262207, 16.643896102905273], [20.298437118530273, -19.345809936523438, 10.213212966918945, -26.281784057617188, -18.58928871154785, -27.531269073486328, -35.65166473388672, 31.926284790039062, 20.65835189819336, 29.29996109008789, 1.1793919801712036, 23.304231643676758, -52.838680267333984, 25.89262580871582, 5.313594818115234, 18.81328773498535, 11.4006986618042, -29.61165428161621, -44.941192626953125, -21.36162567138672, 16.947519302368164, 20.702741622924805, -41.03826904296875, -29.432981491088867, -21.697071075439453, -10.719151496887207, -40.86152648925781, -42.66155242919922, -24.655059814453125, 2.5876150131225586, -35.41783142089844, -3.4741053581237793, 31.75658416748047, -39.05339050292969, -6.19337272644043, 30.74270248413086, -12.548051834106445, -17.21009063720703, 34.90487289428711, -9.186062812805176, 33.134849548339844, 12.868961334228516, -10.28533935546875, 14.390715599060059, 9.613485336303711, 30.626483917236328, -16.450366973876953, 1.2361819744110107, -1.8454704284667969, 21.12444496154785, -30.755374908447266, 30.50931167602539, 13.989863395690918, -39.50775146484375, 36.247886657714844, 67.02278900146484, 30.05501937866211, 42.80954360961914, 2.5611414909362793, -49.84514617919922, -55.323612213134766, 7.652461528778076, -17.75936508178711, -27.437288284301758], [-1.8824005126953125, 18.467557907104492, -43.582725524902344, -13.466328620910645, 45.296119689941406, -32.25865173339844, 26.092836380004883, -14.541922569274902, 16.37381362915039, 32.91982650756836, 22.238452911376953, 27.27478790283203, -34.54780578613281, 33.4097900390625, -25.580793380737305, 7.786252975463867, 7.747385501861572, 14.55367660522461, 10.1087007522583, -25.314298629760742, -40.49602127075195, 26.170459747314453, 5.575037479400635, 34.48130416870117, -1.34198796749115, -42.154930114746094, 28.983890533447266, -11.11111831665039, 10.260104179382324, -19.423782348632812, 49.32905197143555, -26.296945571899414, -10.122892379760742, 3.223461151123047, -32.76546859741211, 9.9155855178833, 48.35731506347656, -14.713865280151367, 11.854284286499023, -82.48306274414062, -1.1678760051727295, -9.391268730163574, -24.554927825927734, 32.44351577758789, -9.787310600280762, -5.324324607849121, -11.315786361694336, -57.50447463989258, -33.9595947265625, 28.83280372619629, -63.298519134521484, -22.989459991455078, -11.991601943969727, 52.30865478515625, 23.05811309814453, -16.715116500854492, 39.13541030883789, -22.21547508239746, 30.272994995117188, -40.766319274902344, 32.19773483276367, -12.184009552001953, -30.855127334594727, -6.514308452606201], [8.32032585144043, -14.842453002929688, 16.61860466003418, -17.489816665649414, 18.61117172241211, 13.266661643981934, -1.827691674232483, -13.241469383239746, 4.5462236404418945, 16.507612228393555, 61.853973388671875, 28.182613372802734, 29.279699325561523, 92.65536499023438, 15.478754043579102, 14.539305686950684, 21.120452880859375, -30.268980026245117, -13.317049026489258, -15.185077667236328, 19.42841911315918, -5.225236892700195, 4.146702766418457, 32.62363815307617, 25.894521713256836, -4.851630210876465, -20.099853515625, 34.05824279785156, 23.21673583984375, -48.187705993652344, -9.394618034362793, 4.965463638305664, 23.364891052246094, 27.30304527282715, 8.517180442810059, 17.55742645263672, 1.92410409450531, -20.215551376342773, -7.8379316329956055, 0.569735586643219, -1.0403774976730347, -56.56718826293945, -11.924500465393066, 1.6861696243286133, -8.441100120544434, -7.4047136306762695, 14.639740943908691, -32.368717193603516, -48.84126663208008, -10.445094108581543, 22.923431396484375, 50.626590728759766, 23.463848114013672, -44.04791259765625, -6.971017360687256, 7.699031352996826, 5.327733516693115, 16.179136276245117, -27.5114688873291, 1.94829261302948, 12.609684944152832, 24.77012825012207, -24.171545028686523, -10.354705810546875], [-81.11453247070312, -18.64385986328125, -15.732446670532227, -30.587535858154297, 4.968364715576172, -1.6783629655838013, -67.77641296386719, 6.46523904800415, -27.84972381591797, 14.570669174194336, -38.455535888671875, 0.9001961350440979, -8.49489974975586, 9.093056678771973, 3.7409183979034424, -7.654925346374512, -46.08816909790039, -44.31623840332031, -17.5166072845459, -0.24839651584625244, -18.247438430786133, 18.756357192993164, -8.376627922058105, -28.950382232666016, -8.342888832092285, 6.775794982910156, 9.746541976928711, 56.351375579833984, -20.067279815673828, -2.204359769821167, 12.78371810913086, -0.7026887536048889, -15.01854419708252, -46.47771453857422, -15.325037002563477, -32.2110710144043, -51.79819869995117, -23.870365142822266, -31.412752151489258, 17.513595581054688, 18.03275489807129, -27.116378784179688, -7.806859016418457, -55.74728012084961, -14.830198287963867, 11.969456672668457, 24.748136520385742, -21.39141273498535, 31.555644989013672, -45.47075653076172, -37.4288330078125, 21.28985595703125, 50.133636474609375, 14.802959442138672, -1.2102715969085693, -65.85551452636719, -60.67341232299805, 3.8822860717773438, -10.741880416870117, -33.17505645751953, 1.965977430343628, 3.4539411067962646, 28.332475662231445, 4.098068714141846], [36.851829528808594, 0.23982088267803192, -3.460049867630005, 31.929365158081055, -29.353225708007812, 84.43342590332031, 11.25772476196289, 11.424877166748047, 15.75287914276123, 56.31409454345703, 23.775222778320312, -48.15167999267578, 42.10411071777344, -10.226897239685059, 3.2146522998809814, -38.377620697021484, 15.786060333251953, -19.541229248046875, 3.005089044570923, -8.522970199584961, 57.75038528442383, 59.46780776977539, 1.3953951597213745, 1.9193313121795654, -11.203289031982422, 33.1801872253418, -9.867595672607422, -12.640118598937988, 29.357786178588867, -16.14897346496582, -1.1326510906219482, -47.43932342529297, 21.125709533691406, 58.226009368896484, 21.636760711669922, 43.853973388671875, -25.38306427001953, 7.089728355407715, -28.294797897338867, -6.832966327667236, 23.166343688964844, 3.0546183586120605, -42.37923049926758, -39.45087432861328, -41.43491744995117, 14.74357795715332, 61.91683578491211, 40.873748779296875, -56.879554748535156, 42.63227844238281, 28.092012405395508, -46.3589973449707, -9.096048355102539, 32.86567306518555, 16.51300621032715, -39.272850036621094, 18.322265625, -64.12501525878906, -6.684665203094482, 34.138702392578125, 0.5518264770507812, -14.96812629699707, 26.680387496948242, 29.2425537109375], [39.30118942260742, 26.23027229309082, 32.44124221801758, 21.98725700378418, 7.965859413146973, 2.685270071029663, -7.117791652679443, -36.37104797363281, -28.659345626831055, -8.460434913635254, 7.243564605712891, 43.2243766784668, -1.25391685962677, 30.10428810119629, 32.37947082519531, -6.943403720855713, 17.84070587158203, -7.305841445922852, 39.66407775878906, 19.075300216674805, -35.71451187133789, 4.845739364624023, 49.443363189697266, -5.436931133270264, 13.431450843811035, 16.376365661621094, -31.09868621826172, 15.32123851776123, -41.81505584716797, -18.768722534179688, 32.271690368652344, 10.702689170837402, -9.134334564208984, 13.393830299377441, -1.8027673959732056, 0.37735506892204285, 19.70455551147461, -21.82103157043457, 1.2100921869277954, -9.368045806884766, -47.56809997558594, -27.613672256469727, 47.04619216918945, 11.541699409484863, -39.70764923095703, -46.21063232421875, -3.364917039871216, 46.835594177246094, -5.602744102478027, 39.20998764038086, 10.557923316955566, 14.137514114379883, -40.071617126464844, 28.125688552856445, -10.047961235046387, 25.892772674560547, -24.425100326538086, 5.311480522155762, 10.560088157653809, -4.620999336242676, 12.13635540008545, -7.991912364959717, -19.471792221069336, 48.16450119018555], [31.468585968017578, -22.825942993164062, 17.053329467773438, -22.280364990234375, -35.5405387878418, 31.014284133911133, -51.122989654541016, 27.80567741394043, -7.581669330596924, -6.346397876739502, 12.292536735534668, 17.435964584350586, -24.790294647216797, 21.531768798828125, 3.401116132736206, -37.681644439697266, -5.096234321594238, 41.43836975097656, 50.29784393310547, 21.52045440673828, -35.710304260253906, 14.718207359313965, 25.525827407836914, 21.771240234375, 23.548627853393555, 10.897891998291016, -2.172370195388794, -18.25102424621582, -56.93806457519531, 10.189252853393555, 5.58158540725708, 16.727275848388672, -40.126312255859375, 19.350500106811523, 61.6363525390625, 2.353503942489624, -1.8499596118927002, 9.270071029663086, 60.97011184692383, -30.70943260192871, 6.096208095550537, -23.989723205566406, -16.410140991210938, -1.4877092838287354, 40.41969680786133, 54.02971649169922, 13.44426441192627, -11.297842025756836, 74.78216552734375, -12.826634407043457, -17.134077072143555, -53.248291015625, 41.882747650146484, 29.047367095947266, 72.53416442871094, -41.685699462890625, 43.74512481689453, 2.1040656566619873, 24.183006286621094, -49.43331527709961, 44.04245376586914, 23.595502853393555, -1.167301058769226, 19.35338020324707], [-22.964859008789062, -17.383203506469727, -13.903338432312012, -55.58183670043945, 25.852754592895508, 35.747806549072266, -21.040325164794922, 10.446090698242188, -7.966214179992676, 9.84821891784668, -18.827316284179688, -26.50275421142578, 31.562246322631836, -1.916117548942566, 60.66326904296875, 1.389298915863037, -27.78234100341797, 36.8082275390625, 18.92400550842285, 21.80618667602539, 35.718021392822266, -19.58388900756836, -3.817599296569824, -5.143495082855225, -38.72239685058594, -7.9077043533325195, 40.09138488769531, 7.120142936706543, -40.381839752197266, -46.6751823425293, 49.94328308105469, -2.7095704078674316, -19.422306060791016, -8.21975326538086, -44.31923294067383, 42.69285583496094, -11.791587829589844, 41.02204513549805, 1.5495514869689941, 42.88595962524414, 7.70083475112915, 25.59697914123535, 21.572792053222656, 18.698732376098633, 10.438034057617188, -6.98552131652832, 57.46303939819336, -27.20810890197754, 0.12922927737236023, 0.4465762674808502, 12.073222160339355, -12.88417911529541, 9.442389488220215, -5.065277576446533, -47.45857620239258, 0.2865685820579529, -33.01696014404297, -52.03157424926758, 4.106199741363525, 14.636500358581543, 15.800941467285156, 17.015121459960938, -57.97766876220703, -40.830509185791016], [59.20564270019531, -31.416263580322266, 18.325231552124023, -20.608129501342773, 2.3901822566986084, -18.99762535095215, 16.846235275268555, 4.1032233238220215, -13.44869327545166, -31.505163192749023, 8.033226013183594, -36.38156509399414, 12.824431419372559, 78.79454803466797, 40.50510025024414, 21.55043601989746, -2.568218946456909, -58.16257858276367, -39.6685905456543, 38.483787536621094, -23.69268226623535, -20.977563858032227, -4.9695143699646, -31.861387252807617, -36.803062438964844, -1.5142221450805664, -26.74588394165039, -22.23055648803711, -23.43572998046875, 15.670806884765625, -6.963770866394043, -5.35715913772583, 17.908376693725586, -4.26118803024292, -8.347613334655762, 14.113511085510254, 20.045934677124023, -4.412205219268799, 2.4182331562042236, -23.453706741333008, -25.820491790771484, 23.074424743652344, 13.348261833190918, 13.658952713012695, 8.425630569458008, 1.3860982656478882, 20.892356872558594, 28.05326271057129, 27.33683967590332, -3.0233373641967773, 41.60978317260742, 6.853820323944092, 18.971284866333008, 8.462636947631836, 7.410201072692871, -22.072559356689453, 21.040525436401367, 85.35883331298828, -11.243852615356445, 2.1298415660858154, -13.386157989501953, -14.640597343444824, 46.3334846496582, -20.189130783081055], [-14.222414016723633, -47.624786376953125, -22.728351593017578, -51.25199508666992, 18.95414161682129, 83.99091339111328, -11.00036334991455, -50.00273895263672, 26.999990463256836, -21.807748794555664, 38.98040771484375, -8.460871696472168, -18.954471588134766, -5.33872652053833, 22.73330307006836, -6.8226189613342285, -14.901741027832031, -5.9649481773376465, -38.15565490722656, 11.910947799682617, -53.60179901123047, -13.323643684387207, 22.781557083129883, 9.091448783874512, 17.709617614746094, -1.9513702392578125, 35.092384338378906, 27.740976333618164, 13.38422966003418, 27.826374053955078, 22.512706756591797, -13.183905601501465, -40.31696701049805, 12.61086368560791, 16.38267707824707, 32.71416091918945, -28.089637756347656, -27.19293212890625, 13.79700756072998, -2.135009765625, -56.57284927368164, 11.25324535369873, -55.572689056396484, -28.357696533203125, 40.55044174194336, 54.74637222290039, -36.46969223022461, -9.881746292114258, 1.2786756753921509, 53.43680953979492, -14.786067962646484, -14.306520462036133, 46.46176528930664, -11.243674278259277, 21.666051864624023, -44.31428909301758, -5.136982440948486, -49.787906646728516, -31.663013458251953, 25.176040649414062, -1.4971617460250854, -9.573172569274902, -22.462488174438477, -6.619682312011719], [9.033394813537598, 29.487485885620117, -78.75440216064453, -24.952835083007812, 4.772887229919434, -2.965893507003784, 12.101286888122559, 8.026208877563477, -9.146036148071289, -15.805596351623535, -7.365044593811035, 34.4622688293457, -21.960237503051758, -1.8842016458511353, 5.5002665519714355, 46.81052780151367, -9.009247779846191, 28.76420021057129, 36.959381103515625, -1.912627935409546, -6.321569442749023, -9.871609687805176, -7.674694061279297, 25.627195358276367, 6.930299282073975, 7.058691501617432, -33.75670623779297, -29.024702072143555, 64.30340576171875, -5.334771633148193, -23.243160247802734, 42.93297576904297, 8.454256057739258, 17.344648361206055, -4.141412258148193, -9.560193061828613, -5.087591171264648, -41.48214340209961, -2.637822389602661, 8.88099193572998, 55.51389694213867, 2.5799720287323, -70.02531433105469, -2.8195629119873047, 82.18512725830078, 25.99098777770996, -24.32666778564453, 17.733196258544922, -22.26040267944336, -40.614227294921875, -10.451081275939941, 2.745793104171753, -56.16466522216797, -34.47477722167969, 0.31006690859794617, -12.245195388793945, 11.844338417053223, -36.44004821777344, 21.423412322998047, -35.51277160644531, 49.28202819824219, -51.472747802734375, -32.77662658691406, 12.521123886108398], [49.926998138427734, 9.45129108428955, 16.885738372802734, 0.5261070728302002, -30.791954040527344, -14.704150199890137, -36.35729217529297, -38.361297607421875, 12.052467346191406, -33.69533920288086, 9.586620330810547, -39.054683685302734, 8.891490936279297, -14.660476684570312, 42.19846725463867, 9.711566925048828, 37.073486328125, -22.669401168823242, 1.684300184249878, -29.432615280151367, -12.408157348632812, -7.863126754760742, -21.775747299194336, -36.465084075927734, -42.405521392822266, -70.19244384765625, -31.690393447875977, 18.824817657470703, -47.9549560546875, -41.04368591308594, 53.72695541381836, -19.094881057739258, 9.631380081176758, 19.269514083862305, 19.875783920288086, -45.66942596435547, -6.931790351867676, -48.546817779541016, 73.38953399658203, -47.846256256103516, -31.43520736694336, -31.21802520751953, -44.89981460571289, 40.577816009521484, 72.6734390258789, -14.272473335266113, -39.612693786621094, -67.1077651977539, -7.323261737823486, -9.490962982177734, -19.538883209228516, 6.6412577629089355, 11.855194091796875, 9.692498207092285, -9.483746528625488, -33.04865646362305, 19.974992752075195, 0.37541279196739197, 26.650726318359375, 23.702266693115234, -42.652862548828125, -43.02666473388672, -22.666221618652344, -31.68796157836914], [22.65755271911621, 10.539290428161621, 32.975521087646484, 11.894604682922363, 6.259487628936768, 35.99189376831055, 14.091819763183594, 2.0430288314819336, 44.41358947753906, 3.4554264545440674, -9.130218505859375, -25.13469696044922, 21.090763092041016, 37.111515045166016, -1.5176219940185547, 39.23746871948242, -59.1004524230957, -1.663601279258728, -1.2028473615646362, -21.809831619262695, 71.51583099365234, -1.9857789278030396, -12.23192310333252, 0.7788634300231934, -60.08441162109375, -22.796037673950195, 29.49720001220703, -18.86836051940918, -24.866254806518555, 39.285770416259766, 40.2076301574707, 15.509714126586914, -13.653519630432129, -7.184132099151611, -24.12449836730957, -29.84319305419922, -31.332494735717773, -18.5416316986084, -31.041629791259766, -46.91115188598633, 6.582079887390137, 7.037867546081543, 51.02378463745117, -20.050151824951172, 8.883017539978027, 9.381377220153809, -7.615126609802246, 22.330007553100586, 23.25687599182129, 9.822332382202148, -7.669191360473633, -10.725903511047363, -41.032615661621094, -24.177534103393555, -18.66022491455078, -33.719154357910156, 13.197718620300293, 35.898834228515625, -27.241872787475586, 8.503243446350098, -31.023334503173828, -9.147665023803711, -20.70988655090332, 18.056869506835938], [-20.363645553588867, 30.349811553955078, -7.3716230392456055, -19.48811149597168, -37.599159240722656, 7.596135139465332, 26.856552124023438, 8.199637413024902, 67.16494750976562, 42.89362335205078, -9.22583293914795, -9.540822982788086, 14.090901374816895, -25.140575408935547, 34.9886474609375, 10.848015785217285, 7.693767547607422, 20.608610153198242, 51.684898376464844, 5.128660678863525, -3.3912110328674316, 45.26158905029297, 29.026512145996094, -9.629395484924316, 0.15559345483779907, 5.861427307128906, 10.854388236999512, -40.697994232177734, -46.274539947509766, -26.621700286865234, -28.942153930664062, -19.314720153808594, 19.041318893432617, -5.067874908447266, 5.891838073730469, -36.729103088378906, -13.849266052246094, 17.61772918701172, 17.848573684692383, -21.737215042114258, 4.849871635437012, 65.96543884277344, -0.5279378890991211, -20.795608520507812, -5.212871551513672, 2.765313148498535, -4.535051345825195, -0.9406410455703735, -43.30340576171875, -28.87891387939453, -4.480057239532471, -4.958039283752441, -16.98567771911621, -5.310407638549805, 1.830445647239685, 26.682403564453125, -2.9763519763946533, 11.973095893859863, -16.248008728027344, 17.162872314453125, 11.2719087600708, 25.974510192871094, 21.4038143157959, 17.66206932067871], [-2.9525222778320312, 7.348176956176758, 31.9052677154541, -19.562236785888672, -18.85223388671875, 5.442942142486572, 12.458390235900879, -24.723194122314453, -2.2428770065307617, -2.819963216781616, -66.45840454101562, -6.587819576263428, -4.719534873962402, -4.808788299560547, 42.4774284362793, -36.47394943237305, -24.738677978515625, -78.36587524414062, -14.427499771118164, -11.869306564331055, -15.299315452575684, 33.23031997680664, 25.818525314331055, 2.3357017040252686, -11.118391990661621, 30.785690307617188, -0.592735767364502, -24.5481014251709, -51.48746109008789, -13.239435195922852, 22.63930892944336, 68.29615783691406, 1.8483103513717651, -22.960664749145508, -35.54895782470703, 16.55864715576172, -52.938480377197266, 16.510353088378906, 24.531347274780273, 13.088383674621582, -6.589849948883057, 16.948219299316406, 5.128150939941406, -23.34371566772461, 55.51237106323242, -10.550381660461426, 23.27256965637207, -72.55842590332031, 38.294124603271484, 44.06291198730469, 0.8950136303901672, 16.76313018798828, 16.755455017089844, -19.283506393432617, 16.41366958618164, -24.258914947509766, -18.262706756591797, 33.80829620361328, 29.239730834960938, -21.191265106201172, -22.00728988647461, -19.36789894104004, 21.97918701171875, -29.999544143676758], [10.70137882232666, 22.405405044555664, -2.6473634243011475, 30.301910400390625, -9.129304885864258, -6.41566801071167, -29.317386627197266, 30.51531982421875, -22.99561309814453, 10.493154525756836, -2.139482021331787, -31.583492279052734, -18.273094177246094, 41.486167907714844, 6.049726486206055, -30.27450942993164, 9.1598482131958, -4.383420944213867, 103.62139129638672, 8.553715705871582, -11.418745040893555, 42.60083770751953, -55.23108673095703, 2.8296046257019043, 59.832855224609375, -3.485978603363037, -51.99903106689453, -11.877538681030273, 43.63554000854492, 11.282088279724121, -25.16568374633789, -20.557283401489258, -16.79100799560547, 10.835936546325684, 40.1458854675293, 31.572690963745117, 13.680365562438965, -39.949344635009766, -39.837974548339844, 15.997560501098633, -40.364131927490234, -6.767010688781738, 9.124513626098633, -34.1278190612793, -30.72787094116211, -2.823632001876831, 15.109508514404297, 31.845563888549805, 18.21160316467285, 2.209697961807251, -22.38584327697754, -32.1272087097168, -15.326313018798828, 12.298067092895508, 10.098711013793945, -62.9970588684082, -23.570310592651367, 36.13522720336914, 2.3764514923095703, -42.51026916503906, 70.67463684082031, -0.8682519793510437, 11.53001880645752, 11.985742568969727]], "dtype": "float32"}], "e": {"__nd__": [[-0.03433268517255783, -0.2774338126182556, 0.10125406831502914, 0.16081537306308746, 0.08484281599521637, -0.11578677594661713, 0.08869112282991409, 0.11327098309993744, 0.20518599450588226, -0.1153218001127243, -0.029808074235916138, 0.09836294502019882, 0.007556851487606764, -0.14707311987876892, 0.05457568168640137, 0.07616886496543884, 0.011578500270843506, 0.02890382520854473, -0.034053292125463486, 0.194584921002388, -0.11855754256248474, -0.07730866223573685, 0.0014460161328315735, 0.11569615453481674, -0.1970982849597931, -0.17247353494167328, -0.26822489500045776, 0.004303651861846447, -0.14720189571380615, -0.05061808601021767, 0.08705678582191467, -0.08444585651159286, 0.030443506315350533, -0.03940165787935257, 0.0884699821472168, 0.06254080682992935, -0.09107879549264908, -0.022199522703886032, 0.13738727569580078, 0.05177994817495346, -0.1838812381029129, 0.08629858493804932, -0.14276660978794098, 0.18592819571495056, 0.1348014771938324, 0.038157444447278976, 0.020696459338068962, -0.027514781802892685, 0.08931367844343185, -0.048805296421051025, 0.031007913872599602, 0.11477217078208923, -0.07395023852586746, -0.08110906183719635, -0.15721356868743896, -0.1401158720254898, -0.24067051708698273, 0.0960668995976448, 0.23810628056526184, 0.18761514127254486, -0.21404047310352325, 0.09541215747594833, 0.1871909499168396, -0.06012960523366928], [0.011094319634139538, -0.21958374977111816, 0.029170341789722443, -0.11631119251251221, 0.021832142025232315, -0.17073358595371246, 0.05936713144183159, 0.11204099655151367, -0.08564876765012741, -0.1243341788649559, -0.23578330874443054, -0.02913505956530571, -0.06360328197479248, -0.16961991786956787, 0.04954225942492485, -0.04920888692140579, -0.11764279007911682, -0.06064819544553757, 0.11715817451477051, 0.0659373551607132, -0.014219687320291996, 0.06770549714565277, -0.12695765495300293, 0.003540003439411521, 0.016111211851239204, -0.05138145387172699, 0.047621846199035645, -0.08534842729568481, 0.05575413629412651, 0.060351431369781494, -0.31902143359184265, 0.13017216324806213, -0.3332899510860443, -0.06737129390239716, 0.11213234812021255, -0.03469856455922127, -0.1603425145149231, -0.11961138993501663, 0.11230270564556122, -0.210136279463768, 0.12346678227186203, -0.006025981158018112, -0.13040068745613098, -0.06376963108778, -0.11124394088983536, 0.09473063051700592, 0.08009850978851318, -0.18800795078277588, -0.04413694888353348, -0.09439823031425476, 0.14145861566066742, -0.1476472169160843, 0.17543895542621613, -0.00364112202078104, -0.12706947326660156, -0.0740668997168541, -0.08766525983810425, -0.13939262926578522, 0.010370291769504547, -0.13428519666194916, -0.2384454756975174, -0.041521526873111725, -0.19716086983680725, -0.08225652575492859], [-0.13270121812820435, -0.12717090547084808, 0.2920794188976288, 0.008234363980591297, 0.03581666201353073, -0.2592998445034027, -0.09545861929655075, 0.03142759948968887, -0.14954988658428192, 0.13254940509796143, -0.17555692791938782, -0.04180070012807846, -0.085545614361763, 0.023985791951417923, -0.09349196404218674, -0.03954218700528145, -0.03446777164936066, -0.10719815641641617, 0.07080887258052826, 0.034797411412000656, -0.012934925965964794, -0.0628337487578392, 0.11627106368541718, -0.034424588084220886, -0.04476842284202576, 0.01528154220432043, 0.24764977395534515, -0.19565100967884064, 0.11237556487321854, -0.2763451039791107, 0.031715236604213715, 0.22991392016410828, 0.07133065909147263, 0.024906044825911522, -0.08180848509073257, 0.004531941842287779, -0.03679346293210983, -0.11472854763269424, -0.042312972247600555, 0.028957581147551537, -0.11199022829532623, -0.1176924854516983, -0.19855394959449768, 0.05703390762209892, 0.07960052043199539, 0.03381533920764923, 0.1408853679895401, 0.012825987301766872, 0.057221610099077225, 0.04435097426176071, -0.013362043537199497, -0.07748375087976456, 0.3231985569000244, -0.0659710243344307, -0.09719778597354889, 0.06335216015577316, 0.03676304221153259, -0.20505660772323608, 0.1638766974210739, -0.013749527744948864, -0.16216197609901428, 0.17352613806724548, 0.09132950007915497, -0.18458446860313416], [0.2208467423915863, 0.08186735212802887, -0.0012670495780184865, 0.19761312007904053, 0.019795596599578857, -0.0034898242447525263, -0.08205409348011017, 0.07230531424283981, 0.05922643840312958, 0.0018242698861286044, -0.09816992282867432, 0.016146106645464897, -0.008673747070133686, -0.1657080203294754, -0.18289898335933685, 0.0702304095029831, -0.3233311176300049, 0.025525929406285286, -0.05941036716103554, 0.16257302463054657, -0.06132224574685097, 0.08195270597934723, -0.04932364076375961, 0.014135241508483887, 0.09156511723995209, -0.057480815798044205, 0.12383975833654404, -0.12348472326993942, 0.03660842776298523, 0.11465447396039963, 0.08202145248651505, 0.23815253376960754, -0.09487097710371017, -0.17138394713401794, -0.12356390804052353, 0.08207178860902786, 0.06187582388520241, 0.01270721759647131, -0.13954749703407288, 0.09529697149991989, -0.15219146013259888, 0.1440201997756958, 0.3550877869129181, 0.04553810507059097, 0.07246161252260208, -0.061246637254953384, -0.22495137155056, -0.07489865273237228, -0.13581177592277527, -0.17981618642807007, 0.21403402090072632, 0.033908046782016754, 0.047182969748973846, -0.03840155154466629, -0.021482322365045547, 0.11986266076564789, -0.004018465057015419, 0.06594821065664291, -0.024357037618756294, -0.04251861199736595, -0.1748342365026474, -0.2457706779241562, -0.03250374272465706, 0.007020724005997181], [-0.19093477725982666, -0.005118028726428747, 0.007979934103786945, -0.32007408142089844, -0.135856494307518, 0.060461100190877914, 0.05746150761842728, -0.17127090692520142, 0.023522691801190376, 0.1517833173274994, 0.1153617799282074, -0.014145825058221817, -0.06947427242994308, -0.09343048185110092, 0.06945103406906128, 0.028964515775442123, 0.09343931823968887, 0.019577566534280777, 0.12155108898878098, 0.01404290646314621, 0.1164923906326294, 0.12927958369255066, -0.11943746358156204, -0.26843130588531494, 0.22914204001426697, 0.03810273855924606, -0.25452712178230286, -0.10597137361764908, 0.11459000408649445, -0.04151993244886398, -0.07601900398731232, 0.20797045528888702, -0.15547692775726318, -0.11849130690097809, 0.0313817635178566, 0.05796569585800171, 0.07368551194667816, 0.13527950644493103, 0.029480351135134697, 0.07265238463878632, -0.03848138079047203, 0.009307990781962872, 0.01596418209373951, 0.03193278610706329, 0.11468880623579025, 0.08241516351699829, 0.0065643154084682465, -0.12012026458978653, 0.010046259500086308, -0.00415943143889308, 0.028679123148322105, 0.3275737464427948, -0.07775764167308807, 0.06269168108701706, 0.15862049162387848, 0.014549053274095058, 0.34706905484199524, -0.05612251162528992, 0.048113252967596054, -0.1185881569981575, -0.05218624323606491, 0.03695593401789665, 0.12563960254192352, 0.04737180471420288], [0.16021203994750977, -0.01208355650305748, -0.042265720665454865, 0.34405994415283203, 0.05016350746154785, -0.0915481299161911, -0.08833937346935272, -0.11994829028844833, 0.11917077004909515, -0.04636402428150177, 0.01620583049952984, -0.008080958388745785, 0.09307641535997391, -0.10774124413728714, -0.1368577629327774, 0.0285399928689003, 0.07646714150905609, 0.04231201112270355, -0.1029963493347168, 0.08620438724756241, -0.09465020895004272, -0.06636151671409607, -0.12011034786701202, 0.1655617356300354, 0.08059769868850708, 0.07698044925928116, 0.15869742631912231, 0.35788655281066895, -0.21748501062393188, -0.09395236521959305, -0.07840322703123093, -0.10825982689857483, -0.13711363077163696, -0.03812839463353157, -0.1548028439283371, 0.042631395161151886, 0.006578187923878431, 0.06120695173740387, 0.06460428237915039, 0.07672768086194992, -0.04751322418451309, 0.26989811658859253, 0.029829438775777817, -0.2369859218597412, -0.1275440901517868, -0.22146472334861755, 0.0022056896705180407, -0.01549871638417244, -0.05960332974791527, 0.2558738589286804, 0.06985357403755188, 0.09239006787538528, -0.11847147345542908, -0.12713299691677094, 0.004683197941631079, 0.022013740614056587, -0.23620042204856873, -0.022942591458559036, 0.0017905571730807424, -0.027375968173146248, -0.10349725931882858, -0.06865143030881882, -0.02250388264656067, 0.011553558520972729], [0.07980717718601227, 0.1426277756690979, -0.03884157910943031, -0.06019352376461029, 0.057752229273319244, 0.08060596138238907, -0.05729628726840019, 0.10429885238409042, 0.12926426529884338, 0.17044855654239655, 0.2346639335155487, -0.11433067172765732, 0.0018339819507673383, 0.0014881838578730822, -0.012941606342792511, -0.08567553013563156, -0.19603200256824493, 0.012229807674884796, -0.1013907939195633, 0.011347540654242039, -0.10478051006793976, -0.1451088786125183, 0.03091757744550705, 0.019915323704481125, 0.0764409750699997, 0.2884425222873688, 0.036590609699487686, 0.06401049345731735, -0.1421929895877838, -0.16981792449951172, 0.20886723697185516, -0.12186068296432495, -0.13775691390037537, -0.005159421358257532, -0.01190162356942892, -0.11909308284521103, -0.04024370014667511, 0.162226140499115, -0.10635741800069809, -0.1839064359664917, 0.12438290566205978, 0.06496258825063705, -0.08188248425722122, -0.20877982676029205, -0.09825912117958069, -0.0072186957113444805, 0.15381991863250732, -0.13678543269634247, -0.011153086088597775, 0.08909083157777786, -0.3142276108264923, 0.136416494846344, -0.15603399276733398, 0.04380950704216957, 0.032551851123571396, 0.14312000572681427, 0.07269895821809769, -0.31370285153388977, -0.1035580113530159, 0.018237292766571045, -0.025729184970259666, 0.0630246251821518, 0.10416334122419357, 0.08029016107320786], [-0.022435372695326805, -0.19331826269626617, -0.047690555453300476, 0.043595269322395325, -0.11417382210493088, 0.06609678268432617, 0.09511210769414902, 0.013283438049256802, -0.32078900933265686, -0.1338205486536026, 0.19749091565608978, -0.05722451210021973, -0.20774707198143005, 0.1293349415063858, 0.001398379448801279, 0.11569132655858994, 0.028998591005802155, 0.04801282286643982, 0.03800840303301811, 0.07302958518266678, 0.23845532536506653, 0.10166792571544647, -0.16546732187271118, 0.04676709696650505, 0.11951722204685211, 0.1259271800518036, -0.04466710612177849, 0.12197395414113998, 0.09713803231716156, 0.2693520188331604, -0.07852792739868164, -0.05312670022249222, -0.03205178678035736, -0.05132805556058884, -0.09318843483924866, -0.2892622649669647, -0.045514628291130066, 0.038673143833875656, -0.05164213851094246, 0.20264151692390442, 0.0014004966942593455, 0.13517548143863678, 0.02737167663872242, 0.08652787655591965, 0.029237614944577217, 0.14331676065921783, 0.005857055075466633, 0.051945656538009644, 0.03835640475153923, -0.030184542760252953, 0.04240313544869423, 0.23394842445850372, -0.028311550617218018, 0.04391951486468315, 0.13169406354427338, 0.11938870698213577, 0.0012267518322914839, -0.24954955279827118, 0.054100681096315384, -0.23238709568977356, 0.15390782058238983, 0.017744535580277443, 0.10466954112052917, -0.0944543406367302], [-0.17611083388328552, 0.029579034075140953, -0.0038118527736514807, -0.05575861036777496, 0.009603668935596943, 0.31392791867256165, 0.09466555714607239, 0.11905132979154587, 0.0957331508398056, 0.16782483458518982, -0.021561043336987495, -0.23308467864990234, -0.02139252983033657, 0.09208036214113235, -0.11001064628362656, -0.020006315782666206, -0.07847185432910919, 0.17038476467132568, 0.0850156918168068, 0.07390841096639633, 0.07051220536231995, -0.0042254202999174595, 0.022410070523619652, -0.13444305956363678, -0.24802207946777344, 0.017024660483002663, 0.036752279847860336, 0.1371746063232422, 0.1782466471195221, 0.17809493839740753, -0.11321218311786652, -0.03995967283844948, 0.05448738485574722, 0.2872101068496704, 0.16789710521697998, 0.12698034942150116, -0.043989259749650955, -0.10018697381019592, 0.0894908532500267, -0.12485502660274506, -0.027047356590628624, -0.013019994832575321, 0.18069417774677277, -0.04023557901382446, -0.27669966220855713, 0.05722237750887871, -0.07929451018571854, 0.004835909232497215, -0.009835467673838139, -0.031047990545630455, 0.0896533727645874, -0.1738826483488083, -0.05864822119474411, 0.07759478688240051, -0.06965955346822739, -0.18249721825122833, 0.07583849877119064, 0.12739980220794678, -0.005104093812406063, -0.2224312275648117, 0.08008265495300293, -0.18829166889190674, 0.03502907231450081, -0.10067068785429001], [-0.13673089444637299, -0.052095796912908554, -0.021608544513583183, -0.07594428956508636, 0.06940092891454697, -0.052586134523153305, 0.11555253714323044, 0.03314688429236412, -0.0822334811091423, -0.10208374261856079, -0.194219708442688, 0.034273892641067505, -0.04180154576897621, -0.2796528935432434, -0.06315424293279648, -0.10622091591358185, 0.18575315177440643, -0.20073682069778442, -0.16991767287254333, 0.1308518648147583, 0.15994581580162048, -0.08782438933849335, -0.074515201151371, 0.07264663279056549, -0.27433666586875916, -0.21941202878952026, -0.033095039427280426, 0.18271318078041077, -0.05136725679039955, 0.01189921610057354, 0.13365991413593292, 0.2483759969472885, -0.06800339370965958, -0.0019848442170768976, 0.21479806303977966, -0.1794487088918686, 0.12113926559686661, -0.12004696577787399, 0.012080241926014423, -0.02860182523727417, -0.11293787509202957, -0.11589571833610535, 0.03258587792515755, 0.11736194044351578, -0.015294314362108707, 0.008265485987067223, 0.06733445078134537, -0.11454199254512787, 0.08720071613788605, 0.096590556204319, 0.30693575739860535, -0.04433809220790863, -0.05400225147604942, 0.09065169095993042, 0.09656238555908203, -0.06308024376630783, -0.04752795770764351, 0.06379272043704987, -0.09756208211183548, -0.04839694872498512, 0.010561893694102764, 0.11740901321172714, -0.07127479463815689, -0.21719464659690857], [-0.2465652972459793, -0.1620202213525772, -0.1689356118440628, -0.019890272989869118, -0.1529463529586792, -0.0904076024889946, -0.09958413988351822, -0.05667019262909889, 0.17013674974441528, 0.0851578339934349, 0.0020750074181705713, -0.059309836477041245, -0.05573425814509392, 0.042163729667663574, 0.07515183836221695, -0.11676263809204102, 0.004693154711276293, -0.26136258244514465, 0.02723642997443676, 0.09804852306842804, -0.03603010252118111, 0.2336914986371994, -0.11336904019117355, 0.027973145246505737, -0.004608761053532362, -0.012903034687042236, -0.014707250520586967, 0.013778784312307835, 0.10908769071102142, 0.1772422045469284, -0.00028948811814188957, 0.10488829761743546, -0.09028050303459167, 0.09853818267583847, -0.09802822768688202, 0.14031057059764862, 0.28439322113990784, -0.01570577546954155, -0.2820650339126587, -0.07050815969705582, 0.1584775298833847, 0.07115189731121063, 0.03202208876609802, 0.12976256012916565, 0.0623236745595932, 0.15665718913078308, 0.11828146874904633, 0.19576281309127808, 0.046158187091350555, 0.11645927280187607, -0.039511099457740784, -0.07737637311220169, 0.21957123279571533, 0.027814170345664024, 0.10354579985141754, 0.015918832272291183, -0.07545372098684311, 0.0014129206538200378, 0.18473689258098602, 0.07143019139766693, -0.2201765924692154, -0.027690639719367027, 0.22928747534751892, -0.027504410594701767], [-0.21736319363117218, 0.22853873670101166, 0.08238198608160019, -0.008452317677438259, -0.044839490205049515, 0.13066717982292175, -0.06543106585741043, 0.13862350583076477, 0.06545664370059967, 0.029539279639720917, 0.017567306756973267, -0.02112305536866188, 0.12623514235019684, -0.1044687032699585, -0.19580726325511932, 0.07417842000722885, 0.07620638608932495, 0.05797721818089485, 0.049223095178604126, -0.03869730606675148, 0.095069020986557, -0.0055000148713588715, -0.0588068924844265, -0.03765314817428589, 0.24394217133522034, -0.20945537090301514, 0.06033702939748764, -0.20271098613739014, -0.013066493906080723, -0.052349772304296494, -0.2517155110836029, 0.04696664959192276, -0.04588017985224724, -0.2534271478652954, 0.10135704278945923, 0.17324143648147583, 0.17989559471607208, 0.043335095047950745, 0.005560379009693861, 0.1603420078754425, 0.2031061351299286, 0.13685032725334167, -0.11366733908653259, 0.024307401850819588, 0.053060680627822876, -0.21846641600131989, 0.056663669645786285, 0.27440744638442993, 0.07725644111633301, 0.17787812650203705, 0.02208242192864418, -0.008839298039674759, 0.046029072254896164, -0.09849085658788681, -0.11635197699069977, 0.1528749018907547, 0.1656763255596161, -0.11965152621269226, 0.11686187237501144, -0.010498262010514736, 0.041023463010787964, 0.055560000240802765, 0.12004318088293076, 0.00998617336153984], [0.17424814403057098, 0.038232918828725815, 0.07712798565626144, -0.10296296328306198, 0.2667369544506073, 0.05805440619587898, 0.08810227364301682, -0.1987171620130539, -0.10165803879499435, 0.1529603898525238, 0.03545000031590462, -0.23647819459438324, 0.12829738855361938, -0.06719309091567993, 0.17439085245132446, -0.010853507556021214, 0.025294795632362366, 0.14567875862121582, -0.1108996570110321, -0.12268215417861938, -0.02428053319454193, 0.10221046954393387, -0.1644512414932251, -0.20611244440078735, 0.10178393870592117, -0.07450764626264572, -0.04846581816673279, -0.22943870723247528, -0.08389879018068314, -0.000935060903429985, 0.1149631068110466, -0.09529786556959152, 0.20479540526866913, -0.08458787947893143, -0.08440694957971573, 0.18996518850326538, 0.02612929418683052, -0.06821151077747345, -0.019169187173247337, 0.1671939492225647, 0.18117287755012512, -0.10092790424823761, -0.005243378225713968, 0.01114032231271267, -0.12186579406261444, -0.00688078161329031, 0.000657724856864661, 0.1408911645412445, 0.14919674396514893, -0.04067910090088844, -0.026437344029545784, 0.08208397030830383, -0.07230941951274872, 0.07273593544960022, -0.010862257331609726, -0.11978474259376526, -0.11823252588510513, 0.1788073182106018, -0.01932838372886181, 0.20269207656383514, -0.06346800178289413, 0.03598307818174362, 0.2956536114215851, -0.13065709173679352], [-0.25298210978507996, 0.17150479555130005, -0.11021514981985092, -0.0027601956389844418, -0.07810395956039429, 0.035746585577726364, -0.12670819461345673, -0.023986337706446648, 0.05996119603514671, -0.13440939784049988, -0.09332247078418732, -0.046306759119033813, -0.19046367704868317, -0.05632814019918442, 0.015835793688893318, -0.053260594606399536, -0.03617497533559799, -0.002652928698807955, -0.0874117910861969, 0.09211988747119904, 0.19141846895217896, -0.06190274655818939, 0.06467780470848083, -0.03194139897823334, 0.04872012510895729, 0.20872123539447784, 0.06780523806810379, -0.2363993525505066, 0.06051843985915184, 0.1643429696559906, -0.02953917160630226, 0.13271991908550262, -0.09199365228414536, -0.19802504777908325, 0.0188180860131979, -0.2341415137052536, -0.03196191042661667, 0.17153839766979218, -0.25457486510276794, -0.17055922746658325, -0.1375717669725418, -0.008440650999546051, -0.048750706017017365, 0.18650740385055542, 0.12721635401248932, 0.06602077931165695, 0.019412783905863762, 0.052418239414691925, -0.11490380764007568, 0.11426769196987152, -0.14823751151561737, 0.11135726422071457, 0.1581149399280548, 0.028108634054660797, -0.023489518091082573, 0.053362034261226654, -0.3814455568790436, 0.04029156640172005, -0.015110292471945286, -0.006748819258064032, 0.09418206661939621, -0.009966832585632801, -0.15199163556098938, -0.01884923130273819], [0.008240547962486744, 0.0140198590233922, 0.019751235842704773, 0.12678423523902893, -0.1324765831232071, 0.02143227867782116, 0.15734809637069702, -0.21159124374389648, 0.021913647651672363, -0.07140559703111649, -0.04521996155381203, -0.11037015169858932, -0.02068212255835533, 0.2245998978614807, -0.15423764288425446, -0.14267127215862274, -0.1588124781847, 0.21071380376815796, -0.05639708787202835, -0.0031275018118321896, -0.04913223907351494, -0.11520251631736755, -0.02868412621319294, 0.29986128211021423, 0.16980747878551483, -0.10404299199581146, 0.0789923369884491, -0.101430743932724, -0.2166847437620163, 0.12436189502477646, 0.14151035249233246, 0.12759946286678314, -0.05731506645679474, -0.02034362405538559, 0.09145783632993698, 0.04775543510913849, -0.04801665246486664, -0.19749748706817627, 0.013890714384615421, 0.1316196620464325, -0.14608412981033325, 0.07791709154844284, 0.04044209420681, 0.0915045291185379, 0.10636524856090546, -0.0847122073173523, -0.19723981618881226, -0.06507974117994308, 0.10845745354890823, -0.06611467152833939, -0.08466073870658875, -0.222171813249588, 0.21379131078720093, 0.10072214156389236, 0.09391339123249054, 0.11804500222206116, 0.07034799456596375, -0.029803702607750893, -0.019976260140538216, -0.05215512216091156, 0.32139071822166443, 0.06781121343374252, -0.056575268507003784, 0.05678218975663185], [-0.134455606341362, 0.20011433959007263, -0.08792875707149506, 0.06664934009313583, 0.1417856067419052, 0.051178038120269775, 0.012565447948873043, 0.0014367846306413412, -0.007507226429879665, -0.07670662552118301, -0.059785760939121246, 0.19680021703243256, -0.0144364507868886, -0.10451871156692505, -0.06867270916700363, 0.14636023342609406, 0.0898623988032341, 0.07452934980392456, -0.03425287827849388, 0.07626290619373322, -0.08560001850128174, -0.03453124687075615, 0.12899363040924072, 0.11628761887550354, -0.13060376048088074, -0.1137935221195221, 0.014458466321229935, 0.22959275543689728, -0.25478920340538025, 0.03702104464173317, -0.18302743136882782, -0.08437232673168182, -0.04847260192036629, -0.05593178793787956, -0.07100015878677368, 0.0009586106170900166, -0.07119430601596832, -0.19452136754989624, 0.10373040288686752, 0.10013947635889053, 0.11215055733919144, -0.20493298768997192, 0.039759084582328796, 0.08991190046072006, -0.13716012239456177, 0.0012201331555843353, -0.015464618802070618, 0.24502433836460114, 0.2559770941734314, -0.22137592732906342, 0.3317170739173889, -0.12173411250114441, 0.13043929636478424, 0.12908586859703064, -0.13243669271469116, 0.06560951471328735, -0.014303055591881275, 0.07373140007257462, 0.030681706964969635, 0.15912504494190216, 0.01002888847142458, -0.022328918799757957, 0.14578743278980255, 0.07719843089580536], [0.09350750595331192, -0.08491761237382889, -0.13355852663516998, -0.07254073768854141, -0.07932549715042114, 0.10045666992664337, 0.12470643222332001, -0.019821783527731895, 0.18283578753471375, 0.1748562753200531, 0.031100314110517502, -0.1050172820687294, 0.20129387080669403, -0.007796496618539095, 0.01612076722085476, -0.043393056839704514, -0.11369302868843079, -0.187655508518219, 0.0808647945523262, -0.1000991240143776, 0.3909318149089813, -0.1387731283903122, -0.16066212952136993, 0.06518301367759705, 0.1954081505537033, 0.008252947591245174, 0.07311831414699554, -0.2355594038963318, -0.08656908571720123, -0.1371813267469406, -0.24213962256908417, 0.01494715828448534, -0.2865774631500244, -0.16894368827342987, -0.06863997876644135, -0.10621760785579681, 0.050651125609874725, 0.039045531302690506, 0.11420520395040512, 0.09613588452339172, 0.1457674354314804, -0.00024512369418516755, 0.038775503635406494, 0.06535869836807251, 0.01810019463300705, -0.05075153335928917, -0.042477868497371674, -0.10446358472108841, 0.05387212336063385, -0.012012796476483345, -0.08473706990480423, -0.08461837470531464, -0.15465879440307617, 0.12507152557373047, 0.010045378468930721, -0.11024433374404907, 0.10331224650144577, -0.07161138951778412, -0.17971572279930115, 0.13048675656318665, -0.06862760335206985, -0.05224711075425148, 0.005261779762804508, 0.1459944099187851], [0.0021606185473501682, -0.12930065393447876, 0.05864619091153145, -0.05229638144373894, 0.20357459783554077, -0.00012073948892066255, -0.2410035878419876, 0.03853461891412735, 0.20968572795391083, 0.16993127763271332, 0.1974315643310547, -0.10996061563491821, 0.09835728257894516, -0.07785257697105408, 0.024185096845030785, -0.13726229965686798, -0.06949158757925034, -0.07291959971189499, 0.09689110517501831, 0.08634144067764282, 0.02984943799674511, -0.14561577141284943, -0.03360404074192047, -0.08045364916324615, 0.09377703070640564, 0.12704353034496307, 0.11390992999076843, -0.08143521100282669, -0.29540351033210754, -0.19053128361701965, 0.15329384803771973, -0.04326726868748665, -0.0809505358338356, 0.17712785303592682, 0.1364220380783081, 0.0026363027282059193, -0.03163761645555496, -0.01424445677548647, 0.02449493668973446, 0.17253290116786957, 0.04555460810661316, 0.07828761637210846, 0.12608224153518677, -0.1261260062456131, 0.0813966616988182, -0.031763311475515366, -0.2713102400302887, -0.22948025166988373, -0.10978349298238754, 0.0572817325592041, 0.012270992621779442, 0.01988023892045021, 0.2103699892759323, -0.13684576749801636, -0.0029581349808722734, 0.03282724320888519, -0.07939223945140839, 0.003054469358175993, -0.17252233624458313, 0.15203002095222473, -0.07349841296672821, 0.21904562413692474, -0.0035017021000385284, 0.11085488647222519], [-0.0063212551176548, -0.003645949997007847, -0.06926078349351883, 0.01412705797702074, 0.037711869925260544, -0.09300621598958969, -0.014829176478087902, -0.07235239446163177, 0.2727200984954834, -0.01064128614962101, 0.1832510530948639, -0.02319910190999508, 0.030749252066016197, -0.15670225024223328, -0.05301890894770622, 0.024967573583126068, 0.18808621168136597, 0.026325220242142677, 0.1640971601009369, 0.075984887778759, 0.1929955929517746, -0.007001891266554594, 0.28345954418182373, -0.05756375193595886, -0.09089568257331848, -0.14350171387195587, 0.27643918991088867, 0.09717933088541031, -0.03962380066514015, 0.14812318980693817, 0.08155429363250732, 0.23316556215286255, -0.18355947732925415, -0.03146939352154732, -0.054850999265909195, -0.23406153917312622, 0.12574554979801178, -0.032066673040390015, 0.0877138078212738, 0.2798483669757843, -0.04189161956310272, -0.04994506761431694, 0.014072534628212452, 0.11707892268896103, -0.0672072172164917, -0.07482653856277466, 0.12537342309951782, 0.04490331932902336, -0.24248692393302917, -0.14532077312469482, 0.1780334711074829, 0.04766518995165825, -0.09038978070020676, -0.020209098234772682, 0.0015697538619861007, 0.08725155144929886, -0.16107818484306335, -0.09972922503948212, 0.03237017244100571, -0.011839509941637516, 0.06781750917434692, -0.1458088606595993, 0.042352344840765, 0.12945228815078735], [0.06444865465164185, 0.018416645005345345, -0.012693783268332481, -0.11052379012107849, -0.009531708434224129, 0.16676922142505646, -0.015757877379655838, -0.11779886484146118, -0.06516733020544052, 0.026590052992105484, -0.04944855347275734, 0.06424804776906967, 0.06921794265508652, 0.11769019067287445, 0.18846136331558228, 0.13183917105197906, -0.12368150800466537, -0.11961659789085388, -0.010031190700829029, -0.05972994491457939, -0.1507001519203186, -0.05847443640232086, -0.06281954795122147, -0.08894418925046921, -0.04180655628442764, 0.15601207315921783, 0.02637661248445511, 0.32723644375801086, 0.022573139518499374, -0.03840105980634689, 0.0005471417098306119, 0.03395631164312363, 0.05762215331196785, 0.2749304473400116, -0.11190979927778244, 0.1137268915772438, 0.027503564953804016, -0.20799358189105988, 0.10203082859516144, 0.16781741380691528, -0.1134793758392334, -0.0011827318230643868, -0.22841520607471466, -0.16665014624595642, 0.13874593377113342, -0.061747681349515915, -0.14679156243801117, 0.030377667397260666, -0.0691608339548111, -0.06233086809515953, 0.09787747263908386, -0.1537988930940628, -0.11985896527767181, 0.020398253574967384, 0.12833766639232635, 0.16391685605049133, -0.08804580569267273, 0.12409692257642746, -0.10381057858467102, 0.24176715314388275, -0.15009702742099762, 0.17587387561798096, -0.08691050857305527, 0.26652851700782776], [0.08173880726099014, -0.14044488966464996, -0.07218950241804123, 0.10798338800668716, 0.14550799131393433, -0.009775080718100071, 0.006173154804855585, 0.10482782870531082, 0.018218616023659706, -0.17880235612392426, -0.11237814277410507, 0.032500844448804855, 0.16801725327968597, 0.2233787178993225, 0.10775506496429443, 0.039712462574243546, 0.10350922495126724, -0.09216557443141937, -0.021181253716349602, -0.11392033845186234, -0.17261481285095215, 0.04610678553581238, 0.025643672794103622, 0.09505482763051987, 0.07548953592777252, 0.033540938049554825, 0.07694162428379059, -0.005900123156607151, 0.057074353098869324, -0.10037560015916824, 0.05110478773713112, 0.09239598363637924, -0.09257066994905472, 0.0071647982113063335, -0.08620113879442215, -0.07277949899435043, -0.15549999475479126, 0.17496922612190247, 0.14731062948703766, 0.07428997755050659, 0.27261844277381897, -0.026159605011343956, 0.41908329725265503, -0.031451597809791565, 0.03527560830116272, -0.054985202848911285, 0.05146690085530281, 0.1155078113079071, -0.015746766701340675, 0.24559760093688965, 0.03305782377719879, 0.03738944232463837, -0.12565156817436218, 0.05784951150417328, -0.018526704981923103, -0.08407753705978394, -0.1073327586054802, 0.369320273399353, -0.03642069920897484, -0.20140042901039124, -0.09651578962802887, -0.007128269411623478, 0.07556628435850143, 0.07669003307819366], [0.09005014598369598, -0.08582399785518646, 0.045308973640203476, -0.11659412831068039, -0.08246783912181854, -0.12213722616434097, -0.15816181898117065, 0.14163488149642944, 0.09164684265851974, 0.12998369336128235, 0.0052321478724479675, 0.10338478535413742, -0.23440873622894287, 0.11486770212650299, 0.023572750389575958, 0.08346156775951385, 0.05057702586054802, -0.13136646151542664, -0.1993730366230011, -0.0947667807340622, 0.07518444210290909, 0.09184376895427704, -0.1820584535598755, -0.1305738091468811, -0.09625492244958878, -0.047553472220897675, -0.18127436935901642, -0.1892598420381546, -0.10937747359275818, 0.011479460634291172, -0.15712445974349976, -0.015412206761538982, 0.14088203012943268, -0.1732529252767563, -0.027475718408823013, 0.13638414442539215, -0.05566704273223877, -0.07634928822517395, 0.15484881401062012, -0.040752217173576355, 0.14699645340442657, 0.057090695947408676, -0.045628949999809265, 0.06384167075157166, 0.042648397386074066, 0.13586856424808502, -0.0729789212346077, 0.005484085995703936, -0.008187077939510345, 0.09371457248926163, -0.13644036650657654, 0.13534875214099884, 0.06206336244940758, -0.17526860535144806, 0.1608068346977234, 0.29733383655548096, 0.13333337008953094, 0.18991638720035553, 0.011362016201019287, -0.22112849354743958, -0.24543265998363495, 0.033948685973882675, -0.07878603786230087, -0.12172029912471771], [-0.007953441701829433, 0.0780283659696579, -0.18414394557476044, -0.05689737945795059, 0.19138331711292267, -0.13629794120788574, 0.11024638265371323, -0.06144193559885025, 0.06918196380138397, 0.13909150660037994, 0.09396100044250488, 0.11524032056331635, -0.14596997201442719, 0.14116168022155762, -0.10808292031288147, 0.032898157835006714, 0.03273393586277962, 0.061491597443819046, 0.042710866779088974, -0.10695693641901016, -0.1711021363735199, 0.11057435721158981, 0.023555420339107513, 0.145688995718956, -0.005670112557709217, -0.17811128497123718, 0.12246154993772507, -0.046946242451667786, 0.043350573629140854, -0.08206856995820999, 0.20842309296131134, -0.11110877990722656, -0.04277082905173302, 0.013619636185467243, -0.13843931257724762, 0.041894927620887756, 0.2043173462152481, -0.0621684230864048, 0.050086237490177155, -0.34850406646728516, -0.004934461787343025, -0.03967960551381111, -0.10374847799539566, 0.13707901537418365, -0.04135294258594513, -0.022496119141578674, -0.04781099781394005, -0.24296554923057556, -0.1434846967458725, 0.12182318419218063, -0.2674463093280792, -0.09713412821292877, -0.05066642537713051, 0.22101238369941711, 0.09742419421672821, -0.07062402367591858, 0.16535332798957825, -0.09386391937732697, 0.12790821492671967, -0.17224417626857758, 0.1360405534505844, -0.05147937685251236, -0.13036781549453735, -0.027523988857865334], [0.04038573056459427, -0.07204325497150421, 0.08066444844007492, -0.08489319682121277, 0.09033609926700592, 0.06439457088708878, -0.008871367201209068, -0.06427229195833206, 0.0220667514950037, 0.08012570440769196, 0.30023080110549927, 0.1367945820093155, 0.14211969077587128, 0.4497365653514862, 0.07513176649808884, 0.07057181745767593, 0.102515809237957, -0.14692151546478271, -0.064639151096344, -0.07370630651712418, 0.09430290758609772, -0.025362590327858925, 0.02012753300368786, 0.15835070610046387, 0.12568849325180054, -0.0235491544008255, -0.09756196290254593, 0.16531407833099365, 0.11269088089466095, -0.23389658331871033, -0.045600201934576035, 0.024101685732603073, 0.11341001093387604, 0.1325252801179886, 0.04134123772382736, 0.08522136509418488, 0.009339340031147003, -0.09812354296445847, -0.03804425522685051, 0.0027654191944748163, -0.005049851257354021, -0.27456945180892944, -0.05787990614771843, 0.008184438571333885, -0.04097195342183113, -0.0359414741396904, 0.07105931639671326, -0.15711335837841034, -0.23706887662410736, -0.05069907009601593, 0.11126722395420074, 0.24573460221290588, 0.11389033496379852, -0.21380259096622467, -0.03383637219667435, 0.03737005218863487, 0.025860095396637917, 0.07853133231401443, -0.1335369348526001, 0.009456748142838478, 0.061205700039863586, 0.12023083865642548, -0.11732539534568787, -0.050260335206985474], [-0.33970171213150024, -0.07807911932468414, -0.06588633358478546, -0.12809835374355316, 0.02080714702606201, -0.007028860971331596, -0.28384265303611755, 0.02707594633102417, -0.1166326031088829, 0.06102089211344719, -0.16104896366596222, 0.003769955364987254, -0.03557601571083069, 0.038081053644418716, 0.015666691586375237, -0.032058265060186386, -0.19301387667655945, -0.1855931580066681, -0.07335826754570007, -0.0010402663610875607, -0.076418936252594, 0.07855024933815002, -0.03508070483803749, -0.12124207615852356, -0.03493940830230713, 0.028376532718539238, 0.040817804634571075, 0.23599542677402496, -0.08404029905796051, -0.009231696836650372, 0.0535372719168663, -0.0029428089037537575, -0.06289656460285187, -0.1946452558040619, -0.06418012827634811, -0.1348976045846939, -0.21692705154418945, -0.0999673381447792, -0.13155430555343628, 0.07334565371274948, 0.0755198523402214, -0.11356140673160553, -0.03269455209374428, -0.2334655225276947, -0.06210778281092644, 0.05012720823287964, 0.10364338010549545, -0.08958566933870316, 0.13215272128582, -0.19042819738388062, -0.1567492038011551, 0.08916035294532776, 0.20995599031448364, 0.06199370697140694, -0.005068528465926647, -0.2757980525493622, -0.25409579277038574, 0.016258729621767998, -0.04498620703816414, -0.13893470168113708, 0.008233369328081608, 0.014464852400124073, 0.11865433305501938, 0.01716241054236889], [0.13901495933532715, 0.000904668471775949, -0.01305223349481821, 0.1204461008310318, -0.11072821915149689, 0.3185054659843445, 0.0424671471118927, 0.043097689747810364, 0.05942407250404358, 0.21243181824684143, 0.08968649804592133, -0.18164101243019104, 0.15882796049118042, -0.038578592240810394, 0.012126527726650238, -0.14477065205574036, 0.0595492385327816, -0.0737147405743599, 0.011335999704897404, -0.03215092420578003, 0.21784989535808563, 0.22432847321033478, 0.005263803526759148, 0.007240230683237314, -0.04226180166006088, 0.12516453862190247, -0.0372232086956501, -0.04768190532922745, 0.11074542254209518, -0.06091824546456337, -0.004272662568837404, -0.17895381152629852, 0.0796918272972107, 0.21964406967163086, 0.08161965012550354, 0.16542890667915344, -0.09575170278549194, 0.026744350790977478, -0.10673554241657257, -0.02577577531337738, 0.08738964051008224, 0.011522836983203888, -0.15986578166484833, -0.14881925284862518, -0.15630358457565308, 0.05561671778559685, 0.23356686532497406, 0.15418671071529388, -0.21456488966941833, 0.1608203500509262, 0.1059705838561058, -0.1748785376548767, -0.03431272506713867, 0.12397810816764832, 0.062291473150253296, -0.14814768731594086, 0.0691164880990982, -0.24189670383930206, -0.025216344743967056, 0.1287803202867508, 0.0020816370379179716, -0.05646377429366112, 0.10064555704593658, 0.11031073331832886], [0.19121341407299042, 0.12761902809143066, 0.15783748030662537, 0.10697534680366516, 0.038756564259529114, 0.013064735569059849, -0.0346304327249527, -0.17695729434490204, -0.13943728804588318, -0.04116284102201462, 0.03524236008524895, 0.21030102670192719, -0.00610072398558259, 0.1464674174785614, 0.15753693878650665, -0.033781979233026505, 0.08680099248886108, -0.03554535657167435, 0.19297897815704346, 0.09280770272016525, -0.17376302182674408, 0.02357614040374756, 0.2405584752559662, -0.026452485471963882, 0.06534849107265472, 0.07967648655176163, -0.15130549669265747, 0.07454293966293335, -0.2034442126750946, -0.0913161039352417, 0.15701255202293396, 0.052072156220674515, -0.0444415882229805, 0.0651654601097107, -0.00877106562256813, 0.001835958450101316, 0.09586924314498901, -0.10616660118103027, 0.005887502338737249, -0.045578669756650925, -0.2314346879720688, -0.1343497335910797, 0.22889544069766998, 0.056154221296310425, -0.19319097697734833, -0.22483015060424805, -0.01637144573032856, 0.22787080705165863, -0.02725922130048275, 0.19076968729496002, 0.05136782303452492, 0.0687837302684784, -0.1949618011713028, 0.13684086501598358, -0.04888668656349182, 0.12597697973251343, -0.11883626878261566, 0.025842126458883286, 0.05137835443019867, -0.022482704371213913, 0.05904741957783699, -0.03888332098722458, -0.09473676979541779, 0.23433637619018555], [0.12288273870944977, -0.08913379162549973, 0.06659211963415146, -0.08700334280729294, -0.13878343999385834, 0.12110871821641922, -0.19963188469409943, 0.10857932269573212, -0.029605915769934654, -0.02478226274251938, 0.04800153896212578, 0.06808628141880035, -0.09680445492267609, 0.08408012986183167, 0.013281133025884628, -0.14714431762695312, -0.019900456070899963, 0.16181407868862152, 0.19640973210334778, 0.0840359479188919, -0.13944636285305023, 0.057473622262477875, 0.09967666119337082, 0.08501524478197098, 0.09195582568645477, 0.0425555445253849, -0.008482960984110832, -0.07126903533935547, -0.22233936190605164, 0.03978835418820381, 0.021795719861984253, 0.06531889736652374, -0.1566905826330185, 0.07556241750717163, 0.2406858652830124, 0.009190276265144348, -0.007223969325423241, 0.03619901090860367, 0.23808424174785614, -0.11991829425096512, 0.0238052885979414, -0.09367827326059341, -0.06408051401376724, -0.005809405818581581, 0.15783622860908508, 0.21098245680332184, 0.052498959004879, -0.04411732405424118, 0.29201939702033997, -0.050087153911590576, -0.06690743565559387, -0.2079310417175293, 0.16354934871196747, 0.1134280413389206, 0.2832410931587219, -0.16277988255023956, 0.17082180082798004, 0.008216236717998981, 0.09443303197622299, -0.19303381443023682, 0.17198285460472107, 0.09213887155056, -0.004558233078569174, 0.07557366043329239], [-0.10013051331043243, -0.07579358667135239, -0.060620810836553574, -0.24234582483768463, 0.11272220313549042, 0.15586623549461365, -0.0917392298579216, 0.045546650886535645, -0.034733984619379044, 0.04293983057141304, -0.08209015429019928, -0.11555630713701248, 0.13761651515960693, -0.008354582823812962, 0.264501690864563, 0.006057568825781345, -0.12113551050424576, 0.16048984229564667, 0.08251173049211502, 0.09507851302623749, 0.15573637187480927, -0.08538892865180969, -0.01664535142481327, -0.02242647297680378, -0.16883593797683716, -0.03447887301445007, 0.1748049408197403, 0.03104497864842415, -0.1760713756084442, -0.20351137220859528, 0.2177608162164688, -0.01181416679173708, -0.08468440175056458, -0.03583945706486702, -0.19323904812335968, 0.1861477792263031, -0.05141323432326317, 0.1788627654314041, 0.006756295915693045, 0.18698973953723907, 0.03357688710093498, 0.11160697788000107, 0.09406087547540665, 0.08152950555086136, 0.04551152139902115, -0.030458005145192146, 0.25054818391799927, -0.1186317652463913, 0.0005634606350213289, 0.001947144977748394, 0.05264120548963547, -0.056177109479904175, 0.04117035120725632, -0.022085431963205338, -0.206927090883255, 0.0012494855327531695, -0.14395929872989655, -0.22686609625816345, 0.01790369674563408, 0.06381751596927643, 0.06889466941356659, 0.07418868690729141, -0.25279203057289124, -0.17802764475345612], [0.27124103903770447, -0.14392851293087006, 0.08395407348871231, -0.09441280364990234, 0.010950231924653053, -0.08703453838825226, 0.07717829197645187, 0.018798252567648888, -0.06161300837993622, -0.1443357914686203, 0.03680292144417763, -0.16667623817920685, 0.05875305086374283, 0.3609844446182251, 0.18556754291057587, 0.09872982650995255, -0.011765878647565842, -0.26646241545677185, -0.18173521757125854, 0.17630723118782043, -0.10854417830705643, -0.09610530734062195, -0.022767024114727974, -0.1459677815437317, -0.16860726475715637, -0.006937162950634956, -0.12253192812204361, -0.10184568911790848, -0.10736699402332306, 0.07179325819015503, -0.03190338611602783, -0.024542955681681633, 0.0820443257689476, -0.019521942362189293, -0.038243237882852554, 0.0646587610244751, 0.09183719754219055, -0.02021380327641964, 0.011078743264079094, -0.10744935274124146, -0.11829239130020142, 0.10571173578500748, 0.06115289777517319, 0.0625762790441513, 0.03860066086053848, 0.006350184325128794, 0.09571494162082672, 0.12852147221565247, 0.125239297747612, -0.013850930146872997, 0.19062846899032593, 0.031399667263031006, 0.08691386133432388, 0.038770198822021484, 0.0339486338198185, -0.10112185031175613, 0.09639375656843185, 0.3910576403141022, -0.05151188746094704, 0.00975752342492342, -0.061326511204242706, -0.0670735239982605, 0.2122693508863449, -0.09249322861433029], [-0.05817258730530739, -0.1947951316833496, -0.09296361356973648, -0.20963115990161896, 0.07752632349729538, 0.3435400426387787, -0.04499373957514763, -0.20452144742012024, 0.11043549329042435, -0.08919815719127655, 0.1594378501176834, -0.0346066989004612, -0.07752767205238342, -0.02183648571372032, 0.09298386424779892, -0.027905909344553947, -0.060951173305511475, -0.02439785934984684, -0.1560644507408142, 0.04871821776032448, -0.21924233436584473, -0.05449643358588219, 0.09318123757839203, 0.03718588873744011, 0.07243596017360687, -0.007981504313647747, 0.14353503286838531, 0.113466277718544, 0.05474423989653587, 0.11381556838750839, 0.09208158403635025, -0.05392487347126007, -0.1649046540260315, 0.051581013947725296, 0.06700850278139114, 0.13380761444568634, -0.1148923709988594, -0.11122465878725052, 0.056432586163282394, -0.008732627145946026, -0.23139454424381256, 0.046028077602386475, -0.22730368375778198, -0.11598879098892212, 0.16585960984230042, 0.22392387688159943, -0.14916850626468658, -0.040418367832899094, 0.005230045411735773, 0.21856749057769775, -0.06047804653644562, -0.058516599237918854, 0.19003812968730927, -0.04598893225193024, 0.08861859142780304, -0.1812545210123062, -0.021011309698224068, -0.20364274084568024, -0.12950821220874786, 0.10297515988349915, -0.006123698316514492, -0.0391562357544899, -0.09187617897987366, -0.027075856924057007], [0.03720224276185036, 0.12143835425376892, -0.32433435320854187, -0.10276329517364502, 0.019656188786029816, -0.012214443646371365, 0.04983674734830856, 0.03305434808135033, -0.03766613453626633, -0.06509221345186234, -0.030331473797559738, 0.14192600548267365, -0.09043888002634048, -0.007759710308164358, 0.022651756182312965, 0.19277986884117126, -0.037102799862623215, 0.11845964938402176, 0.15220987796783447, -0.007876778021454811, -0.026034127920866013, -0.0406542643904686, -0.031606703996658325, 0.10554051399230957, 0.028541062027215958, 0.029069818556308746, -0.13902029395103455, -0.11953247338533401, 0.26482081413269043, -0.021970197558403015, -0.09572234004735947, 0.17681093513965607, 0.034817174077034, 0.07143048942089081, -0.017055584117770195, -0.039371758699417114, -0.02095223404467106, -0.1708359718322754, -0.0108633479103446, 0.036574602127075195, 0.22862295806407928, 0.010625102557241917, -0.2883853614330292, -0.011611809954047203, 0.3384631276130676, 0.1070387214422226, -0.10018455237150192, 0.07303065061569214, -0.0916750505566597, -0.16726163029670715, -0.04304070398211479, 0.011308004148304462, -0.23130302131175995, -0.14197751879692078, 0.00127694895491004, -0.05042940750718117, 0.048778556287288666, -0.15007109940052032, 0.08822806924581528, -0.14625230431556702, 0.20295824110507965, -0.21198029816150665, -0.13498403131961823, 0.05156576260924339], [0.18731677532196045, 0.0354594811797142, 0.06335213780403137, 0.0019738555420190096, -0.11552567034959793, -0.05516722798347473, -0.13640578091144562, -0.14392443001270294, 0.045218609273433685, -0.12641862034797668, 0.03596721217036247, -0.14652588963508606, 0.033359214663505554, -0.05500337481498718, 0.15832076966762543, 0.036435987800359726, 0.1390928030014038, -0.08505136519670486, 0.006319180130958557, -0.1104256808757782, -0.04655309021472931, -0.029500985518097878, -0.08169854432344437, -0.13681019842624664, -0.1590976119041443, -0.2633489668369293, -0.11889644712209702, 0.07062720507383347, -0.1799180507659912, -0.15398825705051422, 0.20157350599765778, -0.07164043188095093, 0.03613514080643654, 0.07229562103748322, 0.07457023113965988, -0.17134316265583038, -0.026006784290075302, -0.1821386069059372, 0.2753438353538513, -0.1795102208852768, -0.11793903261423111, -0.11712420731782913, -0.16845573484897614, 0.15224039554595947, 0.272657185792923, -0.053547658026218414, -0.14861944317817688, -0.2517758011817932, -0.02747551165521145, -0.03560832142829895, -0.07330624759197235, 0.024916760623455048, 0.04447847604751587, 0.036364443600177765, -0.03558124601840973, -0.12399239093065262, 0.074942447245121, 0.0014084787108004093, 0.09998854994773865, 0.08892647922039032, -0.16002558171749115, -0.16142801940441132, -0.08503943681716919, -0.11888732016086578], [0.10480654239654541, 0.04875136539340019, 0.1525341421365738, 0.055020611733198166, 0.02895437553524971, 0.166486918926239, 0.06518422067165375, 0.009450393728911877, 0.20544296503067017, 0.015983689576387405, -0.04223345220088959, -0.11626501381397247, 0.09755907952785492, 0.17166592180728912, -0.00702003063634038, 0.1814998984336853, -0.2733796536922455, -0.007695283740758896, -0.005563984625041485, -0.10088525712490082, 0.33080920577049255, -0.009185574017465115, -0.05658093839883804, 0.0036027715541422367, -0.2779311537742615, -0.10544712841510773, 0.136444553732872, -0.08727896213531494, -0.11502329260110855, 0.18172332644462585, 0.18598754703998566, 0.071742944419384, -0.06315678358078003, -0.03323148190975189, -0.11159216612577438, -0.13804499804973602, -0.1449340283870697, -0.08576761931180954, -0.143588587641716, -0.21699588000774384, 0.030446583405137062, 0.03255490958690643, 0.23601959645748138, -0.09274554997682571, 0.041089978069067, 0.043395232409238815, -0.035225123167037964, 0.10329142957925797, 0.1075788214802742, 0.045434948056936264, -0.03547520935535431, -0.04961457476019859, -0.18980367481708527, -0.1118374913930893, -0.08631619065999985, -0.15597395598888397, 0.061048395931720734, 0.16605645418167114, -0.12601213157176971, 0.03933326527476311, -0.14350396394729614, -0.04231415316462517, -0.09579727053642273, 0.08352526277303696], [-0.10341635346412659, 0.15413089096546173, -0.03743663430213928, -0.098969966173172, -0.1909465491771698, 0.03857681527733803, 0.13639044761657715, 0.041641686111688614, 0.34109577536582947, 0.21783436834812164, -0.0468531996011734, -0.048452869057655334, 0.0715603455901146, -0.1276758909225464, 0.17768912017345428, 0.05509142205119133, 0.039072636514902115, 0.10466039925813675, 0.2624806761741638, 0.026045797392725945, -0.017222194001078606, 0.22986003756523132, 0.1474105417728424, -0.04890268296003342, 0.0007901781355030835, 0.02976713702082634, 0.05512378364801407, -0.2066839188337326, -0.23500429093837738, -0.1351977437734604, -0.1469821333885193, -0.09808940440416336, 0.09670094400644302, -0.0257370974868536, 0.029921576380729675, -0.18652798235416412, -0.07033321261405945, 0.08947127312421799, 0.09064360707998276, -0.1103919968008995, 0.024629972875118256, 0.33500412106513977, -0.002681121462956071, -0.10561006516218185, -0.026473458856344223, 0.014043585397303104, -0.023031163960695267, -0.004777025897055864, -0.2199154496192932, -0.1466609686613083, -0.022751877084374428, -0.025179298594594002, -0.08626141399145126, -0.02696879394352436, 0.009295879863202572, 0.1355060338973999, -0.015115341171622276, 0.06080511957406998, -0.0825151726603508, 0.08716128766536713, 0.057244155555963516, 0.1319110095500946, 0.10869882255792618, 0.08969645202159882], [-0.012578248046338558, 0.031304486095905304, 0.13592186570167542, -0.08333846181631088, -0.08031372725963593, 0.02318786084651947, 0.05307486280798912, -0.10532501339912415, -0.009555038064718246, -0.01201352383941412, -0.2831241190433502, -0.028065232560038567, -0.02010602131485939, -0.02048625610768795, 0.18096107244491577, -0.15538524091243744, -0.10539098083972931, -0.33385196328163147, -0.0614636056125164, -0.050565268844366074, -0.06517768651247025, 0.14156682789325714, 0.10999131202697754, 0.009950486943125725, -0.04736624285578728, 0.13115228712558746, -0.0025251551996916533, -0.10457909107208252, -0.2193453460931778, -0.05640224739909172, 0.09644731134176254, 0.29095324873924255, 0.007874117232859135, -0.0978163406252861, -0.15144461393356323, 0.07054265588521957, -0.22552694380283356, 0.07033691555261612, 0.10450771450996399, 0.05575874447822571, -0.028073882684111595, 0.07220230251550674, 0.021846795454621315, -0.09944820404052734, 0.23649215698242188, -0.044946420937776566, 0.09914510697126389, -0.30911123752593994, 0.1631394922733307, 0.18771550059318542, 0.0038129105232656, 0.0714137852191925, 0.0713810920715332, -0.08215101808309555, 0.06992502510547638, -0.10334710776805878, -0.07780224084854126, 0.14402909576892853, 0.12456623464822769, -0.09027839452028275, -0.09375480562448502, -0.08251054584980011, 0.09363508224487305, -0.12780316174030304], [0.04348130151629448, 0.09103650599718094, -0.01075663324445486, 0.12312118709087372, -0.03709372878074646, -0.026067817583680153, -0.11912092566490173, 0.1239883080124855, -0.09343460947275162, 0.04263525456190109, -0.008693034760653973, -0.12832844257354736, -0.07424630969762802, 0.16856449842453003, 0.02458094246685505, -0.12300985306501389, 0.03721783310174942, -0.01781049370765686, 0.4210292100906372, 0.034755025058984756, -0.04639606922864914, 0.17309357225894928, -0.22441215813159943, 0.011497106403112411, 0.2431098371744156, -0.014164051972329617, -0.21127983927726746, -0.04826021566987038, 0.17729772627353668, 0.04584081098437309, -0.10225193202495575, -0.08352731168270111, -0.06822437793016434, 0.04402802884578705, 0.16311873495578766, 0.12828455865383148, 0.05558536946773529, -0.16232015192508698, -0.1618676483631134, 0.06500048190355301, -0.16400550305843353, -0.027495374903082848, 0.037074264138936996, -0.13866643607616425, -0.12485193461179733, -0.011472838930785656, 0.06139219179749489, 0.12939327955245972, 0.07399646192789078, 0.008978333324193954, -0.09095702320337296, -0.13053764402866364, -0.06227310001850128, 0.04996888339519501, 0.0410325713455677, -0.255966454744339, -0.09576969593763351, 0.14682282507419586, 0.009655877947807312, -0.17272557318210602, 0.28716161847114563, -0.003527837572619319, 0.04684818908572197, 0.048699863255023956]], "dtype": "float32"}}]'
import json, types
harness = types.ModuleType('harness'); exec(_HARNESS, harness.__dict__)
res = harness.grade(l2_normalize_rows, json.loads(_TESTS), 'cuda', *[1e-05, 1e-07])
for i, r in enumerate(res): print(f"test {i+1}: {'pass' if r['ok'] else 'FAIL  ' + r['why']}")
print(f"{sum(r['ok'] for r in res)}/{len(res)} passed")